# PTCG Phase 2 round 3: re-collect with the redesigned opponent pool, retrain, gate

2026-07-07: round 2's regression was root-caused to severe win/loss label
imbalance (searching side won 96.7% of self-play-mirror games; the real
ladder is ~47% positive) — see `docs/report-log.md` "Root cause of the
Phase 2 round 2 regression". Fix: `training/nn/opponent_pool.py`, a
real-ladder-meta-weighted pool of the project's existing rule-based
archetype bots (lucario/dragapult/starmie/abomasnow with real decks+logic,
several more archetypes' reconstructed decks piloted by `generic_pilot.py`,
a mirror slice piloted by the real heuristic `main.py`), replacing the
same-checkpoint mirror opponent in `mcts_collect.py`.

Validated locally at n=20 games: win rate dropped 96.7%→43.1% (matches the
real ladder's 47.1%), `value_target` mean moved from +0.72 (collapsed) to
-0.03 (well-centered). This notebook re-collects at full scale (300 games),
retrains via `train_sp.py`, and gates via `dmc_replay_gate.py --value-source
head` against real replays — same protocol as round 2, for a clean
before/after: pre-training baseline ALL 0.584, round-2 (imbalanced-corpus)
post-training ALL 0.446.

**Data attached:** `jander6364/ptcg-sp-p2-r2-data` (init checkpoint),
`jander6364/ptcg-ladder-replays-bulk` (real replay JSONs for the gate),
`kiyotah/cg-lib` (cg.api source, paired with kaggle_environments' native
binary — needed both for `threat.py`'s feature AND for the archetype bots'
own cg.api usage, and for `mcts.py`'s real `search_begin`/`search_step`
calls).

**Accelerator:** CPU (no GPU needed — the collection step is the
expensive part here, not the retrain).


In [ ]:
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
!echo '--- /kaggle/input ---'; find /kaggle/input -maxdepth 5 2>&1 | head -60


## Assemble the `cg` package (native binary from kaggle_environments + Python source from kiyotah/cg-lib)

In [ ]:
import glob, os, shutil, sys

get_ipython().system('pip install kaggle_environments --no-deps -q')

import kaggle_environments
cabt_cg_dir = os.path.join(os.path.dirname(kaggle_environments.__file__), "envs", "cabt", "cg")
print("cabt cg dir:", cabt_cg_dir, os.listdir(cabt_cg_dir))

native_candidates = [f for f in os.listdir(cabt_cg_dir) if f.startswith("libcg") and f.endswith(".so")]
assert native_candidates, f"no native libcg*.so found in {cabt_cg_dir}"
native_lib_name = native_candidates[0]
native_lib_path = os.path.join(cabt_cg_dir, native_lib_name)
print("using native lib:", native_lib_path)

cglib_src_candidates = glob.glob("/kaggle/input/**/api.py", recursive=True)
assert cglib_src_candidates, "kiyotah/cg-lib dataset not found under /kaggle/input"
cglib_dir = os.path.dirname(cglib_src_candidates[0])
print("cg-lib source dir:", cglib_dir, os.listdir(cglib_dir))

out_dir = "/kaggle/working/repo/training/local_cg/cg"
os.makedirs(out_dir, exist_ok=True)
for fname in ("api.py", "sim.py", "utils.py", "game.py", "__init__.py"):
    src = os.path.join(cglib_dir, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(out_dir, fname))
shutil.copy(native_lib_path, os.path.join(out_dir, native_lib_name))
if native_lib_name != "libcg.so":
    shutil.copy(native_lib_path, os.path.join(out_dir, "libcg.so"))

print("assembled cg package:", os.listdir(out_dir))


## Write out the repo tree

Preserves the real relative-path structure (`training/nn/`, `opponents/`,
repo-root `main.py`) exactly as in the actual repo, since several modules
rely on `__file__`-relative `sys.path`/data-file lookups
(`opponent_pool.py`'s `training/archetype_decks.json` load,
`generic_pilot.py`'s `from main import DECK`, `mcts.py`'s `training/local_cg`
lookup, etc.) — writing everything flat into one directory would break
those.

In [ ]:
import os
os.makedirs("/kaggle/working/repo/training/nn", exist_ok=True)
os.makedirs("/kaggle/working/repo/opponents", exist_ok=True)
os.chdir("/kaggle/working/repo")
print("cwd:", os.getcwd())


In [ ]:
%%writefile main.py
import sys, glob, random, math

for _pat in ['/kaggle/input/**/cg-lib', '/kaggle/input/cg-lib']:
    _paths = glob.glob(_pat, recursive=True)
    if _paths: sys.path.insert(0, _paths[0]); break

try:
    from cg.api import all_card_data, all_attack
except ImportError:
    all_card_data = lambda: []
    all_attack    = lambda: []

NUMBER,YES,NO,CARD,TOOL_CARD,ENERGY_CARD,ENERGY,PLAY,ATTACH,EVOLVE,\
    ABILITY,DISCARD,RETREAT,ATTACK,END,SKILL,SPECIAL_CONDITION = range(17)

CTX_SETUP_ACTIVE = 1
CTX_SETUP_BENCH  = 2
LOG_PLAY         = 10
LOG_HP_CHANGE    = 16
CARDTYPE_SUPPORTER = 3

ABRA,KADABRA,ALAKAZAM             = 741,742,743
DUNSPARCE,DUNSPARCE2,DUDUNSPARCE  = 305,65,66
GENESECT,SHAYMIN,PSYDUCK,FEZ      = 142,343,858,140
POFFIN,POKE_PAD,HANDHELD_FAN      = 1086,1152,1161
BOSS,LANA,BATTLE_CAGE,DAWN        = 1182,1184,1264,1231
WONDROUS_PATCH,SACRED_ASH,HILDA   = 1146,1129,1225
ENHANCED_HAMMER,RARE_CANDY        = 1081,1079
BASIC_P,ENRICHING,TELEPATH_P      = 5,13,19

MIST_ENERGY = 11
ROCK_ENERGY = 20
PSYCHIC_TYPE = 5

ALAKAZAM_LINE_IDS      = {ABRA,KADABRA,ALAKAZAM}
DRAW_ABILITY_CARD_IDS  = {KADABRA,ALAKAZAM,DUDUNSPARCE}
SUPPRESS_ABILITY_IDS   = set()  # No own-pokemon abilities to suppress in this deck
PSYCHIC_ENERGY_IDS     = {BASIC_P,TELEPATH_P}
BENCHABLE_BASIC_IDS    = {ABRA,DUNSPARCE,DUNSPARCE2,PSYDUCK,SHAYMIN,GENESECT,FEZ}
LINE_SEARCH_HAND_IDS   = {DAWN,HILDA}
DRAW_ENGINE_IDS        = {DUNSPARCE,DUNSPARCE2,DUDUNSPARCE}
NON_ATTACKER_IDS       = {DUNSPARCE,DUNSPARCE2,DUDUNSPARCE,GENESECT,SHAYMIN,PSYDUCK,FEZ,ABRA,KADABRA}
# Kadabra has a real attack (Super Psy Bolt, {P}->30 flat dmg) but this deck's
# ATTACK scoring only ever rewards Alakazam's Powerful Hand (active_can_attack
# requires is_alak; any Kadabra attack scores -5 regardless of state) -- Kadabra
# is functionally a non-attacker here. It was missing from this set, so a stuck
# Kadabra active (no attack, e.g. no energy) with a ready, fully-fueled Alakazam
# waiting on bench fell through every retreat-priority tier and scored a flat
# 0.5 -- LOWER than just ending the turn (1.0). Verified via score_options_main
# on a synthetic Kadabra-active/fueled-bench-Alakazam state before this fix.
PIVOT_FREE_RETREAT_IDS = {SHAYMIN}
TOOL_IDS               = {HANDHELD_FAN}
PH_DMG_PER_CARD = 20

# v23 deck: reverted from v24 (Alakazam/Dunsparce 4th copies, Genesect/Psyduck cut)
# 2026-07-03 — user call to revert on v24's early ladder trend (680 at 7h, 780
# at 24h), ahead of the documented 48h decision-rule checkpoint.
DECK = ([ABRA]*4+[KADABRA]*4+[ALAKAZAM]*3+[DUNSPARCE]*3+[DUDUNSPARCE]*3+
        [GENESECT]+[SHAYMIN]+[PSYDUCK]+[FEZ]+
        [POFFIN]*4+[POKE_PAD]*4+[HANDHELD_FAN]*2+[BOSS]*3+[LANA]+
        [BATTLE_CAGE]*4+[DAWN]*4+[WONDROUS_PATCH]+[SACRED_ASH]+[HILDA]*3+
        [ENHANCED_HAMMER]*2+[RARE_CANDY]*3+[BASIC_P]*2+[ENRICHING]+[TELEPATH_P]*4)

_CARDS = None; _ATKMAP = None
_STALL_MEMO = {}

# Tunable scoring weights (v22). Defaults ARE v21 behavior; tuning harnesses may
# mutate this dict in-process — the submission never reads env/files.
W = {
    'atk_threshold':150.0,'atk_default':7.0,
    'dudun_base':11.0,'fez_base':8.0,
    'evo_bench_establish':40.0,'evo_bench_convert':25.0,'evo_bench_late':12.0,
    'boss_target':16.0,'boss_mega_chip':18.0,'boss_mist_escape':12.0,
    'cage_base':6.0,'cage_reactive':22.0,'hammer_mist':45.0,
    'poffin_estab':25.0,'poffin_need':19.0,
    'dawn_estab':22.0,'dawn_need':11.0,'hilda_estab':24.0,'hilda_immobile':18.0,
    'pad_no_backup':13.0,'end_convert':4.0,
    'candy_ready':30.0,'candy_estab':25.0,'candy_active_abra':45.0,
    'attach_kadabra':9.0,'attach_abra':6.0,
    'retreat_nonatk_ready':30.0,'retreat_alak_stuck':22.0,'desperation_draw':30.0,
    # MCTS heuristic-prior blend temperatures (used by training/nn/mcts.py).
    # score_options_main's scale spans ~4 (default) to 600 (KO) — prior_T_h_main
    # is tuned large enough that a softmax doesn't fully saturate on the KO/
    # boss-snipe outliers alone. Non-MAIN select types (_score_deck_search,
    # _score_bench_target, etc.) score in a much flatter ~0-100 range, hence the
    # separate, smaller default temperature. prior_T_net is the net policy
    # logits' own softmax temperature (unrelated scale, tuned independently).
    'prior_T_h_main':40.0,'prior_T_h_default':3.0,'prior_T_net':1.0,
}

def _select_fingerprint(obs, sel):
    """Coarse signature of a select + game state, for detecting a genuinely stuck
    engine selection (same question, same state, asked again). Deliberately loose:
    false negatives (missing a real stall) are free; false positives just cost one
    extra, still-valid rotation of an equally-good blind pick."""
    opts = sel.get('option', [])
    opt_sig = tuple(sorted(
        (o.get('area'), o.get('type'), o.get('playerIndex')) for o in opts))
    cur = obs.get('current') or {}
    me_idx = cur.get('yourIndex', 0)
    pl = cur.get('players') or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    opp = pl[1-me_idx] if len(pl) == 2 else {}
    state_sig = (
        me.get('deckCount'), me.get('handCount'), len(me.get('prize') or []),
        opp.get('deckCount'), len(opp.get('prize') or []))
    return (sel.get('type'), sel.get('context'), sel.get('minCount'),
            sel.get('maxCount'), opt_sig, state_sig)

def _resolve_stalled_or(obs, sel, fallback_indices):
    """If the exact same select+state was seen on a prior call with no progress,
    rotate to a different (still valid) combination instead of repeating the
    identical answer forever. First occurrence always uses fallback_indices."""
    global _STALL_MEMO
    fp = _select_fingerprint(obs, sel)
    seen = _STALL_MEMO.get(fp, 0)
    _STALL_MEMO[fp] = seen + 1
    if seen == 0:
        return fallback_indices
    n = len(sel.get('option', []))
    mx = sel.get('maxCount', 1) or 1
    mn = sel.get('minCount', 0) or 0
    if n == 0: return fallback_indices
    k = max(mn, min(mx, n))
    offset = (seen * k) % n
    idxs = [(offset + i) % n for i in range(k)]
    return idxs

def _meta():
    global _CARDS,_ATKMAP
    if _CARDS is None:
        _CARDS={}; _ATKMAP={}
        try:
            for a in all_attack():
                _ATKMAP[a.attackId]=(getattr(a,'damage',0) or 0,
                                     tuple(getattr(a,'energies',()) or ()))
            for c in all_card_data(): _CARDS[c.cardId]=c
        except: pass
    return _CARDS,_ATKMAP

def _prize_value_pk(pk):
    if not pk: return 1
    if pk.get('megaEx',False): return 3
    if pk.get('ex',False):     return 2
    return 1

def _analyze_logs(obs_dict, me_idx):
    logs = obs_dict.get('logs') or []
    opp_idx = 1 - me_idx
    opp_played = set()
    bench_damage_received = False
    we_were_kod_last_turn = False
    C, _ = _meta()
    for log in logs[-30:]:
        ltype = log.get('type')
        lplayer = log.get('playerIndex')
        if ltype == LOG_PLAY and lplayer == opp_idx:
            cid = log.get('cardId', 0)
            if cid: opp_played.add(cid)
        if ltype == LOG_HP_CHANGE and lplayer == me_idx:
            if log.get('putDamageCounter', False):
                bench_damage_received = True
        if log.get('type') == 6 and lplayer == me_idx:
            from_area = log.get('fromArea'); to_area = log.get('toArea')
            if from_area in (4, 5) and to_area == 3:
                we_were_kod_last_turn = True
    opp_has_ace_spec = any(
        C.get(cid) and getattr(C.get(cid), 'aceSpec', False)
        for cid in opp_played)
    return opp_played, bench_damage_received, we_were_kod_last_turn, opp_has_ace_spec

def _pk_id(pk): return (pk or {}).get('id',-1)
def _energies(pk): return (pk or {}).get('energies') or []
def _has_psychic(pk): return PSYCHIC_TYPE in _energies(pk)
def _can_ph(pk): return _pk_id(pk)==ALAKAZAM and _has_psychic(pk)
def _hand_list(p): return p.get('hand') or []
def _hand_size(state,me):
    try:
        p=state['players'][me]; h=p.get('handCount')
        return h if h is not None else len(_hand_list(p))
    except: return 0
def _active(p):
    a=p.get('active'); return a[0] if a and len(a)>0 and a[0] else None
def _is_non_attacker(pk): return not pk or _pk_id(pk) in NON_ATTACKER_IDS

def _opp_has_blocking_energy(opp_active):
    if not opp_active: return False
    for ec in (opp_active.get('energyCards') or []):
        if ec.get('id') in (MIST_ENERGY, ROCK_ENERGY): return True
    return False

# ---- Stage 3 Phase C: archetype belief model (docs/belief-model.md) ----
# Phase A logistic-regression weights (training/belief/train.py), embedded so
# the submission stays a single dependency-free file. Inference is a sparse
# dot product + softmax over just the features present this decision.
_BELIEF_CLASSES=['abomasnow', 'alakazam', 'dragapult', 'lucario', 'starmie']
_BELIEF_INTERCEPT=[3.6972,-13.9598,3.4858,4.3845,2.3923]
_BELIEF_COEF={
 'card_1030':(-1.6093,-2.0067,-1.0127,-0.8535,5.4823),
 'card_1031':(-0.3760,-0.2426,-0.0696,-0.0163,0.7045),
 'card_1071':(-0.6380,-0.9967,3.3556,-0.6908,-1.0300),
 'card_1079':(-0.1592,0.1880,0.2210,-0.1622,-0.0876),
 'card_1080':(-0.0642,-0.0413,0.2111,-0.0847,-0.0208),
 'card_1081':(-0.2300,1.5441,-0.8033,-0.2518,-0.2590),
 'card_1086':(-0.7273,0.0711,0.3475,-0.3196,0.6283),
 'card_1097':(-0.2998,-0.1298,0.4002,-0.1516,0.1809),
 'card_1102':(-0.1612,-0.1813,-0.2128,0.5981,-0.0428),
 'card_1120':(-0.1754,-0.2313,0.6789,-0.1959,-0.0763),
 'card_1121':(0.1163,-0.2989,0.3812,-0.4025,0.2040),
 'card_1123':(-0.1153,-0.0986,-0.1569,0.3985,-0.0277),
 'card_1126':(0.4494,-0.1648,-0.1196,-0.1057,-0.0592),
 'card_1129':(-0.0463,0.1489,-0.0566,-0.0270,-0.0190),
 'card_1141':(-0.1681,-0.1361,-0.2254,0.5781,-0.0484),
 'card_1142':(-0.1599,-0.1826,-0.2086,0.5925,-0.0414),
 'card_1146':(-0.0277,0.0803,-0.0319,-0.0146,-0.0062),
 'card_1152':(-0.4481,0.0622,0.4874,0.1958,-0.2973),
 'card_1156':(-0.1114,-0.3169,0.6368,-0.1264,-0.0821),
 'card_1159':(-0.0780,-0.0770,-0.1020,0.2752,-0.0181),
 'card_1161':(-0.4762,2.2399,-1.0466,-0.3546,-0.3626),
 'card_1182':(-0.5774,1.4469,-0.2799,-0.2101,-0.3795),
 'card_1184':(-0.0167,0.0720,-0.0297,-0.0156,-0.0101),
 'card_119':(-0.9974,-1.3342,4.6081,-0.9459,-1.3307),
 'card_1192':(-0.1778,-0.2764,-0.4494,0.1865,0.7171),
 'card_1198':(-0.1603,-0.4678,0.8769,-0.1837,-0.0651),
 'card_120':(-0.1611,-0.3324,0.7406,-0.1959,-0.0511),
 'card_121':(-0.1332,-0.1489,0.4891,-0.1653,-0.0417),
 'card_1210':(-0.2257,-0.2067,0.4775,-0.1363,0.0912),
 'card_1225':(-0.1097,0.5067,-0.1838,-0.0922,-0.1210),
 'card_1227':(0.2629,-0.9058,0.4827,0.1064,0.0538),
 'card_1231':(-0.0729,0.3130,-0.1159,-0.0674,-0.0568),
 'card_1252':(-0.0569,-0.0346,-0.0701,0.1713,-0.0097),
 'card_1256':(-0.1103,-0.2322,0.5103,-0.1038,-0.0639),
 'card_1262':(0.0887,-0.1398,-0.1762,-0.1365,0.3638),
 'card_13':(-0.0207,0.0907,-0.0273,-0.0155,-0.0273),
 'card_140':(-1.4882,3.0688,2.0158,-1.5750,-2.0214),
 'card_142':(-0.8622,4.4454,-1.2781,-1.0592,-1.2459),
 'card_184':(-0.6768,-1.0155,3.4719,-0.7185,-1.0612),
 'card_19':(-0.1237,0.4879,-0.1747,-0.0861,-0.1034),
 'card_2':(-0.1585,-0.4050,0.8307,-0.1801,-0.0870),
 'card_235':(-0.8030,-1.1330,3.9113,-0.8302,-1.1451),
 'card_3':(0.8316,-1.3201,-0.6358,-0.3415,1.4658),
 'card_305':(-1.1519,5.1270,-1.3307,-1.2522,-1.3921),
 'card_343':(-0.8381,4.3288,-1.2361,-1.0143,-1.2403),
 'card_5':(-0.4774,0.9052,0.1570,-0.3684,-0.2164),
 'card_6':(-0.1851,-0.1843,-0.2404,0.6628,-0.0531),
 'card_66':(-0.0358,0.1355,-0.0405,-0.0291,-0.0301),
 'card_673':(-0.7785,-0.8822,-0.8680,3.6639,-1.1352),
 'card_674':(-0.1067,-0.0763,-0.1485,0.3568,-0.0252),
 'card_675':(-0.7794,-1.0258,-0.8766,3.8579,-1.1761),
 'card_676':(-0.9059,-0.9769,-0.9723,4.1075,-1.2524),
 'card_677':(-0.9320,-0.9377,-0.9911,4.1270,-1.2663),
 'card_678':(-0.1295,-0.1512,-0.1867,0.4979,-0.0306),
 'card_721':(4.5619,-0.8983,-1.0592,-0.9764,-1.6281),
 'card_722':(4.9754,-0.9643,-1.1832,-1.1094,-1.7185),
 'card_723':(0.6568,-0.1137,-0.1999,-0.1723,-0.1708),
 'card_741':(-1.4255,6.2033,-2.0198,-1.3126,-1.4454),
 'card_742':(-0.1017,0.5202,-0.2496,-0.0797,-0.0893),
 'card_743':(-0.0864,0.3866,-0.1405,-0.0776,-0.0821),
 'card_858':(-0.8104,4.1247,-1.1835,-0.9561,-1.1746),
 'energy_0':(-0.0021,0.0155,-0.0051,-0.0032,-0.0051),
 'energy_2':(-0.1628,-0.3699,0.8553,-0.2020,-0.1206),
 'energy_3':(1.4939,-1.2081,-0.6295,-0.5068,0.8505),
 'energy_5':(-0.3017,0.2192,0.7092,-0.3459,-0.2808),
 'energy_6':(-0.2714,-0.2535,-0.4088,1.0222,-0.0886),
 'opp_bench_n':(-0.8345,1.8342,-0.0563,-0.3744,-0.5690),
 'opp_discard_n':(0.2012,0.0345,0.0536,-0.0474,-0.2419),
 'opp_hand_n':(-0.6089,2.3041,-0.5815,-0.7374,-0.3762),
 'opp_prizes_taken':(0.0559,-0.2472,0.0926,0.1285,-0.0298),
 'turn':(0.0096,-0.5821,0.1320,0.0106,0.4299),
}
# Ace-spec reveal rate by archetype across 679 real ladder replays (tech
# survey 2026-07-04 -- see docs/report-log.md Phase C entry).
_ACE_RATE={'lucario':0.60,'alakazam':0.63,'dragapult':0.46,'abomasnow':0.48,'starmie':0.29}
# Dwebble/Crustle line: the one recognized archetype that meaningfully techs
# Mist/Rock (35.8% of its 53 tagged ladder replays; every other recognized
# archetype is 0-3%). The unrecognized long tail techs 29.0%.
_CRUSTLE_LINE_IDS={344,345,532,533}

def _belief_posterior(opp, turn):
    """P(archetype | opponent's public board+discard). Mirrors
    training/belief/collect.py's feature extraction: revealed card ids from
    active/bench (self + preEvolution chain + tools) + discard; energy TYPE
    counts read fresh from the board; scalar counts. Returns (posterior dict,
    wall_energy_revealed, crustle_line_seen); (None, False, False) on any
    failure so callers fall back to pre-Phase-C behavior. Pure -- safe under
    score_options_main's thousands-of-calls contract."""
    try:
        z=list(_BELIEF_INTERCEPT)
        def add(name,val):
            w=_BELIEF_COEF.get(name)
            if w:
                for k in range(5): z[k]+=w[k]*val
        seen=set(); energy={}; wall=False
        for pk in list(opp.get('active') or[])+list(opp.get('bench') or[]):
            if not pk: continue
            if pk.get('id'): seen.add(pk['id'])
            for pre in pk.get('preEvolution') or[]:
                if (pre or{}).get('id'): seen.add(pre['id'])
            for t in pk.get('tools') or[]:
                if (t or{}).get('id'): seen.add(t['id'])
            for et in pk.get('energies') or[]:
                energy[et]=energy.get(et,0)+1
            for ec in pk.get('energyCards') or[]:
                if (ec or{}).get('id') in(MIST_ENERGY,ROCK_ENERGY): wall=True
        for c in opp.get('discard') or[]:
            if (c or{}).get('id'): seen.add(c['id'])
        if MIST_ENERGY in seen or ROCK_ENERGY in seen: wall=True
        for cid in seen: add('card_%d'%cid,1.0)
        for et,cnt in energy.items(): add('energy_%d'%et,float(cnt))
        add('turn',float(turn))
        add('opp_bench_n',float(len(opp.get('bench') or[])))
        add('opp_discard_n',float(len(opp.get('discard') or[])))
        add('opp_hand_n',float(opp.get('handCount') or 0))
        add('opp_prizes_taken',float(6-len(opp.get('prize') or[])))
        m=max(z); exps=[math.exp(v-m) for v in z]; s=sum(exps)
        return ({c:e/s for c,e in zip(_BELIEF_CLASSES,exps)},
                wall, bool(seen&_CRUSTLE_LINE_IDS))
    except: return None,False,False

def _opt_card_id(o,hand,my_active,bench):
    ot=o.get('type')
    if ot in(PLAY,ATTACH,EVOLVE,TOOL_CARD,ENERGY_CARD):
        idx=o.get('index')
        if idx is not None and 0<=idx<len(hand): return _pk_id(hand[idx])
        return -1
    if ot==ABILITY:
        area=o.get('area'); idx=o.get('index',0)
        if area==4: return _pk_id(my_active)
        if area==5 and 0<=idx<len(bench): return _pk_id(bench[idx])
        return -1
    return -1
def _attach_target(o,my_active,bench):
    ta=o.get('inPlayArea',-1); ti=o.get('inPlayIndex',0)
    if ta==4: return my_active
    if ta==5 and bench and 0<=ti<len(bench): return bench[ti]
    return None
def _clamp(indices,sel):
    mn=sel.get('minCount',0) or 0; mx=sel.get('maxCount',1) or 1
    n=len(sel.get('option',[])); out=[]
    for i in indices:
        if 0<=i<n and i not in out: out.append(i)
        if len(out)>=mx: break
    i=0
    while len(out)<mn and i<n:
        if i not in out: out.append(i)
        i+=1
    return out

def _census(my_active, bench):
    all_mine=[my_active]+list(bench)
    abra_count=sum(1 for p in all_mine if _pk_id(p)==ABRA)
    genesect_with_tool = any(
        _pk_id(b)==GENESECT and (b or {}).get('tools')
        for b in bench if b)
    return {
        'abra_count':     abra_count,
        'backup_abra':    abra_count>=2,
        'line_count':     sum(1 for p in all_mine if _pk_id(p) in ALAKAZAM_LINE_IDS),
        'draw_count':     sum(1 for p in all_mine if _pk_id(p) in DRAW_ENGINE_IDS),
        'dudun_bench':    sum(1 for p in bench if _pk_id(p)==DUDUNSPARCE),
        'need_line':      sum(1 for p in all_mine if _pk_id(p) in ALAKAZAM_LINE_IDS)<2,
        'need_draw':      sum(1 for p in all_mine if _pk_id(p) in DRAW_ENGINE_IDS)<2,
        'has_alakazam':   any(_pk_id(p)==ALAKAZAM for p in all_mine),
        'kadabra_can_evolve': any(
            _pk_id(p)==KADABRA and not (p or {}).get('appearThisTurn',False)
            for p in all_mine if p),
        'dudun_no_energy': any(
            _pk_id(p)==DUDUNSPARCE and not _energies(p)
            for p in bench if p),
        'has_energy_plan': any(_has_psychic(p) for p in all_mine if p),
        'bench_count':    len([b for b in bench if b]),
        'has_psyduck':    any(_pk_id(b)==PSYDUCK for b in bench if b),
        'has_shaymin':    any(_pk_id(b)==SHAYMIN for b in bench if b),
        'has_genesect':   any(_pk_id(b)==GENESECT for b in bench if b),
        'has_fez':        any(_pk_id(b)==FEZ for b in bench if b),
        'genesect_active': genesect_with_tool,
        'active_is_kadabra': _pk_id(my_active)==KADABRA,
    }

PHASE_ESTABLISH=1; PHASE_CONVERT=2; PHASE_PRESSURE=3; PHASE_CLOSING=4

def _detect_phase(cen,can_ko,at_threshold,opp_prizes_left,hand_n):
    if opp_prizes_left<=2: return PHASE_CLOSING
    # NOTE: backup_abra (2+ Abra) and draw_count (Dunsparce/Dudunsparce currently in
    # play) are deliberately excluded here. Both are ephemeral, self-cycling resources
    # — Dudunsparce's own ability shuffles ITSELF back into the deck on use, so
    # draw_count routinely drops to 0 mid-game the instant you use the engine, and a
    # backup Abra is rarely available once it's been used climbing the evolution line.
    # Gating phase progression on either kept whole games stuck in ESTABLISH (and its
    # overdraw-permissive scoring) even with a fully-fueled, attacking Alakazam and a
    # huge hand. The only real gate for "established" is: is Alakazam up and fed.
    not_established=(not cen['has_alakazam'] or not cen['has_energy_plan'])
    if not_established: return PHASE_ESTABLISH
    if can_ko or at_threshold: return PHASE_PRESSURE
    return PHASE_CONVERT

def _score_setup_active(obs,opts):
    # v22 fix: options never carry cardId — they are {type:CARD, area:2(HAND), index}.
    # Resolve through our hand. (v21 read o['cardId'], which is always None, so every
    # option scored the default 20 and it silently picked opts[0] every game.)
    PREF={ABRA:90,DUNSPARCE:100,DUNSPARCE2:100,PSYDUCK:40,SHAYMIN:30,GENESECT:25,FEZ:5}
    cur=obs.get('current') or{}; me=cur.get('yourIndex',0)
    players=cur.get('players',[])
    hand=_hand_list(players[me]) if len(players)>me else []
    scores=[]
    for o in opts:
        idx=o.get('index')
        cid=_pk_id(hand[idx]) if idx is not None and 0<=idx<len(hand) else -1
        scores.append(float(PREF.get(cid,20)))
    return scores

def _pick_setup_active(obs,opts):
    if not opts: return[0]
    s=_score_setup_active(obs,opts)
    return[max(range(len(s)),key=lambda i:s[i])]

def _score_deck_search(obs,sel):
    """v22: deck searches are NOT blind — sel['deck'] lists the deck and each option's
    'index' points into it. Score candidates by what the board actually needs instead
    of taking the first N options (which is what the old generic fallback did)."""
    deck=sel.get('deck') or []
    opts=sel.get('option',[])
    cur=obs.get('current') or{}; me=cur.get('yourIndex',0)
    players=cur.get('players',[])
    my=players[me] if len(players)>me else {}
    bench=my.get('bench') or[]; my_active=_active(my)
    hand=_hand_list(my); hand_ids=[_pk_id(c) for c in hand]
    cen=_census(my_active,bench)
    candy_in_hand=RARE_CANDY in hand_ids
    kadabra_in_hand=KADABRA in hand_ids
    alak_in_hand=ALAKAZAM in hand_ids
    abra_in_hand=ABRA in hand_ids
    # cen['line_count'] here is entirely Abra/Kadabra (has_alakazam is false in this
    # branch) -- i.e. whether we have anything IN PLAY to promote into an Alakazam.
    # Deliberately excludes kadabra_in_hand: a Kadabra card sitting in hand with no
    # Abra in play (or hand) can't be played AT ALL -- it isn't a Basic, and Rare
    # Candy also needs a Basic already in play to Candy from. It's just as dead as
    # an Alakazam in that state, so it doesn't count as "having a line piece"
    # either (confirmed via replay 83461698: Hilda's search offered Alakazam/
    # Kadabra/Dudunsparce with zero Abra anywhere -- both evolution stages were
    # equally unplayable that turn).
    have_line_piece=cen['line_count']>0 or abra_in_hand
    def card_score(cid):
        if cid==ALAKAZAM:
            if not cen['has_alakazam'] and not alak_in_hand:
                # Fetching Alakazam with no Abra/Kadabra anywhere (play or hand) to
                # evolve it from is dead weight in hand -- grab the line piece
                # instead (Abra scores 90 below when abra_count==0). Confirmed via
                # replay: Poke Pad fetched Alakazam turn 1 with zero Abra in play
                # or hand, leaving it stuck unplayable.
                return 95.0 if have_line_piece else 20.0
            return 40.0
        if cid==KADABRA:
            if cen['abra_count']>0 and not kadabra_in_hand and not candy_in_hand: return 70.0
            return 30.0
        if cid==ABRA:
            if cen['abra_count']==0: return 90.0
            if not cen['backup_abra']: return 60.0
            return 25.0
        if cid in(DUNSPARCE,DUNSPARCE2):
            if cen['draw_count']==0: return 55.0
            return 20.0
        if cid==DUDUNSPARCE:
            if cen['draw_count']>0 and cen['dudun_bench']==0: return 50.0
            return 18.0
        if cid in PSYCHIC_ENERGY_IDS:
            if not cen['has_energy_plan']: return 85.0
            return 35.0
        if cid==ENRICHING: return 22.0
        if cid==PSYDUCK:  return 12.0
        if cid==SHAYMIN:  return 10.0
        if cid==GENESECT: return 11.0
        if cid==FEZ:      return 4.0
        if cid==RARE_CANDY: return 28.0
        if cid==BOSS:     return 26.0
        return 15.0
    scores=[]
    for o in opts:
        di=o.get('index')
        cid=_pk_id(deck[di]) if di is not None and 0<=di<len(deck) else -1
        scores.append(card_score(cid))
    return scores

def _deck_search_pick(obs,sel):
    opts=sel.get('option',[])
    scores=_score_deck_search(obs,sel)
    order=sorted(range(len(opts)),key=lambda i:-scores[i])
    mn=sel.get('minCount',0) or 0; mx=sel.get('maxCount',1) or 1
    picks=order[:mx]
    return _clamp(picks,sel) if len(picks)>=mn else _clamp(list(range(len(opts))),sel)

def _score_energy_discard(obs,sel):
    """v22: Enhanced Hammer / discard-energy selects (stype=4). Options carry
    area/index/energyIndex into the target pokemon's energyCards. Prefer discarding
    Mist/Rocky (the Powerful Hand blockers); otherwise keep the old first-pick."""
    cur=obs.get('current') or{}; me=cur.get('yourIndex',0)
    players=cur.get('players',[])
    opts=sel.get('option',[])
    scores=[]
    for o in opts:
        pidx=o.get('playerIndex'); area=o.get('area'); idx=o.get('index',0)
        ei=o.get('energyIndex',0)
        eid=None
        if pidx is not None and pidx<len(players):
            pl=players[pidx]
            pk=_active(pl) if area==4 else ((pl.get('bench') or[None]*5)[idx] if idx<len(pl.get('bench') or[]) else None)
            ecs=(pk or{}).get('energyCards') or []
            eid=ecs[ei].get('id') if 0<=ei<len(ecs) else None
        scores.append(100.0 if eid in (MIST_ENERGY,ROCK_ENERGY) else 1.0)
    return scores

def _pick_energy_discard(obs,sel):
    opts=sel.get('option',[])
    scores=_score_energy_discard(obs,sel)
    order=sorted(range(len(opts)),key=lambda i:-scores[i])
    return _clamp(order,sel)

def _score_evolve_target(obs,sel):
    """v22: Rare Candy target select (stype=7, ctx=EVOLVE). Options carry
    inPlayArea/inPlayIndex (which Abra). Prefer the ACTIVE Abra — it's already
    positioned to attack, so Candying it gets an attacker online immediately,
    whereas Candying a bench Abra still needs a promotion/retreat afterward.
    Psychic-fueled is a secondary tiebreak on top of that."""
    cur=obs.get('current') or{}; me=cur.get('yourIndex',0)
    players=cur.get('players',[])
    my=players[me] if len(players)>me else {}
    bench=my.get('bench') or[]; my_active=_active(my)
    opts=sel.get('option',[])
    scores=[]
    for o in opts:
        tgt=_attach_target(o,my_active,bench)
        s=0.0
        if o.get('inPlayArea')==4: s+=15.0
        if _has_psychic(tgt): s+=10.0
        scores.append(s)
    return scores

def _pick_evolve_target(obs,sel):
    if not sel.get('option'): return[0]
    s=_score_evolve_target(obs,sel)
    return[max(range(len(s)),key=lambda i:s[i])]

def _pick_setup_bench(opts): return list(range(len(opts)))

def _score_bench_target(obs,opts):
    cur=obs.get('current') or{}; me_idx=cur.get('yourIndex',0)
    players=cur.get('players',[]); me=players[me_idx] if players and len(players)>me_idx else{}
    bench=me.get('bench') or[]
    area5=[(i,o) for i,o in enumerate(opts) if o.get('area')==5]
    scores=[-1e9]*len(opts)
    for order,(i,o) in enumerate(area5):
        idx=o.get('index',order)
        pk=bench[idx] if 0<=idx<len(bench) else(bench[order] if order<len(bench) else None)
        pid=_pk_id(pk)
        # Promoting after a KO is a hard forced pick with no do-over — an un-evolved
        # line piece (Kadabra especially) is a much better bet than a pure-support
        # mon (Psyduck/Fez/Genesect/Dunsparce) that can never become the attacker.
        # These used to share the same -10 fallback, so ties broke on array order
        # instead of board value (confirmed losing this exact coinflip in a replay:
        # promoted Psyduck over an already-energized Kadabra sitting right next to it).
        if pid==ALAKAZAM and _has_psychic(pk): s=100
        elif pid==ALAKAZAM: s=80
        elif pid==KADABRA and _has_psychic(pk): s=70
        elif pid==KADABRA: s=55
        elif pid==DUDUNSPARCE: s=50
        elif pid in PIVOT_FREE_RETREAT_IDS: s=40
        elif pid==ABRA and _has_psychic(pk): s=30
        elif pid==ABRA: s=20
        else: s=-10
        scores[i]=s
    return scores

def _score_wondrous_patch_target(obs,opts):
    """Wondrous Patch attaches a recovered Basic {P} Energy to the selected benched
    {P} Pokemon -- the OPPOSITE tiebreak from retreat/promotion targeting (which
    wants whoever can ALREADY attack): here we want whoever NEEDS the energy.
    Reusing _score_bench_target for this (both are stype==1, area==5 selects) would
    prefer an already-fueled Alakazam over an unfueled Kadabra/Abra, wasting the
    attach. Distinguished via sel['effect']['id']==WONDROUS_PATCH -- confirmed via
    replay that this is unambiguous (plain retreat/promotion selects have effect=
    None or a different card's id, e.g. Boss's Orders' 1182)."""
    cur=obs.get('current') or{}; me_idx=cur.get('yourIndex',0)
    players=cur.get('players',[]); me=players[me_idx] if players and len(players)>me_idx else{}
    bench=me.get('bench') or[]
    scores=[-1e9]*len(opts)
    for i,o in enumerate(opts):
        if o.get('area')!=5: continue
        idx=o.get('index',i)
        pk=bench[idx] if 0<=idx<len(bench) else None
        pid=_pk_id(pk)
        if _has_psychic(pk): s=-10.0  # already fueled -- this attach would be wasted
        elif pid==ALAKAZAM: s=100.0
        elif pid==KADABRA:  s=70.0
        elif pid==ABRA:     s=40.0
        else: s=0.0
        scores[i]=s
    return scores

def _pick_wondrous_patch_target(obs,opts):
    if not opts: return[0]
    s=_score_wondrous_patch_target(obs,opts)
    return[max(range(len(s)),key=lambda i:s[i])]

def _pick_bench_target(obs,opts):
    if not opts: return[0]
    s=_score_bench_target(obs,opts)
    return[max(range(len(s)),key=lambda i:s[i])]

def _score_boss_target(obs,sel):
    """Standalone scoring mirror of _pick_boss_target's tiered KO > damage >
    mist-KO > fallback logic, for use as an MCTS prior (score_options). Kept as
    a SEPARATE function rather than refactoring _pick_boss_target into a thin
    wrapper around it — Boss's Orders targeting is too game-critical to risk
    subtle drift from collapsing tiered tie-break logic into one scalar via
    lexicographic weight encoding. _pick_boss_target's own decision logic is
    untouched. Encoding: tiers are separated by 1e9 (KO=3e9, damage=2e9,
    mist-KO=1e9); within a tier, weights (1e5, 1e2, 1) are wide enough apart
    that pv<=3, hp<1000, and energy-count<100 can never bleed into the next
    weight's digit range, so tie-break ordering exactly matches the tuple-key
    max() used by _pick_boss_target."""
    cur=obs.get('current') or{}; me=cur.get('yourIndex',0)
    players=cur.get('players',[]); opp_idx=1-me
    opp=players[opp_idx] if len(players)>opp_idx else{}
    opp_bench=opp.get('bench') or[]
    hand_n=_hand_size(cur,me); boss_dmg=(hand_n-1)*PH_DMG_PER_CARD
    opts=sel.get('option',[])
    scores=[0.0]*len(opts)
    any_candidate=False
    for i,o in enumerate(opts):
        bi=o.get('index',0)
        pk=opp_bench[bi] if 0<=bi<len(opp_bench) else None
        if not pk:
            scores[i]=-1e9; continue
        any_candidate=True
        pk_hp=(pk.get('hp',99999) or 99999); pv=_prize_value_pk(pk)
        ec=len((pk or{}).get('energies') or [])
        pid=_pk_id(pk)
        pk_walled=_opp_has_blocking_energy(pk)
        if boss_dmg>=pk_hp and not pk_walled:
            scores[i]=3e9 + pv*1e5 + min(pk_hp,999)*1e2 + min(ec,99)
        elif boss_dmg>=pk_hp:
            scores[i]=1e9 + pv*1e5 + min(pk_hp,999)*1e2 + min(ec,99)
        elif not pk_walled:
            dmg_pct=min(boss_dmg/pk_hp, 1.0) if pk_hp>0 else 0
            threat_score=100 if pid==ALAKAZAM else(40 if pid==KADABRA else(30 if pid in PIVOT_FREE_RETREAT_IDS else 10))
            scores[i]=2e9 + pv*300 + dmg_pct*100 + threat_score
        else:
            scores[i]=-1e9
    if not any_candidate:
        scores=[0.0]+[-1e9]*(len(opts)-1) if opts else []
    return scores

def _pick_boss_target(obs,sel):
    cur=obs.get('current') or{}; me=cur.get('yourIndex',0)
    players=cur.get('players',[]); opp_idx=1-me
    opp=players[opp_idx] if len(players)>opp_idx else{}
    opp_bench=opp.get('bench') or[]
    hand_n=_hand_size(cur,me); boss_dmg=(hand_n-1)*PH_DMG_PER_CARD
    opts=sel.get('option',[]); ko_targets=[]; dmg_targets=[]; mist_ko_targets=[]
    for i,o in enumerate(opts):
        bi=o.get('index',0)
        pk=opp_bench[bi] if 0<=bi<len(opp_bench) else None
        if not pk: continue
        pk_hp=(pk.get('hp',99999) or 99999); pv=_prize_value_pk(pk)
        ec=len((pk or{}).get('energies') or [])
        pid=_pk_id(pk)
        # Gusting a Pokemon that ALSO has Mist/Rocky Energy just recreates the wall —
        # only counts as a real KO/damage option if the wall isn't there too.
        pk_walled=_opp_has_blocking_energy(pk)
        if boss_dmg>=pk_hp and not pk_walled:
            ko_targets.append((i,pv,pk_hp,ec))
        elif boss_dmg>=pk_hp:
            mist_ko_targets.append((i,pv,pk_hp,ec))
        elif not pk_walled:
            dmg_pct=min(boss_dmg/pk_hp, 1.0) if pk_hp>0 else 0
            dmg_targets.append((i,pv,pid,dmg_pct,pk_hp))
    if ko_targets:
        best=max(ko_targets,key=lambda x:(x[1],x[2],x[3]))
        return[best[0]]
    if dmg_targets:
        def dmg_value(t):
            i,pv,pid,dmg_pct,hp=t
            threat_score=0
            if pid==ALAKAZAM: threat_score=100
            elif pid==KADABRA: threat_score=40
            elif pid in PIVOT_FREE_RETREAT_IDS: threat_score=30
            else: threat_score=10
            return pv*300 + dmg_pct*100 + threat_score
        best=max(dmg_targets,key=dmg_value)
        return[best[0]]
    # Every bench option is also Mist/Rocky-walled — no choice makes progress, so fall
    # back to the highest-prize KO available (denies the biggest investment at least).
    if mist_ko_targets:
        best=max(mist_ko_targets,key=lambda x:(x[1],x[2],x[3]))
        return[best[0]]
    return[0]

def _main_phase_features(obs,sel):
    """Computes every board-state local the MAIN-phase scorer needs, then
    returns the `score(o)` closure itself (unchanged body, just relocated so
    it's reachable outside the argmax-and-return control flow that used to
    own it). This lets score_options_main(obs,sel) expose a per-option score
    vector — the heuristic side of the MCTS prior blend — via one call,
    instead of re-deriving this analysis inline. Pure: no I/O, no RNG, no
    mutation of _STALL_MEMO or any other global (search may call this
    thousands of times per game)."""
    opts=sel['option']
    cur=obs.get('current') or{}; me_idx=cur.get('yourIndex',0)
    players=cur.get('players',[])
    my=players[me_idx]   if players and len(players)>me_idx else{}
    opp=players[1-me_idx] if players and len(players)==2     else{}
    supporter_played=cur.get('supporterPlayed',False)
    energy_attached =cur.get('energyAttached',False)
    retreated       =cur.get('retreated',False)
    my_active=_active(my); opp_active=_active(opp)
    hand_n=_hand_size(cur,me_idx); hand=_hand_list(my); bench=my.get('bench') or[]
    lone=len([x for x in([my_active]+bench) if x])<=1
    cen=_census(my_active,bench)
    deck_count=my.get('deckCount',0) or 0
    discard=my.get('discard') or[]
    deck_critical=deck_count<10
    deck_danger  =deck_count<5
    prizes=len(my.get('prize') or[]); opp_prizes=len(opp.get('prize') or[])
    # Desperation: opponent can end the game on their next KO -- either they only
    # need 1 more prize, or they need 2 and we have an ex in play/bench (a single
    # KO on it hands over both at once). Confirmed via replay 83429870: at 1-1
    # prizes we lost trying to out-survive a lethal swing, when the actual winning
    # line was maxing hand size and Boss-sniping for the last prize instead (see
    # docs/report-log.md). Nothing here is about surviving longer -- if we can't
    # close it out this turn, next turn is likely a loss anyway, so deck-out risk
    # stops mattering relative to just building toward lethal.
    we_have_ex=any((p or{}).get('ex',False) for p in([my_active]+bench) if p)
    desperation=opp_prizes<=1 or(opp_prizes<=2 and we_have_ex)
    opp_played, bench_dmg_received, we_were_kod, opp_ace_observed = _analyze_logs(obs, me_idx)
    opp_mist=_opp_has_blocking_energy(opp_active)
    # ---- Stage 3 Phase C: belief-driven reads (docs/belief-model.md §Phase C) ----
    turn_n=cur.get('turn') or 0
    belief_post,opp_wall_revealed,opp_crustle_seen=_belief_posterior(opp,turn_n)
    if belief_post:
        b_top=max(belief_post,key=belief_post.get); b_conf=belief_post[b_top]
    else:
        b_top=None; b_conf=0.0
    # Ace-spec read: observed (logs) beats inferred; a low-confidence read keeps
    # the pre-Phase-C conservative True default; a confident read uses the
    # archetype's real ladder ace-spec reveal rate (679-replay tech survey).
    opp_likely_ace=opp_ace_observed or b_conf<0.97 or _ACE_RATE.get(b_top,1.0)>=0.35
    opp_hp=(opp_active or{}).get('hp',99999) or 99999
    attack_available=any(o.get('type')==ATTACK for o in opts)
    active_is_alak=_pk_id(my_active)==ALAKAZAM
    active_can_attack=active_is_alak and attack_available
    active_non_atk=_is_non_attacker(my_active)
    alak_stuck=active_is_alak and not attack_available
    bench_has_alak=any(_pk_id(b)==ALAKAZAM for b in bench if b)
    bench_has_alak_ready=any(_can_ph(b) for b in bench if b)
    bench_has_attacker=any(_pk_id(b) in{ALAKAZAM,KADABRA} for b in bench if b)
    my_dmg=(PH_DMG_PER_CARD*hand_n) if active_can_attack and not opp_mist else 0
    can_ko=active_can_attack and opp_active is not None and my_dmg>=opp_hp and not opp_mist
    opp_bench=opp.get('bench') or[]
    opp_bench_empty=len([b for b in opp_bench if b])==0
    opp_bench_wall=any(_opp_has_blocking_energy(b) for b in opp_bench if b)
    # Mist/Rocky ANTICIPATION (not just reaction, which opp_mist covers): real-
    # ladder walls live almost entirely in crustle decks and the unrecognized
    # long tail (tech survey 2026-07-04: crustle 35.8%, unknown 29.0%, the 5
    # classifier archetypes 0-3%). Threat = wall energy already revealed
    # anywhere on their side, a Crustle/Dwebble line, or a deck the classifier
    # can't confidently place by turn 2+. On belief failure (belief_post None)
    # this stays reactive-only -- pre-Phase-C behavior.
    mist_threat=(opp_wall_revealed or opp_bench_wall or opp_crustle_seen or
                 (belief_post is not None and b_conf<0.97 and turn_n>=2))
    # Cheap, rough lookahead (deliberately NOT a real turn simulation): if the
    # opponent has nothing but their Active in play, a big enough hand ends
    # their turn staring at an empty board, which is worth spending reserved
    # draw sources on NOW rather than banking them for a hypothetical future
    # hand-disruption effect (the piloting-guide's normal "bank surplus draw"
    # advice) -- there's no bigger prize than winning the board back outright.
    # Headroom estimate: current hand + each ready Dudunsparce's Run Away Draw
    # (+3 each, piloting-guide §3) + the best not-yet-played supporter in hand
    # (Hilda's fetch chain or Dawn) + an unattached Enriching Energy's draw 4.
    _untapped_draw=3*cen['dudun_bench']
    if not supporter_played:
        if any(_pk_id(c)==HILDA for c in hand): _untapped_draw+=4
        elif any(_pk_id(c)==DAWN for c in hand): _untapped_draw+=2
    if any(_pk_id(c)==ENRICHING for c in hand): _untapped_draw+=3
    max_hand_estimate=hand_n+_untapped_draw
    lone_active_opportunity=(
        opp_bench_empty and opp_active is not None and opp_hp<99999 and
        not opp_mist and max_hand_estimate*PH_DMG_PER_CARD>=opp_hp)
    opp_active_pv=_prize_value_pk(opp_active)
    boss_in_hand=any(_pk_id(c)==BOSS for c in hand)
    hammer_in_hand=any(_pk_id(c)==ENHANCED_HAMMER for c in hand)
    tool_in_hand=any(_pk_id(c) in TOOL_IDS for c in hand)
    # Hilda's own search pool is Stage-1/2 (Kadabra/Alakazam) + energy only --
    # docs/piloting-guide.md confirms Dawn alone grabs "Basic + Stage 1 + Stage
    # 2." With zero line pieces in play and no Abra in hand, Hilda cannot fetch
    # the one card that actually unblocks us (confirmed via replay 83461698:
    # Hilda's own deck-search options only ever offered Alakazam/Kadabra/
    # Dudunsparce -- no Abra was even a legal choice). Dawn's flat ESTABLISH
    # weight already narrowly beats Hilda's (22 vs 24) in the scoring below, so
    # this needs to actively suppress Hilda here rather than rely on the
    # existing weight gap.
    abra_in_hand=any(_pk_id(c)==ABRA for c in hand)
    need_basic_abra=cen['line_count']==0 and not abra_in_hand
    # Opponent's Active is permanently walling Powerful Hand (Mist/Rocky Energy) and we
    # have no way left to answer it this turn (no Hammer to strip the energy, no Boss
    # to gust a different, unwalled target) — more searching/drawing can't fix a wall
    # made of card TYPE, only of hand size, so it's pure deck-out risk for zero payoff.
    hopelessly_walled=opp_mist and not hammer_in_hand and not boss_in_hand
    boss_dmg=(hand_n-1)*PH_DMG_PER_CARD
    ready_alak_exists=active_can_attack or bench_has_alak_ready
    opp_bench_ko_gte=any(
        0<(b or{}).get('hp',99999)<=boss_dmg and _prize_value_pk(b)>=opp_active_pv
        for b in opp_bench if b)
    boss_snipe_plan=(boss_in_hand and ready_alak_exists and opp_bench_ko_gte and opp_hp>boss_dmg)
    cards_needed=math.ceil(opp_hp/PH_DMG_PER_CARD) if opp_hp<99999 else 999
    at_threshold=active_can_attack and hand_n>=cards_needed and not opp_mist
    hand_too_small=hand_n<max(3,math.ceil(opp_hp/PH_DMG_PER_CARD/3))
    emergency_draw=hand_n<=4
    phase=_detect_phase(cen,can_ko,at_threshold,opp_prizes,hand_n)
    in_late_phase=phase in(PHASE_CONVERT,PHASE_PRESSURE,PHASE_CLOSING)
    # v23: Boss's Orders is a one-shot resource (3 copies) that should be SAVORED for
    # high-value bench snipes once the hand is actually built up, not spent early to
    # pick off any killable small-fry bench mon — that trades a scarce out for a target
    # that often wasn't threatening anything anyway. Require BOTH: (a) hand already
    # developed (in_late_phase — ESTABLISH means Alakazam isn't even fed yet, far too
    # early to burn a Boss) and (b) the target is actually worth gusting: an ex/mega
    # (2-3 prizes) or a beefy (150+ HP) Pokemon — not just "kills whatever's back there".
    boss_target_exists=(
        # opp_mist doesn't block Boss — it blocks Powerful Hand against the CURRENT
        # active only. A Mist-walled active makes ANY killable bench target the
        # correct play (it's the only way to make progress at all), not a reason to
        # skip this check in favor of the un-informed opp_mist fallback below.
        active_can_attack and (opp_mist or opp_hp>my_dmg) and in_late_phase and
        any(0<(b or{}).get('hp',99999)<=boss_dmg and
            (_prize_value_pk(b)>=2 or (b or{}).get('hp',99999)>=150) and
            _prize_value_pk(b)>=opp_active_pv
            for b in opp_bench if b))
    active_hp=(my_active or{}).get('hp',99999) or 99999
    active_max_hp=(my_active or{}).get('maxHp',999) or 999
    active_below_half=(active_max_hp-active_hp)>active_max_hp//2
    active_vulnerable=active_hp<60 or (active_below_half and opp_prizes<=3)
    boss_ex_snipe=(
        can_ko and boss_in_hand and not opp_mist and
        any(boss_dmg>=(pk.get('hp',99999) or 99999) and _prize_value_pk(pk)>opp_active_pv
            for pk in opp_bench if pk))
    boss_can_damage_mega=(
        boss_in_hand and not opp_mist and ready_alak_exists and
        any(boss_dmg>=50 and _prize_value_pk(pk)==3
            for pk in opp_bench if pk))
    alak_in_discard=any(_pk_id(c) in ALAKAZAM_LINE_IDS for c in discard)
    opp_bench_low_hp=any(0<(b or{}).get('hp',999)<=100 for b in opp_bench if b)
    enriching_on_dudun=any(_pk_id(b)==DUDUNSPARCE and _energies(b) for b in bench if b)
    active_kadabra_can_evolve=(
        _pk_id(my_active)==KADABRA and
        not (my_active or{}).get('appearThisTurn',False))
    active_abra_can_evolve=(
        _pk_id(my_active)==ABRA and
        not (my_active or{}).get('appearThisTurn',False))
    candy_playable=any(
        o.get('type')==PLAY and _opt_card_id(o,hand,my_active,bench)==RARE_CANDY
        for o in opts)
    retreat_available=any(o.get('type')==RETREAT for o in opts)
    active_free_retreat=_pk_id(my_active) in PIVOT_FREE_RETREAT_IDS
    # Energy only "frees" an immobile Active if it actually enables something:
    # Alakazam can attack with 1 Psychic regardless of bench; anyone else needs a
    # bench Alakazam that's ALREADY ready to attack to retreat INTO (retreating into
    # an un-fueled Kadabra/Abra/support mon still leaves you unable to attack this
    # turn, so it isn't a fix) — and a free-retreater (Shaymin) was never blocked by
    # energy in the first place, so attaching to it fixes nothing either way.
    active_immobile=(
        my_active is not None and not attack_available and not retreat_available and
        len(_energies(my_active))==0 and not active_free_retreat and
        (active_is_alak or bench_has_alak_ready))
    # Threshold discipline (§4/§10 piloting-guide): once a ready attacker exists and the
    # hand is already at the KO threshold, more draw is pure deck-out risk -> stop drawing.
    ready_attacker_exists=active_can_attack or bench_has_alak_ready
    # Board-thinning root cause (replays win_001/win_010): this gate previously required
    # ready_attacker_exists, so it never engaged mid-rebuild (attacker just KO'd, no bench
    # backup) -- exactly when hand was already 2-3x cards_needed from earlier banking.
    # Powerful Hand's damage is capped by opp_hp regardless of attacker readiness, so a
    # hand this far over cards_needed is already-wasted value no matter what; confirmed
    # both replays burn the deck to 0 in a single turn via Poffin/Dawn/Hilda/Poke Pad
    # spam at hand_n=16-23 vs cards_needed=7 while stuck on a non-attacker.
    hand_grossly_over=opp_hp<99999 and hand_n>=cards_needed+6
    hand_surplus=(
        (ready_attacker_exists or hand_grossly_over) and opp_hp<99999 and hand_n>=cards_needed and
        not boss_snipe_plan and not emergency_draw and not desperation and
        not lone_active_opportunity)
    # Down to a single Pokemon in play (no bench at all) is a distinct existential
    # risk regardless of prize lead or hand size: one KO on the Active with nothing
    # to promote is an instant loss. Confirmed losing exactly this way in a replay
    # (140 HP Alakazam active, empty bench, KO'd on the opponent's next turn with
    # a commanding prize lead otherwise) — surplus-hand and phase gating shouldn't
    # suppress rebuilding the bench once it's completely empty.
    bench_empty=cen['bench_count']==0
    # Manual evolve (Abra->Kadabra->Alakazam) banks Kadabra's own +2 Psychic Draw
    # that Rare Candy skips (Abra->Alakazam directly) -- +1 net card overall, per
    # docs/piloting-guide.md §3, but costs a turn's tempo. "Racing" = we have no
    # attacker in play at all AND either we're in danger, past the free-setup
    # phase, or already low on cards -- exactly when the guide says Candy is
    # worth its cost ("speed only... to land a turn-2 Alakazam, or to rebuild
    # after a KO"). Otherwise we have time to bank the extra card.
    # NOTE: deliberately uses active_below_half (relative), not active_vulnerable
    # (which has an active_hp<60 absolute clause -- always true for Abra, whose
    # max HP is 50, making it a false positive for exactly the Pokemon this
    # check most needs to evaluate correctly). Also deliberately excludes
    # emergency_draw (hand_n<=4) -- that's spuriously true turn 1-2 before the
    # draw engine has run at all, which is exactly when we DO have time to
    # climb manually, not an emergency. Both caught by direct testing before
    # this was trusted. phase in(PRESSURE,CLOSING) mostly reduces to "opponent
    # down to <=2 prizes" here, since _detect_phase forces ESTABLISH whenever
    # we have no Alakazam yet (its own not_established gate) except for that
    # opp_prizes_left<=2 short-circuit.
    # Offensive racing trigger: the two conditions above are purely defensive/phase-
    # based and blind to a simpler case -- Candying straight to Alakazam now sets up
    # a next-turn KO on the opponent's CURRENT active (Candy disables attacking the
    # turn it's played, so the actual attack happens next turn; current hand size is
    # the lethal-capacity proxy since it can only grow by then). Worth rushing even
    # turn 1-2, unlike the HP/phase triggers.
    candy_lethal_soon=(
        active_abra_can_evolve and not cen['has_alakazam'] and opp_active is not None and
        not opp_mist and hand_n*PH_DMG_PER_CARD>=opp_hp)
    racing_for_alakazam=(
        desperation or
        (not cen['has_alakazam'] and
         ((active_below_half and opp_prizes<=3) or phase in(PHASE_PRESSURE,PHASE_CLOSING) or
          candy_lethal_soon)))

    def score(o):
        ot=o.get('type'); cid=_opt_card_id(o,hand,my_active,bench)
        if ot==ATTACK:
            if not active_can_attack: return-5
            if opp_mist: return-5
            if can_ko: return 500
            if at_threshold: return W['atk_threshold']
            if hand_too_small: return 0.5
            return W['atk_default']
        if ot==RETREAT:
            if retreated: return-50
            if alak_stuck:
                if bench_has_alak_ready: return W['retreat_alak_stuck']
                return -5.0
            if active_non_atk and bench_has_alak_ready:
                return W['retreat_nonatk_ready'] if active_below_half else W['retreat_alak_stuck']
            if active_non_atk and bench_has_alak:
                return 20.0 if active_below_half else 16.0
            if active_non_atk:                     return -3.0
            if active_can_attack:                  return-2.0
            return 0.5
        if ot==ABILITY:
            if lone: return-10
            if cid in SUPPRESS_ABILITY_IDS: return-10
            if cid in DRAW_ABILITY_CARD_IDS:
                if can_ko: return 2.0
                if cid==DUDUNSPARCE:
                    if hand_surplus: return 0.5
                    if (desperation or lone_active_opportunity) and not emergency_draw: return W['desperation_draw']
                    if hopelessly_walled and not emergency_draw: return-6.0
                    if deck_danger and not emergency_draw:   return-8.0
                    if deck_critical and not emergency_draw: return-2.0
                    if hand_n>=14 and not emergency_draw:    return 1.0
                    if cen['dudun_bench']>1 and not emergency_draw: return 6.0
                    return W['dudun_base']
                if emergency_draw: return 15.0
                return 10.0
            if cid==FEZ:
                if deck_danger: return-5.0
                if hand_surplus: return-3.0
                if deck_critical and not emergency_draw: return-2.0
                return W['fez_base']
            return 5.0
        if ot==EVOLVE:
            evo_area=o.get('inPlayArea',4)
            if cid==ALAKAZAM:
                if evo_area==5:
                    if can_ko: return 3.0
                    if not cen['has_alakazam']: return 50.0
                    if phase==PHASE_ESTABLISH: return W['evo_bench_establish']
                    if phase==PHASE_CONVERT: return W['evo_bench_convert']
                    return W['evo_bench_late']
                post_evo_dmg=(hand_n+3)*PH_DMG_PER_CARD
                if phase==PHASE_ESTABLISH and active_kadabra_can_evolve:
                    return 300
                if not can_ko and post_evo_dmg>=opp_hp and active_kadabra_can_evolve:
                    return 280
                if active_kadabra_can_evolve:
                    return 260
                if can_ko: return 5.0
                if not cen['has_alakazam']: return 16.0
                return 10.0
            if cid==KADABRA:
                if can_ko: return 3.0
                if evo_area==4 and active_abra_can_evolve and candy_playable:
                    if racing_for_alakazam:
                        # Rare Candy can take THIS SAME active Abra straight to
                        # Alakazam this turn — normal-evolving to Kadabra first
                        # burns the turn's evolution on an intermediate stage and
                        # pushes Alakazam a full turn later. Worth it when we're
                        # racing (no attacker yet + in danger/past-establish/low
                        # on cards). Let Candy (scored below) win.
                        return 2.0
                    # Have time: bank Kadabra's own +2 Psychic Draw that Candy
                    # skips (net +1 card over the 2-turn climb, piloting-guide §3).
                    return 15.0
                return 13.0
            if can_ko: return 2.0
            if active_non_atk: return 12.0
            return 8.5
        if ot==PLAY:
            if cid==ENHANCED_HAMMER:
                if opp_mist: return W['hammer_mist']
                # Phase C: a wall energy parked on their BENCH is a wall being
                # prepared -- strip it before it promotes (the select scorer
                # already prefers Mist/Rocky targets).
                if opp_bench_wall: return 36.0
                return 3.0
            if cid==BATTLE_CAGE:
                if bench_dmg_received: return W['cage_reactive']
                if in_late_phase and hand_n>=8: return 1.0
                return W['cage_base']
            if cid==BOSS:
                if boss_ex_snipe:        return 600.0
                if can_ko: return 1.0
                # NOTE: phase==PHASE_CLOSING used to return a flat 199 here with NO
                # target-quality check -- it fired even with an empty/useless bench,
                # or (worse) when the CURRENT active was already our best KO target.
                # Confirmed losing real value this way (replay 83458785, step94):
                # active Alakazam had 0 energy (attack_available False that turn, so
                # boss_target_exists below is also False), opponent's active was
                # Mega Starmie ex at 110/430 (already the best target on their
                # board) -- Boss's Orders still fired at 199, yanking it off active
                # in favor of a fresh 70-HP Staryu, undoing our own damage progress
                # for a 1-prize KO instead of the lined-up 2-3-prize one. Fold the
                # phase bump INTO boss_target_exists (which already verifies a real,
                # worthwhile bench target and that we can actually act this turn)
                # instead of firing unconditionally.
                if boss_target_exists:
                    return 199.0 if phase==PHASE_CLOSING else W['boss_target']
                if boss_can_damage_mega:
                    # Phase C Mist anticipation: mega chip damage is optional;
                    # vs wall-teching decks Boss is the escape of last resort,
                    # so hold it until the wall question is settled.
                    return 4.0 if (mist_threat and not opp_mist) else W['boss_mega_chip']
                if opp_mist and ready_alak_exists: return W['boss_mist_escape']
                return 4.0
            if can_ko: return 1.0
            if cid in BENCHABLE_BASIC_IDS:
                bc=cen['bench_count']
                if cid==FEZ:
                    if bc>=5: return-5.0
                    if active_vulnerable and not cen['has_fez']: return 14.0
                    if opp_bench_low_hp and not cen['has_fez'] and cen['has_alakazam']: return 10.0
                    if we_were_kod and not cen['has_fez']: return 16.0
                    if bc==0: return 3.0
                    return -1.0
                if cid==ABRA:
                    if not cen['backup_abra']:  return 20.0
                    if bc<3:                    return 10.0
                    return 4.0
                if cid in(DUNSPARCE,DUNSPARCE2):
                    if cen['draw_count']==0:    return 18.0
                    if bc<3:                    return 9.0
                    return 3.0
                if cid==PSYDUCK:
                    opp_ability_threat = bool(opp_played & {131, 132, 133})  # Duskull/Dusclops/Dusknoir (Cursed Blast)
                    if opp_ability_threat and not cen['has_psyduck']: return 18.0
                    if phase==PHASE_ESTABLISH and bc<3 and not cen['has_psyduck']: return 7.0
                    return 1.5
                if cid==SHAYMIN:
                    if bench_dmg_received and not cen['has_shaymin']: return 16.0
                    if phase==PHASE_ESTABLISH and bc<3 and not cen['has_shaymin']: return 5.0
                    return 1.0
                if cid==GENESECT:
                    if cen['has_genesect']: return 1.0
                    if tool_in_hand and opp_likely_ace: return 11.0
                    if bc<2: return 6.0
                    return 2.0
                if bc==0: return 10.0
                if bc==1: return 7.0
                return 3.0
            if cid==RARE_CANDY:
                if active_abra_can_evolve:
                    if racing_for_alakazam:
                        # Get the already-positioned Active online NOW. Worth the
                        # skipped Kadabra +2 draw when racing (no attacker yet +
                        # in danger/past-establish/low on cards) — it avoids the
                        # wasted-tempo pattern of normal-evolving to Kadabra this
                        # turn and only Candying a *different* (bench) Abra later,
                        # which leaves the active stuck needing its own extra turn.
                        return W['candy_active_abra']
                    # Have time: manual evolve (scored above, 15.0) banks the
                    # extra card instead — piloting-guide §3's "hold Candy when
                    # you have time" principle. Still usable, just not dominant.
                    return 10.0
                if cen['kadabra_can_evolve']:
                    if at_threshold or phase in(PHASE_PRESSURE,PHASE_CLOSING): return W['candy_ready']
                    if phase==PHASE_ESTABLISH: return W['candy_estab']
                    return 8.0
                if not cen['has_alakazam']: return 28.0
                if cen['need_line']:        return 14.0
                return 4.0
            if cid==POFFIN:
                if desperation or lone_active_opportunity: return W['desperation_draw']
                if deck_danger: return-8.0
                if hopelessly_walled: return-3.0
                if bench_empty: return W['poffin_estab']
                if hand_surplus: return 2.0
                if phase==PHASE_ESTABLISH and (not cen['backup_abra'] or cen['draw_count']==0):
                    if cen['bench_count']<5: return W['poffin_estab']
                if not cen['backup_abra'] or cen['draw_count']==0:
                    if cen['bench_count']<4: return W['poffin_need']
                if cen['bench_count']<2: return 14.0
                if phase==PHASE_ESTABLISH: return 12.0
                if in_late_phase: return 2.0
                return 6.0
            if cid==SACRED_ASH:
                if deck_danger:   return 35.0
                if deck_critical: return 25.0
                if alak_in_discard and not cen['has_alakazam']: return 12.0
                if phase==PHASE_CLOSING: return 5.0
                return 2.0
            if cid==LANA:
                if alak_in_discard and not cen['has_alakazam']: return 15.0
                if alak_in_discard: return 8.0
                if phase==PHASE_CLOSING: return 5.0
                return 2.0
            if cid==DAWN:
                if supporter_played: return-5.0
                if desperation or lone_active_opportunity: return W['desperation_draw']
                if deck_danger: return-8.0
                if active_immobile: return-3.0
                if hopelessly_walled: return-3.0
                if hand_surplus: return 2.0
                if phase==PHASE_ESTABLISH and (cen['need_line'] or not cen['has_alakazam']): return W['dawn_estab']
                if boss_snipe_plan and not emergency_draw: return 1.0
                if deck_critical: return 1.0
                if hand_n>=12: return 2.0
                if emergency_draw: return 14.0
                if cen['need_line'] or not cen['has_alakazam']: return W['dawn_need']
                if phase==PHASE_CONVERT: return 8.0
                return 6.0
            if cid==HILDA:
                if supporter_played: return-5.0
                if desperation or lone_active_opportunity: return W['desperation_draw']
                if deck_danger: return-8.0
                if active_immobile: return W['hilda_immobile']
                if need_basic_abra: return 3.0
                if hopelessly_walled: return-3.0
                if hand_surplus: return 2.0
                if phase==PHASE_ESTABLISH and (cen['need_line'] or not cen['has_alakazam']): return W['hilda_estab']
                if boss_snipe_plan and not emergency_draw: return 1.0
                if deck_critical: return 1.0
                if not enriching_on_dudun and cen['draw_count']>0: return 11.0
                if not cen['has_alakazam']: return 13.0
                if emergency_draw: return 12.0
                if cen['need_line']: return 10.0
                if phase==PHASE_CONVERT: return 7.0
                return 5.0
            if cid==POKE_PAD:
                if desperation or lone_active_opportunity: return W['desperation_draw']
                if deck_danger: return-8.0
                if active_immobile: return-3.0
                if hopelessly_walled: return-3.0
                if bench_empty: return W['pad_no_backup']
                if hand_surplus: return 2.0
                if not cen['backup_abra']: return W['pad_no_backup']
                if cen['need_line']:       return 9.0
                if cen['need_draw']:       return 10.0
                if in_late_phase:          return 2.0
                return 5.0
            if cid==WONDROUS_PATCH:
                if not cen['has_energy_plan']: return 8.0
                if in_late_phase: return 2.0
                return 5.0
            if in_late_phase: return 1.5
            return 4.0
        if ot==SKILL: return 5.0
        if ot==ATTACH:
            if can_ko: return 0.5
            tgt=_attach_target(o,my_active,bench); tid=_pk_id(tgt)
            is_energy_card=cid in PSYCHIC_ENERGY_IDS or cid==ENRICHING
            if active_immobile and tgt is my_active and is_energy_card:
                # Free the stranded Active. Prefer real Psychic so a stuck Alakazam can
                # both retreat AND attack; Enriching still frees it but pays no {P}.
                # Tools (Handheld Fan etc.) provide zero Energy and do NOT belong here
                # — they can't pay a retreat cost or an attack cost.
                if cid in PSYCHIC_ENERGY_IDS: return 65.0
                # Enriching only buys retreat, never attack (see the ENRICHING block
                # below). If tgt is Alakazam and there's no ready bench Alakazam to
                # retreat INTO, unsticking it just swaps into an equally-unable-to-
                # attack Abra/Kadabra/support mon — no upside — while permanently
                # burning the turn's one attach on a card that can never power
                # Powerful Hand (confirmed replay 84136810: softlocked a 5-1-ahead
                # Alakazam attack-dead for the rest of a deck-out loss). Fall through
                # to the normal ENRICHING routing (Dudunsparce priority, Alakazam
                # vetoed) instead of the blanket rescue score.
                if tid==ALAKAZAM and cid==ENRICHING and not bench_has_alak_ready:
                    pass
                else:
                    return 55.0
            if cid==HANDHELD_FAN:
                if tid==GENESECT and not (tgt or{}).get('tools'): return 15.0
                return 1.5
            if cid==ENRICHING:
                # Enriching's "draw 4" fires unconditionally on attach (no may-use
                # prompt, unlike Kadabra/Alakazam's Psychic Draw) -- unlike every
                # other draw source in this file it had NO deck-safety gate at all.
                # Confirmed contributing to a real deck-out loss (replay 83156504):
                # attached at deck=5 with hand already at 18, dropping the deck to 1
                # in one action. Bigger single draw than Dawn/Hilda (2) or Poffin, so
                # gate on deck_critical (<10), not just deck_danger (<5).
                if deck_critical and not emergency_draw and not desperation and not lone_active_opportunity: return -6.0
                if tid==DUDUNSPARCE and cen['dudun_no_energy']: return 20.0
                if tid==DUDUNSPARCE:                             return 13.0
                # Enriching is Colorless — it can NEVER pay Powerful Hand's Psychic
                # cost. Attaching it to Alakazam (fueled or not) wastes the card:
                # fueled, it does nothing; unfueled, it still leaves Alakazam unable
                # to attack while consuming the energy-drop for the turn instead of
                # a real Psychic source. Never route it here (see CLAUDE.md).
                if tid==ALAKAZAM: return -8.0
                return 1.0
            if cid in PSYCHIC_ENERGY_IDS:
                # Powerful Hand costs exactly 1 Psychic — a 2nd energy on the SAME
                # physical card does nothing (damage scales with hand size, not
                # energy count), and unlike a bench pivot there's no cost this ever
                # pays down. Hard cap this at the target level, before the per-line
                # priority order below, so it can never lose to "nothing better
                # this turn" and get attached anyway — the card is strictly more
                # valuable left in hand as +20 future Powerful Hand damage.
                if _has_psychic(tgt): return -6.0
                # Route further Psychic to the next-best un-fueled pre-load target:
                # Alakazam (fuels the attacker itself) > Kadabra > Abra (so the
                # energy is already there the moment it evolves) > other bench
                # support (never preemptively fuel Dudunsparce/Genesect/Shaymin/
                # Psyduck/Fez — they don't attack, and the one legitimate case,
                # paying a real retreat cost into a waiting bench Alakazam, is
                # already handled above via the active_immobile rescue block).
                if tid==ALAKAZAM: return 16.0
                if tid==KADABRA:  return W['attach_kadabra']
                if tid==ABRA:     return W['attach_abra']
                return -2.0
            if active_non_atk:
                if tid==ALAKAZAM: return 11.0
                return 2.0
            if tid==ALAKAZAM: return 6.0
            return 3.0
        if ot==DISCARD: return 0.0
        if ot==END:
            if phase==PHASE_CONVERT and hand_n>=8: return W['end_convert']
            if phase==PHASE_PRESSURE and at_threshold: return 3.0
            return 1.0
        return 2.0

    return score

def score_options_main(obs,sel):
    """Standalone, reusable per-option heuristic score vector for MAIN-phase
    (stype==0) decisions — the heuristic half of the MCTS prior blend."""
    opts=sel.get('option') or[]
    if not opts: return[]
    score=_main_phase_features(obs,sel)
    return[score(o) for o in opts]

def _main_phase(obs,sel):
    opts=sel.get('option') or[]
    if not opts: return[]
    s=score_options_main(obs,sel)
    return[max(range(len(opts)),key=lambda i:s[i])]

def _choose(obs):
    sel=obs.get('select')
    if sel is None: return DECK
    opts=sel.get('option',[]); n=len(opts)
    if n==0: return[]
    stype=sel.get('type'); ctx=sel.get('context',0)
    mn=sel.get('minCount',0) or 0; mx=sel.get('maxCount',1) or 1
    if stype==0: return _main_phase(obs,sel)
    if stype==1:
        if ctx==CTX_SETUP_ACTIVE: return _pick_setup_active(obs,opts)
        if ctx==CTX_SETUP_BENCH:  return _pick_setup_bench(opts)
        cur=obs.get('current') or{}; me=cur.get('yourIndex',0)
        # v22: deck searches expose sel['deck'] (they are NOT blind) — options index
        # into it. Route any deck-area select through need-based scoring.
        if sel.get('deck') and any(o.get('area')==1 for o in opts):
            return _deck_search_pick(obs,sel)
        is_boss_target=(len(opts)>0
            and all(o.get('playerIndex')==(1-me) and o.get('area')==5 for o in opts))
        if is_boss_target: return _pick_boss_target(obs,sel)
        if (sel.get('effect') or{}).get('id')==WONDROUS_PATCH and any(o.get('area')==5 for o in opts):
            return _clamp(_pick_wondrous_patch_target(obs,opts),sel)
        if any(o.get('area')==5 for o in opts):
            return _clamp(_pick_bench_target(obs,opts),sel)
        yes_i=[i for i,o in enumerate(opts) if o.get('type')==YES]
        if yes_i: return _clamp(yes_i,sel)
        # Generic blind pick (this is where prize-card selection lands: same-shaped
        # option list, minCount==maxCount==KO'd Pokemon's prize value). If the engine
        # re-asks the identical question with no state change, rotate the pick instead
        # of resubmitting the same answer forever.
        return _clamp(_resolve_stalled_or(obs,sel,list(range(n))),sel)
    if stype==4: return _pick_energy_discard(obs,sel)
    if stype==7: return _clamp(_pick_evolve_target(obs,sel),sel)
    if stype in(2,3): return _clamp(list(range(n)),sel)
    if stype==5:           return _clamp(list(range(n))[:max(mn,1)],sel)
    if stype==6:
        _,A=_meta()
        return[max(range(n),key=lambda i:(A.get(opts[i].get('attackId'),(0,))[0] or 1))]
    if stype==8:
        return[max(range(n),key=lambda i:opts[i].get('number',0) or 0)]
    if stype==9:
        # "May use this Ability?" prompt (Psychic Draw, Run Away Draw, etc. when
        # asked this way rather than as a main-phase ABILITY option) -- every OTHER
        # draw source in this file (Dawn/Hilda/Poffin/Poke Pad/Dudunsparce-ability)
        # is gated on deck_danger, but this prompt always said yes unconditionally.
        # Confirmed root cause of a real ladder loss (replay 83348630): evolving a
        # bench Kadabra into Alakazam at deck=3 auto-triggered this prompt, and the
        # unconditional yes drew 3 more cards with a hand already at 17 (needing just
        # 7 for lethal) and a 5-2 prize lead -- decked out the same turn in a winning
        # game. Decline once the deck is already at the danger floor AND the hand is
        # already well past the KO threshold, since more cards can't help a banked
        # lethal and drawing is exactly how a winning position mills itself.
        cur=obs.get('current') or{}; me_idx=cur.get('yourIndex',0)
        players=cur.get('players',[])
        me=players[me_idx] if len(players)>me_idx else{}
        opp=players[1-me_idx] if len(players)==2 else{}
        if ctx==42:
            # MULLIGAN (ladder-only; local engine auto-resolves instead): cg-lib
            # api.py documents it as "Would you like to redraw the cards?". Both
            # ladder sightings (replays 83358041, 83455146) hit an opening hand
            # with zero Basic Pokemon. Redraw a dead hand (no Basic = cannot set
            # up), keep anything with a Basic -- the draw-prompt logic below
            # answers on deck count, not hand contents, and would say YES here.
            has_basic=any((c or{}).get('id') in BENCHABLE_BASIC_IDS
                          for c in me.get('hand') or[])
            want=NO if has_basic else YES
            pick=[i for i,o in enumerate(opts) if o.get('type')==want]
            if pick: return pick[:1]
            return[0]
        deck_count=me.get('deckCount',99) or 99
        hand_n=_hand_size(cur,me_idx)
        opp_hp=(_active(opp) or{}).get('hp',99999) or 99999
        cards_needed=math.ceil(opp_hp/PH_DMG_PER_CARD) if opp_hp<99999 else 999
        # Desperation (opponent one KO from winning, or two with an ex of ours in
        # play/bench) overrides the deck-out guard below: if we can't close it out
        # this turn, next turn is likely a loss anyway, so keep drawing toward the
        # biggest possible hand instead of preserving deck count. See the matching
        # `desperation` computation in _main_phase_features.
        opp_prizes=len(opp.get('prize') or[])
        bench=me.get('bench') or[]
        we_have_ex=any((p or{}).get('ex',False) for p in([_active(me)]+bench) if p)
        desperation=opp_prizes<=1 or(opp_prizes<=2 and we_have_ex)
        # Matching lone_active_opportunity from _main_phase_features: opponent has
        # nothing but their Active in play, so a big enough hand ends their turn
        # with an empty board -- worth drawing past the normal deck-out guard.
        opp_bench_empty=len([b for b in opp.get('bench') or[] if b])==0
        lone_active_opportunity=(
            opp_bench_empty and opp_hp<99999 and hand_n+3>=cards_needed)
        if deck_count<5 and hand_n>=cards_needed+3 and not desperation and not lone_active_opportunity:
            no_i=[i for i,o in enumerate(opts) if o.get('type')==NO]
            if no_i: return no_i
        for i,o in enumerate(opts):
            if o.get('type')==YES: return[i]
        return[0]
    if stype==10: return _clamp(list(range(n))[:max(mn,1)],sel)
    k=mn if mn>0 else(1 if mx>=1 else 0)
    return _clamp(list(range(n))[:k] if k else[],sel)

def score_options(obs,sel):
    """Per-option heuristic score vector for ANY select shape, mirroring
    _choose's dispatch. This is the heuristic half of the MCTS prior blend
    (docs/nn-training.md's heuristic-weighted-search plan) — pure, deterministic,
    side-effect-free (never touches _STALL_MEMO), safe to call repeatedly inside
    a search tree. Where no meaningful per-option ranking exists (bare YES/NO,
    forced picks, blind selections), returns a flat/uniform vector — softmax of
    a flat vector is a uniform prior, which is the honest answer there."""
    opts=sel.get('option') or[]
    n=len(opts)
    if n==0: return[]
    stype=sel.get('type'); ctx=sel.get('context',0)
    if stype==0: return score_options_main(obs,sel)
    if stype==1:
        if ctx==CTX_SETUP_ACTIVE: return _score_setup_active(obs,opts)
        if ctx==CTX_SETUP_BENCH:  return[0.0]*n
        cur=obs.get('current') or{}; me=cur.get('yourIndex',0)
        if sel.get('deck') and any(o.get('area')==1 for o in opts):
            return _score_deck_search(obs,sel)
        is_boss_target=(n>0 and all(o.get('playerIndex')==(1-me) and o.get('area')==5 for o in opts))
        if is_boss_target: return _score_boss_target(obs,sel)
        if (sel.get('effect') or{}).get('id')==WONDROUS_PATCH and any(o.get('area')==5 for o in opts):
            return _score_wondrous_patch_target(obs,opts)
        if any(o.get('area')==5 for o in opts):
            return _score_bench_target(obs,opts)
        return[0.0]*n
    if stype==4: return _score_energy_discard(obs,sel)
    if stype==7: return _score_evolve_target(obs,sel)
    return[0.0]*n

def _safe_return(result,sel):
    if not sel: return result
    n=len(sel.get('option',[])); mn=sel.get('minCount',0) or 0; mx=sel.get('maxCount',1) or 1
    if not isinstance(result,list): result=[0]
    result=[i for i in result if 0<=i<n][:mx]; i=0
    while len(result)<mn and i<n:
        if i not in result: result.append(i)
        i+=1
    return result if result else([0] if n>0 else[])

def agent(obs_dict: dict) -> list[int]:
    try:
        sel=obs_dict.get('select'); out=_choose(obs_dict)
        if isinstance(out,list): return _safe_return(out,sel) if sel else out
    except: pass
    try:
        sel=obs_dict.get('select')
        if not sel: return[]
        n=len(sel.get('option',[]))
        if n==0: return[]
        mn=sel.get('minCount',1) or 0
        return _safe_return(list(range(min(max(mn,1),n))),sel)
    except: return[0]

In [ ]:
%%writefile training/harness.py
"""Local game harness for the cabt engine (ships inside kaggle_environments).

Runs full games locally (~0.5-1s each) — no Kaggle session needed. Used by
ab_test.py (A/B evaluation), gauntlet.py, and all training/nn collectors.
Works on any machine with `pip install kaggle_environments --no-deps`
(--no-deps avoids a Windows long-path failure in an unrelated dependency).

Agent modules must expose `agent(obs_dict) -> list[int]` and `DECK` (60 ints).
"""
import importlib.util
import logging
import os
import sys
import time

# silence kaggle_environments' noisy env-registration logging before import
logging.disable(logging.INFO)
os.environ.setdefault("PYTHONWARNINGS", "ignore")

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def load_agent(path):
    """Import an agent module from a file path. Returns (agent_fn, deck, module)."""
    path = os.path.abspath(path)
    name = "agent_" + os.path.splitext(os.path.basename(path))[0] + "_" + str(abs(hash(path)) % 10**8)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod.agent, list(mod.DECK), mod


def play_game(agent0, deck0, agent1, deck1, keep_steps=False, max_steps=None):
    """Run one full game. Returns dict with rewards, steps, wall time, and
    (optionally) the full step trace for BC collection / analysis.
    max_steps caps runaway games (cabt's default episodeSteps is 10M, so two
    passive agents are otherwise bounded only by deck-out); a capped game ends
    as a tie."""
    from kaggle_environments import make

    t0 = time.time()
    config = {"decks": [deck0, deck1]}
    if max_steps:
        config["episodeSteps"] = max_steps
    env = make("cabt", configuration=config)
    env.run([agent0, agent1])
    wall = time.time() - t0
    last = env.steps[-1]
    result = {
        "rewards": [last[0].get("reward"), last[1].get("reward")],
        "statuses": [last[0].get("status"), last[1].get("status")],
        "n_steps": len(env.steps),
        "wall_s": wall,
    }
    if keep_steps:
        result["steps"] = env.steps
    return result


def _worker(job):
    """Multiprocessing worker: loads agents fresh in each process (module-level
    caches like main._STALL_MEMO stay isolated per game batch).

    job's optional 4th element (extra_env, a dict or None) is applied via
    os.environ.update() BEFORE loading either agent -- lets a caller assign
    a per-job identifier (e.g. a game_id an agent module can read at import
    time and embed in its own side-channel logging) without changing the
    job tuple shape for existing callers (extra_env defaults to None,
    meaning "no change," so every pre-existing 3-tuple job still works).

    job's optional 5th element (deck_override, a (deck0, deck1) tuple of
    list[int]-or-None) overrides load_agent's own DECK per side -- lets a
    caller vary the OPPONENT's deck per game (e.g. mcts_collect.py's
    opponent_pool.py sampling a different real-meta archetype deck per
    game for a deck-agnostic pilot like generic_pilot.py) without changing
    the job tuple shape for existing callers (defaults to None, meaning
    "use each agent's own module DECK," identical to prior behavior)."""
    path0, path1, keep_steps = job[0], job[1], job[2]
    extra_env = job[3] if len(job) > 3 else None
    deck_override = job[4] if len(job) > 4 else None
    if extra_env:
        os.environ.update(extra_env)
    a0, d0, _ = load_agent(path0)
    a1, d1, _ = load_agent(path1)
    if deck_override:
        d0 = deck_override[0] if deck_override[0] is not None else d0
        d1 = deck_override[1] if deck_override[1] is not None else d1
    try:
        result = play_game(a0, d0, a1, d1, keep_steps=keep_steps)
    except Exception as e:
        result = {"error": repr(e)}
    if extra_env:
        # imap_unordered returns results in completion order, not submission
        # order -- echo the job's own extra_env back so a caller using it to
        # tag jobs (e.g. a game_id) can tell which job a given result is for.
        result["extra_env"] = extra_env
    return result


def run_matches(path0, path1, n_games, workers=None, keep_steps=False, progress=True,
                 extra_envs=None, decks=None):
    """Play n_games of path0 vs path1 (seat order fixed — caller alternates).
    Returns list of result dicts.

    extra_envs: optional list of length n_games, each a dict of env vars (or
    None) applied inside that specific job's worker process before agents
    load -- e.g. mcts_collect.py uses this to assign a unique per-game
    MCTS_GAME_ID so a search-based collect log can be correlated to game
    outcomes correctly even when workers>1 interleaves multiple games'
    decisions across processes. None (default) preserves prior behavior
    exactly for every other caller.

    decks: optional list of length n_games, each a (deck0, deck1) tuple of
    list[int]-or-None (or None for "no override this game") -- overrides
    load_agent's own DECK per side, e.g. for a deck-agnostic pilot playing a
    specific real-meta archetype deck sampled by opponent_pool.py. None
    (default) preserves prior behavior exactly for every other caller."""
    import multiprocessing as mp

    if extra_envs is None:
        extra_envs = [None] * n_games
    if decks is None:
        decks = [None] * n_games
    jobs = [(path0, path1, keep_steps, extra_envs[i], decks[i]) for i in range(n_games)]
    results = []
    if workers is None:
        workers = max(1, (os.cpu_count() or 2) - 1)
    if workers <= 1:
        for i, job in enumerate(jobs):
            results.append(_worker(job))
            if progress and (i + 1) % 10 == 0:
                print(f"  {i+1}/{n_games}", file=sys.stderr)
    else:
        with mp.Pool(workers) as pool:
            for i, r in enumerate(pool.imap_unordered(_worker, jobs)):
                results.append(r)
                if progress and (i + 1) % 25 == 0:
                    print(f"  {i+1}/{n_games}", file=sys.stderr)
    return results


def summarize(results, name0="A", name1="B"):
    w = l = t = err = 0
    total_wall = 0.0
    for r in results:
        if "error" in r:
            err += 1
            continue
        total_wall += r["wall_s"]
        r0, r1 = r["rewards"][0], r["rewards"][1]
        # A crashed agent gets reward None while its opponent gets 1 (kaggle_environments
        # doesn't symmetrize this to -1/1) -- checking r0 alone silently miscounts those
        # as ties whenever the crash lands in slot 0. Confirmed via opponents/dragapult_agent.py,
        # which crashes on every local game (missing cg.api, Kaggle-dataset-only import) and
        # was showing up as ~50% ties instead of the real ~100% win rate.
        if r0 == 1 or r1 == -1:
            w += 1
        elif r0 == -1 or r1 == 1:
            l += 1
        else:
            t += 1
    n = w + l + t
    wr = (w + 0.5 * t) / n if n else 0.0
    return {
        "n": n, "errors": err,
        f"{name0}_wins": w, f"{name1}_wins": l, "ties": t,
        f"{name0}_winrate": round(wr, 4),
        "avg_game_s": round(total_wall / n, 2) if n else None,
    }


In [ ]:
%%writefile training/bc_collect.py
"""Collect behavior-cloning data from v22 self-play (and vs the opponent pool).

Each game's step trace is reduced to (obs_dict, chosen_action, player, outcome)
tuples for OUR seats and pickled. The NN training notebook converts these into
encoder/decoder tensors on Kaggle (where cg.api / all_card_data is available).

Usage:
  python training/bc_collect.py --games 700 --out bc_data.pkl [--opponent opponents/lucario_agent.py] [--workers N]

Default opponent is self (mirror). ~150+ decisions per game, so 700 games
≈ 100k+ samples. On 16 cores this is well under an hour.
"""
import argparse
import gzip
import os
import pickle
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from harness import load_agent, run_matches

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
MAIN = os.path.join(REPO_ROOT, "main.py")


def extract_decisions(steps, seat):
    """kaggle-env step semantics (reverse-engineered, see tools/analyze_replay.py):
    steps[i]['action'] resolves steps[i-1]['observation'].select's option list."""
    out = []
    for i in range(1, len(steps)):
        prev = steps[i - 1][seat].get("observation", {})
        act = steps[i][seat].get("action")
        sel = prev.get("select")
        if sel is None or act is None:
            continue
        if not sel.get("option"):
            continue
        out.append({"obs": prev, "action": act})
    return out


def write_shard(out, idx, samples):
    path = out if idx == 0 else out.replace(".pkl", f".part{idx}.pkl")
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "wb") as f:
        pickle.dump(samples, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"wrote {path} ({os.path.getsize(path)/1e6:.1f} MB, {len(samples)} samples)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--games", type=int, default=700)
    ap.add_argument("--out", default=os.path.join(REPO_ROOT, "training", "bc_data.pkl.gz"))
    ap.add_argument("--opponent", default=MAIN)
    ap.add_argument("--workers", type=int, default=None)
    args = ap.parse_args()

    print(f"Collecting {args.games} games: {MAIN} vs {args.opponent}")
    samples = []
    games = wins = 0
    shard_idx = total_samples = 0
    CHUNK = 100  # full step traces are heavy — extract and discard per chunk
    SHARD_SAMPLES = 100_000  # flush to disk so memory stays bounded
    remaining = args.games
    while remaining > 0:
        n = min(CHUNK, remaining)
        remaining -= n
        results = run_matches(MAIN, args.opponent, n, workers=args.workers,
                              keep_steps=True, progress=False)
        for r in results:
            if "error" in r or "steps" not in r:
                continue
            games += 1
            outcome = r["rewards"][0]  # our seat is 0
            if outcome == 1:
                wins += 1
            for d in extract_decisions(r["steps"], seat=0):
                d["outcome"] = outcome
                samples.append(d)
            # mirror games: seat 1 is also our agent — harvest it too
            if os.path.abspath(args.opponent) == os.path.abspath(MAIN):
                outcome1 = r["rewards"][1]
                for d in extract_decisions(r["steps"], seat=1):
                    d["outcome"] = outcome1
                    samples.append(d)
        print(f"  {games}/{args.games} games, {len(samples)} samples (shard {shard_idx})", file=sys.stderr)
        if len(samples) >= SHARD_SAMPLES:
            write_shard(args.out, shard_idx, samples)
            total_samples += len(samples)
            samples = []
            shard_idx += 1

    total_samples += len(samples)
    if samples or shard_idx == 0:
        write_shard(args.out, shard_idx, samples)
        shard_idx += 1
    print(f"games={games} our_p0_winrate={wins/max(games,1):.3f} "
          f"samples={total_samples} shards={shard_idx}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile training/generic_pilot.py
"""Deck-agnostic greedy pilot for the Stage 0c tier-2 (controlled) bake-off.

One fixed, deliberately simple policy piloting EVERY deck, so tier-2 rankings
reflect deck strength rather than pilot quality (pre-registered protocol,
docs/report-log.md 2026-07-03). Uses only deck-independent signals: the option
type codes of the current select (no card IDs, no archetype knowledge).

Greedy priority on the main action select: evolve > use abilities > play cards
> attach energy (active first) > attack (highest-index attack = the later,
usually stronger one) > end turn. Never retreats. Sub-selects take the first
minCount..maxCount options; yes/no prefers YES. A small stall guard falls back
to END / last option if the identical select repeats (mirrors the concern
main.py handles with _resolve_stalled_or).

DECK here is a placeholder so harness.load_agent() can import the module; the
bake-off passes each deck explicitly via bakeoff.py's orthogonal (agent, deck)
form.
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from main import DECK  # noqa: E402  (placeholder only — see docstring)

# option type codes (see tools/analyze_replay.py OT_NAMES)
YES, NO = 1, 2
PLAY, ATTACH, EVOLVE, ABILITY, RETREAT, ATTACK, END = 7, 8, 9, 10, 12, 13, 14

# greedy order for the main-phase select; anything unlisted ranks between
# ATTACK and END (neutral), RETREAT ranks below END (never chosen voluntarily)
_PRIORITY = {EVOLVE: 0, ABILITY: 1, PLAY: 2, ATTACH: 3, ATTACK: 5, END: 8, RETREAT: 9}
_DEFAULT_RANK = 7

_stall = {"fp": None, "count": 0}


def _fingerprint(sel):
    opts = sel.get("option", [])
    return (sel.get("type"), len(opts),
            tuple((o.get("type"), o.get("index"), o.get("attackId")) for o in opts[:12]))


def agent(obs_dict: dict) -> list:
    sel = obs_dict.get("select")
    if sel is None:
        return DECK
    opts = sel.get("option", [])
    n = len(opts)
    if n == 0:
        return []
    mx = sel.get("maxCount", 1) or 1
    mn = sel.get("minCount", 0) or 0
    k = max(mn, min(mx, n))

    fp = _fingerprint(sel)
    if fp == _stall["fp"]:
        _stall["count"] += 1
    else:
        _stall["fp"], _stall["count"] = fp, 0
    if _stall["count"] >= 4:
        # stuck repeating the same select: end turn if we can, else last option
        for i, o in enumerate(opts):
            if o.get("type") == END:
                return [i]
        return [n - 1]

    types = [o.get("type") for o in opts]

    # yes/no gates: take YES (proceed with the effect)
    if YES in types or NO in types:
        return [types.index(YES) if YES in types else types.index(NO)]

    if k == 1:
        def rank(i):
            o = opts[i]
            r = _PRIORITY.get(o.get("type"), _DEFAULT_RANK)
            tie = 0
            if o.get("type") == ATTACK:
                tie = -i          # prefer the later-listed (usually stronger) attack
            elif o.get("type") == ATTACH:
                tie = 0 if o.get("inPlayArea") == 4 else 1  # active before bench
            else:
                tie = i           # otherwise stable: first listed
            return (r, tie)
        return [min(range(n), key=rank)]

    # multi-pick sub-selects (choose k cards/targets): first k options
    return list(range(k))


In [ ]:
%%writefile training/nn/opponent_pool.py
"""Weighted real-meta opponent pool for Phase 2 self-play collection
(mcts_collect.py), replacing the too-weak same-checkpoint-vs-itself mirror
that caused severe win/loss label imbalance and collapsed the round-2 value
head (2026-07-07 — see docs/report-log.md "Root cause of the Phase 2 round 2
regression"). The searching side (mcts_leafeval_agent.py) needs opponents it
can actually lose to sometimes; a diverse, real-meta-weighted pool of the
project's existing rule-based archetype bots is a much stronger and more
representative source of that than the collecting net's own unsearched
policy.

Weights are the real ladder meta share (`tools/meta_survey.py --all`, 1595
replays, 2026-07-07): lucario 21.7%, alakazam-mirror 11.9%, dragapult 10.7%,
starmie 9.8%, crustle 8.7%, archaludon 6.6%, abomasnow 5.3%, grimmsnarl 3.5%,
bellibolt 1.3%, rockets-mewtwo 0.9%, kyogre 0.9%, raging-bolt 0.5%,
gardevoir 0.3%, other/unknown 18.1%. "other/unknown" has no reconstructable
deck/pilot and is dropped rather than fabricated; the rest is renormalized
to sum to 1. "kyogre" is also dropped despite having a real archetype-share
entry: its reconstructed decklist (training/archetype_decks.json, built
from only 13 replays' evidence) has just 30/60 card copies — too sparse to
reconstruct into a legal deck, so playing it would need fabricating the
other half.

Four archetypes (lucario, dragapult, abomasnow, starmie) are official Kaggle
sample bots with their own real decklist AND real piloting logic already in
opponents/*_agent.py — used as-is (deck=None -> the module's own DECK).
"alakazam_mirror" uses the real heuristic (main.py, the actual shipped
piloting logic, not the half-trained net's own weak policy) piloting its
own deck -- both a stronger opponent AND a more faithful proxy for what a
competent human Alakazam mirror opponent on the real ladder looks like.
Everything else has only a reconstructed decklist, no dedicated pilot --
piloted by training/generic_pilot.py (deck-agnostic greedy heuristic),
exactly as the Stage 0c tier-2 bake-off used it.
"""
import json
import os
import random

_HERE = os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.dirname(os.path.dirname(_HERE))
_OPPONENTS_DIR = os.path.join(_REPO_ROOT, "opponents")
_GENERIC_PILOT = os.path.join(_REPO_ROOT, "training", "generic_pilot.py")
_MAIN_PY = os.path.join(_REPO_ROOT, "main.py")

with open(os.path.join(_REPO_ROOT, "training", "archetype_decks.json"), encoding="utf-8") as _f:
    _ARCHETYPE_DECKS_RAW = json.load(_f)


def _flatten_deck(cards):
    deck = []
    for c in cards:
        deck.extend([c["cardId"]] * c["copies"])
    return deck


# (label, agent_path, deck_or_None, real-meta-share weight)
_POOL_RAW = [
    ("lucario", os.path.join(_OPPONENTS_DIR, "lucario_agent.py"), None, 21.7),
    ("alakazam_mirror", _MAIN_PY, None, 11.9),
    ("dragapult", os.path.join(_OPPONENTS_DIR, "dragapult_agent.py"), None, 10.7),
    ("starmie", os.path.join(_OPPONENTS_DIR, "starmie_agent.py"), None, 9.8),
    ("crustle", _GENERIC_PILOT, _flatten_deck(_ARCHETYPE_DECKS_RAW["crustle"]), 8.7),
    ("archaludon", _GENERIC_PILOT, _flatten_deck(_ARCHETYPE_DECKS_RAW["archaludon"]), 6.6),
    ("abomasnow", os.path.join(_OPPONENTS_DIR, "abomasnow_agent.py"), None, 5.3),
    ("grimmsnarl", _GENERIC_PILOT, _flatten_deck(_ARCHETYPE_DECKS_RAW["grimmsnarl"]), 3.5),
    ("bellibolt", _GENERIC_PILOT, _flatten_deck(_ARCHETYPE_DECKS_RAW["bellibolt"]), 1.3),
    ("rockets-mewtwo", _GENERIC_PILOT, _flatten_deck(_ARCHETYPE_DECKS_RAW["rockets-mewtwo"]), 0.9),
    ("raging-bolt", _GENERIC_PILOT, _flatten_deck(_ARCHETYPE_DECKS_RAW["raging-bolt"]), 0.5),
    ("gardevoir", _GENERIC_PILOT, _flatten_deck(_ARCHETYPE_DECKS_RAW["gardevoir"]), 0.3),
]
for _label, _path, _deck, _w in _POOL_RAW:
    if _deck is not None:
        assert len(_deck) == 60, f"{_label} deck has {len(_deck)} cards, expected 60"

_TOTAL_WEIGHT = sum(w for _, _, _, w in _POOL_RAW)
POOL = [(label, path, deck, w / _TOTAL_WEIGHT) for label, path, deck, w in _POOL_RAW]


def allocate(n_games, seed=0):
    """Deterministically splits n_games across the pool by weight (largest-
    remainder method, so every archetype gets its exact rounded share and
    the total is always exactly n_games). Returns list of (label, agent_path,
    deck_or_None, n_this_opponent), skipping zero-allocation entries."""
    raw = [(label, path, deck, w * n_games) for label, path, deck, w in POOL]
    base = [(label, path, deck, int(n)) for label, path, deck, n in raw]
    remainder = n_games - sum(n for _, _, _, n in base)
    fracs = sorted(range(len(raw)), key=lambda i: raw[i][3] - base[i][3], reverse=True)
    counts = [n for _, _, _, n in base]
    for i in fracs[:remainder]:
        counts[i] += 1
    return [(POOL[i][0], POOL[i][1], POOL[i][2], counts[i])
            for i in range(len(POOL)) if counts[i] > 0]


def shuffled_assignments(n_games, seed=0):
    """Returns a length-n_games list of (label, agent_path, deck_or_None),
    one per game, shuffled so consecutive games don't cluster by archetype
    (matters for run_matches's progress reporting and for any downstream
    per-chunk analysis, not for correctness)."""
    out = []
    for label, path, deck, n in allocate(n_games, seed):
        out.extend([(label, path, deck)] * n)
    random.Random(seed).shuffle(out)
    return out


In [ ]:
%%writefile training/nn/selfplay_collect.py
"""Self-play data collection: current net (temperature-sampling, for
exploration) plays full games against itself via the local engine. Computes
n-step bootstrapped value targets per docs/nn-training.md §Value Targets:

    G_t = sum_{k<n} gamma^k * r_{t+k} + gamma^n * V(s_{t+n})
    target = 0.7 * terminal_outcome + 0.3 * G_t

using the CURRENT net's own value head for V() (bootstrap) and a shaped
intermediate reward (prizes taken/conceded + hand-vs-KO-threshold progress,
see training/README.md §Curriculum & Reward Shaping). Saved samples add a
`value_target` field on top of the BC sample schema; dataset.py falls back to
plain `outcome` when that field is absent, so BC and SP shards share one
Dataset/collate implementation.

Usage:
  python selfplay_collect.py --games 300 --ckpt ../ptcg_bc_v1.pth --out ../sp_data.pkl.gz
"""
import argparse
import glob
import gzip
import math
import os
import pickle
import re
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))

from harness import run_matches
from bc_collect import extract_decisions
from net_common import load_model, value_estimate

GAMMA = 0.997
N_STEP = 10
TERMINAL_WEIGHT = 0.7


def _pk_hp(obs, me_idx, opp=True):
    cur = obs.get("current") or {}
    pl = cur.get("players") or []
    idx = (1 - me_idx) if opp else me_idx
    p = pl[idx] if len(pl) > idx else {}
    a = (p.get("active") or [None])
    active = a[0] if a and a[0] else None
    return (active or {}).get("hp", 99999) or 99999


def shaped_reward(obs_before, obs_after, me_idx):
    """One decision-to-next-own-decision shaped reward (the turn-level design
    from training/README.md adapted to our per-decision spacing)."""
    cur_b = obs_before.get("current") or {}
    cur_a = obs_after.get("current") or {}
    pl_b, pl_a = cur_b.get("players") or [], cur_a.get("players") or []
    if len(pl_b) < 2 or len(pl_a) < 2:
        return 0.0
    my_b, my_a = pl_b[me_idx], pl_a[me_idx]
    opp_b, opp_a = pl_b[1 - me_idx], pl_a[1 - me_idx]
    r = 0.0
    r += (len(opp_b.get("prize") or []) - len(opp_a.get("prize") or [])) * 1.0
    r -= (len(my_b.get("prize") or []) - len(my_a.get("prize") or [])) * 0.5
    opp_active = (opp_a.get("active") or [None])
    opp_active = opp_active[0] if opp_active and opp_active[0] else None
    opp_hp = (opp_active or {}).get("hp", 99999) or 99999
    hand_n = my_a.get("handCount") or len(my_a.get("hand") or [])
    needed = max(1, math.ceil(opp_hp / 20)) if opp_hp < 99999 else 999
    r += min(hand_n / needed, 1.0) * 0.1
    return r


def compute_value_targets(model, decisions, outcome, mcts_root_values=None):
    """decisions: list of {obs, action}. Adds 'value_target' in place.

    mcts_root_values: optional list (same length as decisions) of MCTS-backed-up
    root Q values, one per decision. Where a decision was actually searched
    (mcts_root_values[t] is not None), that replaces the raw value-head estimate
    as the bootstrap V(s_t) — a stronger leaf/bootstrap signal than the net's
    un-searched value head alone. Decisions without a tree search (None, or when
    this whole argument is omitted) fall back to the plain value head, exactly
    as before — this keeps selfplay_collect.py's direct self-play path
    (no search) working unchanged; mcts_collect.py (Kaggle-only, not yet built)
    is the intended caller that will pass real root values."""
    n = len(decisions)
    if n == 0:
        return
    if mcts_root_values is None:
        mcts_root_values = [None] * n
    values = [
        mcts_root_values[i] if mcts_root_values[i] is not None
        else value_estimate(model, d["obs"], d["obs"]["select"])
        for i, d in enumerate(decisions)
    ]
    rewards = [0.0] * n
    for t in range(n - 1):
        # 2026-07-06 fix: me_idx was hardcoded 0, but `decisions` comes from
        # extract_decisions(steps, seat=net_seat) and net_seat is 1 for half
        # of any collection run that alternates seats (as mcts_collect.py and
        # dmc_collect.py both do) -- for those games this silently computed
        # the shaped reward from the OPPONENT's perspective (swapping "my"
        # and "opp" prize/hand progress). Read the real seat from each
        # decision's own obs instead of assuming 0. Same bug class as the
        # one already found and fixed in dmc_nstep.py's _phi_at this session.
        me_idx = (decisions[t]["obs"].get("current") or {}).get("yourIndex", 0)
        rewards[t] = shaped_reward(decisions[t]["obs"], decisions[t + 1]["obs"], me_idx)
    for t in range(n):
        G = 0.0
        disc = 1.0
        end = min(t + N_STEP, n)
        for k in range(t, end):
            G += disc * rewards[k]
            disc *= GAMMA
        if end < n:
            G += disc * values[end]
        else:
            G += disc * outcome
        target = TERMINAL_WEIGHT * outcome + (1 - TERMINAL_WEIGHT) * G
        # value head is tanh-bounded [-1,1]; shaped-reward accumulation can
        # push raw targets slightly outside that range — clip so training
        # doesn't chase an unreachable label.
        decisions[t]["value_target"] = max(-1.0, min(1.0, target))
        decisions[t]["outcome"] = outcome
        # raw V(s_t) from the collecting net, stored so AWR training (Stage 2)
        # can compute advantage = value_target - v_pred without re-running the
        # value head at train time.
        decisions[t]["v_pred"] = values[t]


def _shard_path(out, idx):
    return out if idx == 0 else out.replace(".pkl", f".part{idx}.pkl")


def _next_shard_idx(out):
    """Scan for existing shards matching `out`'s naming so a second invocation
    (--out pointing at the same base path) ADDS new shards instead of
    overwriting shard 0 — supports the "collect a bit now, add more later"
    workflow without clobbering prior runs."""
    base = out[:-len(".pkl.gz")] if out.endswith(".pkl.gz") else out[:-len(".pkl")]
    ext = ".pkl.gz" if out.endswith(".pkl.gz") else ".pkl"
    pattern = re.compile(re.escape(os.path.basename(base)) + r"(?:\.part(\d+))?" + re.escape(ext) + r"$")
    max_idx = -1
    for f in glob.glob(os.path.join(os.path.dirname(out) or ".", "*")):
        m = pattern.match(os.path.basename(f))
        if m:
            idx = int(m.group(1)) if m.group(1) else 0
            max_idx = max(max_idx, idx)
    return max_idx + 1


def write_shard(out, idx, samples):
    path = _shard_path(out, idx)
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "wb") as f:
        pickle.dump(samples, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"wrote {path} ({os.path.getsize(path)/1e6:.1f} MB, {len(samples)} samples)", file=sys.stderr)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--games", type=int, default=300)
    ap.add_argument("--ckpt", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "ptcg_bc_v1.pth"))
    ap.add_argument("--temp", type=float, default=1.0)
    ap.add_argument("--out", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "sp_data.pkl.gz"))
    ap.add_argument("--workers", type=int, default=None)
    ap.add_argument("--chunk-games", type=int, default=100,
                     help="games per collect+value-target+write cycle — bounds how much "
                          "work a kill can lose, since each chunk is written to disk "
                          "before starting the next")
    ap.add_argument("--shard-samples", type=int, default=50_000,
                     help="flush a new shard file once the in-memory sample buffer "
                          "reaches this size (keeps memory bounded on long runs)")
    args = ap.parse_args()

    os.environ["NET_CKPT"] = os.path.abspath(args.ckpt)
    os.environ["NET_TEMP"] = str(args.temp)
    agent_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "selfplay_agent.py")

    model = load_model(os.path.abspath(args.ckpt))  # for value-target bootstrapping
    shard_idx = _next_shard_idx(args.out)
    if shard_idx > 0:
        print(f"found existing shards — continuing at shard {shard_idx}", file=sys.stderr)

    print(f"Collecting {args.games} self-play games with ckpt={args.ckpt} temp={args.temp}")
    samples = []
    games = total_samples = 0
    remaining = args.games
    while remaining > 0:
        n = min(args.chunk_games, remaining)
        remaining -= n
        results = run_matches(agent_path, agent_path, n, workers=args.workers,
                               keep_steps=True, progress=False)
        for r in results:
            if "error" in r or "steps" not in r:
                continue
            games += 1
            for seat in (0, 1):
                outcome = r["rewards"][seat]
                decs = extract_decisions(r["steps"], seat=seat)
                compute_value_targets(model, decs, outcome or 0)
                samples.extend(decs)
        print(f"  {games}/{args.games} games, {len(samples)} buffered samples "
              f"(next shard {shard_idx})", file=sys.stderr)
        if len(samples) >= args.shard_samples:
            write_shard(args.out, shard_idx, samples)
            total_samples += len(samples)
            samples = []
            shard_idx += 1

    total_samples += len(samples)
    if samples or shard_idx == 0:
        write_shard(args.out, shard_idx, samples)
        shard_idx += 1
    print(f"games={games} samples={total_samples} shards_written_this_run={shard_idx}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile training/nn/selfplay_agent.py
"""Temperature-sampling variant of net_agent.py, used ONLY for self-play data
collection (exploration) — never for ladder/eval, where net_agent.py's
deterministic argmax is used instead.

Env vars: NET_CKPT (checkpoint path), NET_TEMP (softmax temperature, default 1.0).
"""
import os
import sys

# See net_agent.py — must not rely on the top-level launching script's sys.path.
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import torch
import torch.nn.functional as F

from net_common import load_model, encode_batch, clamp, _HERE
from main import DECK

_CKPT = os.environ.get("NET_CKPT") or os.path.join(_HERE, "..", "ptcg_bc_v1.pth")
_TEMP = float(os.environ.get("NET_TEMP", "1.0"))


def agent(obs_dict: dict) -> list:
    sel = obs_dict.get("select")
    if sel is None:
        return DECK
    n = len(sel.get("option", []))
    if n == 0:
        return []
    try:
        model = load_model(_CKPT)
        batch, n_actions = encode_batch(obs_dict, sel)
        with torch.no_grad():
            logits, _ = model(*batch)
        logits = logits[0, :n_actions]
        mx = sel.get("maxCount", 1) or 1
        k = min(mx, n_actions)
        if k >= n_actions:
            picks = list(range(n_actions))
        else:
            probs = F.softmax(logits / max(_TEMP, 1e-3), dim=-1)
            picks = torch.multinomial(probs, k, replacement=False).tolist()
        return clamp(picks, sel)
    except Exception:
        mn = sel.get("minCount", 1) or 0
        return clamp(list(range(max(mn, 1))), sel)


In [ ]:
%%writefile training/nn/mcts_leafeval_agent.py
"""agent(obs_dict) wrapper around MCTSSearcher(leaf_eval="net") -- Phase 0
step-0 probe (docs/nn-training.md Phase 0 amendment (d)): does PUCT search
with the existing ptcg_dmc_r2.pth value net at the leaf show any life over
raw argmax (training/nn/dmc_agent.py), before spending a week improving that
value net? No retraining -- reuses the checkpoint as-is.

Same agent contract as main.py so it can be evaluated with training/ab_test.py.
Needs `cg.api` on sys.path (training/setup_local_search.py for local dev).
"""
import os
import pickle
import sys
import time

_HERE = os.path.dirname(os.path.abspath(__file__))
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)
_REPO_ROOT = os.path.dirname(os.path.dirname(_HERE))
_LOCAL_CG = os.path.join(_REPO_ROOT, "training", "local_cg")
if _LOCAL_CG not in sys.path:
    sys.path.insert(0, _LOCAL_CG)

from mcts import MCTSSearcher  # noqa: E402
from main import DECK  # noqa: E402

_SIMS = int(os.environ.get("MCTS_SIMS", "150"))
_C_PUCT = float(os.environ.get("MCTS_C_PUCT", "1.4"))
_PRIOR_TEMP = float(os.environ.get("MCTS_PRIOR_TEMP", "2.0"))
_NET_CKPT = os.environ.get("NET_CKPT") or os.path.join(_REPO_ROOT, "training", "ptcg_dmc_r2.pth")
_NET_LEAF_MAX_DEPTH = int(os.environ.get("MCTS_NET_LEAF_MAX_DEPTH", "40"))
# "qmax" (default) preserves this module's original behavior, correct for
# dmc_collect.py-trained checkpoints (this module's original Phase 0 probe
# target). Phase 2 collection (mcts_collect.py) sets this to "head" — see
# mcts.py's MCTSSearcher.__init__ docstring and docs/report-log.md 2026-07-07
# "mcts.py _net_leaf_value convention mismatch" for why the distinction matters.
_NET_VALUE_SOURCE = os.environ.get("MCTS_NET_VALUE_SOURCE", "qmax")
# Compute-budget check (docs/report-log.md 2026-07-05 timing probe): if set,
# appends "<elapsed_seconds>\n" per real decision to this path (one file per
# worker process, PID-suffixed, since ab_test.py fans out across
# multiprocessing workers) -- lets mcts_timing_probe.py measure real
# per-decision cost via the already-proven ab_test.py/harness.py runner
# instead of re-implementing game-running logic.
_TIMING_LOG = os.environ.get("MCTS_TIMING_LOG")
# Phase 2 (docs/nn-training.md "AlphaZero-Style Push"): if set, uses
# choose_with_stats() instead of choose() and appends one pickled
# {"obs":..., "action":..., "policy_target":[...], "root_value":...} record
# per real decision to this path (PID-suffixed, same multiprocess-safe
# pattern as MCTS_TIMING_LOG) -- lets mcts_collect.py extract real
# MCTS-derived training targets via the already-proven ab_test.py/
# harness.py multiprocess runner, instead of a hand-rolled direct-in-process
# game loop (which broke on repeated native search calls earlier this
# session -- see the compute-budget-check entry in report-log.md).
_COLLECT_LOG = os.environ.get("MCTS_COLLECT_LOG")
# Parallel collection support: harness.py's run_matches(extra_envs=...) sets
# this uniquely per job/game BEFORE this module is (re-)imported, so each
# fresh import picks up the correct per-game value -- lets mcts_collect.py
# correlate collect-log records to game outcomes correctly even when
# workers>1 interleaves multiple games' decisions across worker processes
# (previously required workers=1 + strict submission order).
_GAME_ID = os.environ.get("MCTS_GAME_ID")

_fallback_count = 0


def agent(obs_dict: dict) -> list:
    global _fallback_count
    sel = obs_dict.get("select")
    if sel is None:
        return DECK
    n = len(sel.get("option", []))
    if n == 0:
        return []
    t0 = time.time() if _TIMING_LOG else None
    try:
        searcher = MCTSSearcher(sims=_SIMS, c_puct=_C_PUCT, prior_temp=_PRIOR_TEMP,
                                 leaf_eval="net", net_ckpt=_NET_CKPT,
                                 net_leaf_max_depth=_NET_LEAF_MAX_DEPTH,
                                 net_value_source=_NET_VALUE_SOURCE)
        if _COLLECT_LOG:
            action, N, root_value = searcher.choose_with_stats(obs_dict)
            if N is not None:
                total = sum(N) or 1
                policy_target = [x / total for x in N]
                path = f"{_COLLECT_LOG}.{os.getpid()}"
                with open(path, "ab") as f:
                    pickle.dump({"obs": obs_dict, "action": action,
                                 "policy_target": policy_target,
                                 "root_value": root_value,
                                 "game_id": _GAME_ID}, f)
            return action
        return searcher.choose(obs_dict)
    except Exception as e:
        _fallback_count += 1
        print(f"[mcts_leafeval_agent] FALLBACK #{_fallback_count} to heuristic: {e!r}",
              file=sys.stderr)
        from main import agent as heuristic_agent
        return heuristic_agent(obs_dict)
    finally:
        if t0 is not None:
            path = f"{_TIMING_LOG}.{os.getpid()}"
            with open(path, "a") as f:
                f.write(f"{time.time() - t0:.4f}\n")


In [ ]:
%%writefile training/nn/mcts.py
"""Heuristic-guided PIMC (perfect-information Monte Carlo) search over the
real cg-lib search API (Stage 5, search-at-inference — see
docs/nn-training.md "Resume Here" 2026-07-04 and docs/engine-api.md "Search
API"). Requires `cg.api` on sys.path (either the live Kaggle environment, or
the local dev shim built by `training/setup_local_search.py`).

Design history (see docs/report-log.md 2026-07-04 "Stage 5 search-at-inference"
for the full story — four gates, three real bugs found and fixed via targeted
diagnostics rather than blind hyperparameter tuning, then a fourth diagnostic
that found the actual limiting mechanism):

1. First two gates came back BELOW 50% — a sign a simple "search adds no
   info" theory can't explain; it means the value signal was misaligned.
2. Bug #1: rollout/opponent-model called `main.score_options` argmax, a
   partial (~85%-agreement) reconstruction of the real teacher — confirmed
   via isolated A/B to win only ~30% alone. Fixed: call the real
   `main.agent` instead.
3. Bug #2: **strategy fusion** — hidden-zone determinization was sampled
   once per real decision and reused across all simulations. Fixed:
   re-determinize (fresh random `filler()` + fresh `search_begin`) every
   simulation, `manual_coin=False`. Structural consequence: tree stats can
   only be shared at the root (single-ply PUCT + full rollout per sim, not
   a deep persistent tree) since different sims are different worlds past
   the root.
4. Bug #3: fixing #1 (routing through the real `main.agent`) reintroduced
   global mutable-state corruption — `main.agent`/`_choose` maintains a
   stall-avoidance cache (`_STALL_MEMO`) safe for one linear real game but
   corrupted by thousands of interleaved simulated-rollout calls (collapsed
   a real gate to 1.7%). Fixed: save/reset/restore `_STALL_MEMO` around
   each simulation's rollout.
5. **The actual limiting mechanism (found via a 2-minute targeted
   diagnostic, not another full gate):** even after all three fixes, PUCT
   visit counts piled entirely on one action every time (zero exploration)
   and every terminating rollout was a WIN (90/90 across 3 test positions,
   0 losses). Root cause: the rollout's simulated "opponent" was our own
   heuristic piloting a random hand of OUR OWN deck — a hapless mirror
   opponent that can't punish a bad root choice, so the rollout carried no
   discriminating signal at all. Per advisor + a Claude Fable consult
   (2026-07-04), fixed as a single time-boxed test: the rollout's opponent
   turns are now played by a real adversarial opponent module
   (`MCTS_OPPONENT_MODULE`, default `opponents/lucario_agent.py`) instead of
   a mirror of our own deck/heuristic, with its own hidden-zone filler
   drawn from THAT opponent's real deck list, and its own module-global
   mutable state (`plan`/`pre_turn`/`ability_used` for lucario_agent) reset
   per simulation the same way `_STALL_MEMO` is.

- **Prior** at the root: `main.score_options` (softmaxed) — weaker than the
  real teacher in isolation, but a weak prior only misguides exploration,
  far less damaging than a weak value signal.
- **Rollout policy:** OUR side uses the real `main.agent`; the OPPONENT
  side uses a real adversarial opponent module's `agent()`, all the way to
  a real terminal result.
- **Multi-select safety:** any node whose `select.minCount >= 2` needs
  multiple simultaneous indices — not modeled combinatorially. The
  top-level real decision defers wholesale to `main.agent` if it's one of
  these; mid-rollout such nodes are handled naturally (rollout just plays
  whichever side's real agent full index list).
- **Leaf evaluator:** the rollout's real terminal result, not the net value
  head (already confirmed saturated bimodally near ±1 during the AWR
  diagnostic — unusable as a leaf evaluator).
"""
import dataclasses
import importlib
import math
import os
import random
import sys

_HERE = os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.dirname(os.path.dirname(_HERE))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

import main as heuristic  # noqa: E402
from net_common import load_model, encode_batch  # noqa: E402

_OPPONENT_MODULE_NAME = os.environ.get("MCTS_OPPONENT_MODULE", "opponents.lucario_agent")
_opponent_module = importlib.import_module(_OPPONENT_MODULE_NAME)

# Known mutable module-globals that must be isolated per-simulation (found by
# inspection — `main.py`'s `_STALL_MEMO` and `lucario_agent`'s `plan`/
# `pre_turn`/`ability_used` are both real, confirmed-necessary cases; add an
# entry here if a different `MCTS_OPPONENT_MODULE` turns out to need one).
_STATEFUL_MODULES = {
    heuristic: {"_STALL_MEMO": dict},
}
if hasattr(_opponent_module, "plan"):
    _STATEFUL_MODULES[_opponent_module] = {
        "plan": type(_opponent_module.plan),
        "pre_turn": lambda: 0,
        "ability_used": lambda: False,
    }


def _obs_to_dict(observation):
    return dataclasses.asdict(observation)


def _softmax(xs, temp=1.0):
    if not xs:
        return []
    m = max(xs)
    exps = [math.exp((x - m) / max(temp, 1e-6)) for x in xs]
    s = sum(exps)
    return [e / s for e in exps] if s > 0 else [1.0 / len(xs)] * len(xs)


def _is_multiselect(sel):
    return sel is not None and (sel.get("minCount", 0) or 0) >= 2


def _terminal_value(obs_dict, our_seat):
    cur = obs_dict.get("current") or {}
    result = cur.get("result", -1)
    if result == our_seat:
        return 1.0
    if result == (1 - our_seat):
        return -1.0
    return 0.0


def _is_terminal(obs_dict):
    sel = obs_dict.get("select")
    return sel is None or not sel.get("option")


class MCTSSearcher:
    """One instance per real decision. Call `choose(obs_dict)` once."""

    def __init__(self, sims=150, max_rollout_depth=250, c_puct=1.4, prior_temp=2.0,
                 leaf_eval="rollout", net_ckpt=None, net_leaf_max_depth=40,
                 net_value_source="qmax"):
        """leaf_eval: "rollout" (default, unchanged — real terminal result) or
        "net" (Phase 0 step-0 probe: stop as soon as it's our own decision
        again — bounded by net_leaf_max_depth — and evaluate a value estimate
        from net_ckpt, instead of rolling out to a real terminal result).
        "net" only ever queries the net at our-own-turn states, matching the
        distribution dmc_collect.py trained it on (see docs/nn-training.md
        Phase 0 amendment (d) — no sign-convention translation needed because
        we never ask the net to value an opponent-turn state).

        net_value_source: "qmax" (default, preserves this class's original
        behavior — max_a Q(s,a) via the DMC convention where the action
        logit itself IS Q; correct for `dmc_collect.py`-trained checkpoints
        like the Phase 0 probe this class was built for) or "head" (the
        model's separate value_head output, via net_common.value_estimate —
        REQUIRED for train_sp.py-trained checkpoints, whose logits are policy
        preferences, not Q-values; matches dmc_replay_gate.py's
        --value-source head). Found 2026-07-07: Phase 2's mcts_collect.py
        passed a train_sp.py-lineage checkpoint through this class with the
        "qmax" default silently unchanged, so every leaf evaluation during
        Phase 2 self-play collection took the max raw policy logit (an
        unbounded, uncalibrated quantity) as if it were a value estimate —
        corrupting the n-step bootstrap term of `value_target` for every
        decision more than N_STEP steps from a game's end. See
        docs/report-log.md 2026-07-07 "mcts.py _net_leaf_value convention
        mismatch" entry for the full diagnosis."""
        self.sims = sims
        self.max_rollout_depth = max_rollout_depth
        self.c_puct = c_puct
        self.prior_temp = prior_temp
        self.depth_cap_hits = 0
        self.leaf_eval = leaf_eval
        self.net_ckpt = net_ckpt or os.path.join(_HERE, "..", "ptcg_dmc_r2.pth")
        self.net_leaf_max_depth = net_leaf_max_depth
        self.net_value_source = net_value_source

    def _action_for(self, obs_dict, our_seat):
        """Real teacher for our own turns, the adversarial opponent module
        for the opponent's turns (see module docstring for why a mirror of
        our own heuristic was a dead end as the rollout opponent)."""
        yours = obs_dict["current"]["yourIndex"]
        agent_fn = heuristic.agent if yours == our_seat else _opponent_module.agent
        out = agent_fn(obs_dict)
        return list(out) if out else [0]

    def _rollout(self, search_id, obs_dict, our_seat):
        saved = {}
        for module, attrs in _STATEFUL_MODULES.items():
            saved[module] = {a: getattr(module, a) for a in attrs if hasattr(module, a)}
            for a, default_factory in attrs.items():
                if hasattr(module, a):
                    setattr(module, a, default_factory())
        try:
            cur_id, cur_obs = search_id, obs_dict
            for _ in range(self.max_rollout_depth):
                if _is_terminal(cur_obs):
                    return _terminal_value(cur_obs, our_seat)
                from cg.api import search_step
                ss = search_step(cur_id, self._action_for(cur_obs, our_seat))
                cur_id, cur_obs = ss.searchId, _obs_to_dict(ss.observation)
            self.depth_cap_hits += 1
            return 0.0
        finally:
            for module, vals in saved.items():
                for a, v in vals.items():
                    setattr(module, a, v)

    def _net_leaf_value(self, search_id, obs_dict, our_seat):
        """Advance real play (opponent turns via _action_for, our turns are
        the query point) up to net_leaf_max_depth, then return a value
        estimate from self.net_ckpt (max_a Q(s,a) if net_value_source is
        "qmax", the value_head output if "head" — see __init__'s docstring)
        at the first state where it's our own decision again. Falls back to
        the real terminal result if the game ends first, or 0.0 (unknown) if
        the depth cap is hit before either happens."""
        saved = {}
        for module, attrs in _STATEFUL_MODULES.items():
            saved[module] = {a: getattr(module, a) for a in attrs if hasattr(module, a)}
            for a, default_factory in attrs.items():
                if hasattr(module, a):
                    setattr(module, a, default_factory())
        try:
            cur_id, cur_obs = search_id, obs_dict
            for _ in range(self.net_leaf_max_depth):
                if _is_terminal(cur_obs):
                    return _terminal_value(cur_obs, our_seat)
                if cur_obs["current"]["yourIndex"] == our_seat:
                    sel = cur_obs.get("select")
                    if not sel or not sel.get("option"):
                        return _terminal_value(cur_obs, our_seat)
                    model = load_model(self.net_ckpt)
                    batch, n = encode_batch(cur_obs, sel)
                    import torch
                    with torch.no_grad():
                        logits, value = model(*batch)
                    if self.net_value_source == "head":
                        return float(value.item())
                    return float(logits[0, :n].max().item())
                from cg.api import search_step
                ss = search_step(cur_id, self._action_for(cur_obs, our_seat))
                cur_id, cur_obs = ss.searchId, _obs_to_dict(ss.observation)
            self.depth_cap_hits += 1
            return 0.0
        finally:
            for module, vals in saved.items():
                for a, v in vals.items():
                    setattr(module, a, v)

    @staticmethod
    def _filler(n, pool):
        pool = list(pool)
        random.shuffle(pool)
        return pool[:n] if n <= len(pool) else [pool[i % len(pool)] for i in range(n)]

    def choose(self, obs_dict):
        """Same contract as main.agent(): returns list[int]."""
        action, _N, _root_value = self.choose_with_stats(obs_dict)
        return action

    def choose_with_stats(self, obs_dict):
        """Phase 2 (docs/nn-training.md "AlphaZero-Style Push"): same search
        as choose(), but also returns the raw visit counts N (the material
        for an AlphaZero-style soft policy target, normalize with
        N[a]/sum(N)) and a root value estimate (sum(W)/sum(N), the
        visit-weighted average backed-up value across all sampled actions --
        standard AlphaZero convention for the value target at this state).
        Returns (action_list, N_or_None, root_value_or_None) -- N/root_value
        are None for the fast-path returns (<=1 option, multiselect) where no
        real search happened, since there's no meaningful policy/value
        target to extract from those."""
        from cg.api import to_observation_class, search_begin, search_step, search_end

        sel = obs_dict.get("select")
        if not sel or not sel.get("option"):
            return [], None, None
        n_opts = len(sel["option"])
        if n_opts <= 1:
            return ([0] if n_opts == 1 else []), None, None
        if _is_multiselect(sel):
            # Combinatorial multi-select isn't modeled here — defer to the
            # real teacher rather than emit a truncated/invalid selection.
            return heuristic.agent(obs_dict), None, None

        our_seat = obs_dict["current"]["yourIndex"]

        scores = heuristic.score_options(obs_dict, sel)
        if not scores or len(scores) != n_opts:
            scores = [0.0] * n_opts
        P = _softmax(scores, temp=self.prior_temp)
        N = [0] * n_opts
        W = [0.0] * n_opts

        for _ in range(self.sims):
            # Fresh determinization + fresh search per simulation — the fix
            # for strategy fusion (see module docstring). Only the hidden
            # zones vary between sims; the real observable state (and hence
            # the root's actual select/options) doesn't change.
            observation = to_observation_class(obs_dict)
            state = observation.current
            my_p = state.players[our_seat]
            opp_p = state.players[1 - our_seat]

            your_deck = self._filler(my_p.deckCount, heuristic.DECK)
            your_prize = self._filler(len(my_p.prize), heuristic.DECK)
            opponent_deck = self._filler(opp_p.deckCount, _opponent_module.DECK)
            opponent_prize = self._filler(len(opp_p.prize), _opponent_module.DECK)
            opponent_hand = self._filler(opp_p.handCount, _opponent_module.DECK)
            opponent_active = []
            active = opp_p.active
            if len(active) > 0 and active[0] is None:
                opponent_active = self._filler(1, _opponent_module.DECK)

            root_ss = search_begin(observation, your_deck, your_prize, opponent_deck,
                                    opponent_prize, opponent_hand, opponent_active,
                                    manual_coin=False)

            total_n = sum(N)
            sqrt_total = math.sqrt(total_n + 1e-8)
            best_score, best_a = -1e18, 0
            for a in range(n_opts):
                q = W[a] / N[a] if N[a] > 0 else 0.0
                u = self.c_puct * P[a] * sqrt_total / (1 + N[a])
                score = q + u
                if score > best_score:
                    best_score, best_a = score, a

            child_ss = search_step(root_ss.searchId, [best_a])
            child_obs = _obs_to_dict(child_ss.observation)
            if self.leaf_eval == "net":
                value = self._net_leaf_value(child_ss.searchId, child_obs, our_seat)
            else:
                value = self._rollout(child_ss.searchId, child_obs, our_seat)

            N[best_a] += 1
            W[best_a] += value
            if os.environ.get("MCTS_DEBUG"):
                print(f"[mcts debug] a={best_a} value={value} N={N} W={[round(w,2) for w in W]}",
                      file=sys.stderr)

        best_a = max(range(n_opts), key=lambda a: N[a])
        search_end()
        total_n = sum(N)
        root_value = (sum(W) / total_n) if total_n > 0 else 0.0
        return [best_a], N, root_value


In [ ]:
%%writefile training/nn/mcts_collect.py
"""Phase 2 (docs/nn-training.md "AlphaZero-Style Push"): self-play data
collection with REAL search-derived targets -- closing the training loop
that Phase 0-era work only used at inference time.

Asymmetric: OUR side plays via `mcts_leafeval_agent.py` (PUCT + leaf-eval
via the value net's own value head, reduced sims -- eval-time's 100 sims/
decision would make a full self-play generation intractable within this
project's runway), which -- when MCTS_COLLECT_LOG is set -- logs the real
MCTS visit counts (-> policy_target) and backed-up root value
(-> mcts_root_value) per decision. The OPPONENT side is sampled per-game
from `opponent_pool.py`'s real-ladder-meta-weighted pool of the project's
existing rule-based archetype bots (lucario/dragapult/starmie/abomasnow
with their own real decks + logic, several more archetypes' reconstructed
decks piloted by `generic_pilot.py`, and a mirror slice piloted by the real
heuristic `main.py`) -- NOT the same checkpoint's own unsearched policy.

**2026-07-07 redesign, replacing the original same-checkpoint-mirror
opponent:** the mirror opponent was far too weak (searching side won 96.7%
of games), which produced a training corpus with severely imbalanced
win/loss labels and collapsed the trained value head toward unconditionally
predicting "winning" -- confirmed as the root cause of round 2's regression
(see docs/report-log.md 2026-07-07 "Root cause of the Phase 2 round 2
regression"). A real, diverse, competitive opponent pool is the actual fix;
see opponent_pool.py's docstring for the exact weights and sourcing.

PARALLEL collection (2026-07-06, Fable-directed follow-up to the 30-game
serial validation milestone): each game is assigned a unique MCTS_GAME_ID via
harness.run_matches(extra_envs=...) (new -- see harness.py), which
mcts_leafeval_agent.py embeds in every collect-log record it writes. This
lets records from DIFFERENT worker processes (each potentially handling many
games over its lifetime, in whatever order the pool schedules them) be
grouped back to the correct game by game_id rather than by process/file
order -- the earlier serial-only version relied on strict submission order
within one process, which doesn't hold once workers>1.

Usage:
  python training/nn/mcts_collect.py --games 150 --workers 10 \
      --ckpt training/ptcg_dmc_p0_v2_n1_richenc_v2.pth --sims 40 \
      --out training/mcts_p2_r2.pkl.gz
"""
import argparse
import glob
import gzip
import json
import os
import pickle
import sys

_HERE = os.path.dirname(os.path.abspath(__file__))
REPO_ROOT = os.path.dirname(os.path.dirname(_HERE))
sys.path.insert(0, _HERE)
sys.path.insert(0, os.path.join(REPO_ROOT, "training"))
sys.path.insert(0, REPO_ROOT)

from harness import run_matches  # noqa: E402
from bc_collect import extract_decisions  # noqa: E402
from selfplay_collect import compute_value_targets  # noqa: E402
from net_common import load_model  # noqa: E402
import opponent_pool  # noqa: E402

OUR_AGENT = os.path.join(_HERE, "mcts_leafeval_agent.py")


def _read_all_collect_logs(collect_log_base):
    """Reads back sequentially-appended pickle records from EVERY worker's
    log file (PID-suffixed, glob-matched) -- with workers>1, many distinct
    processes may each have written some of the games' records."""
    records = []
    for path in glob.glob(f"{collect_log_base}.*"):
        with open(path, "rb") as f:
            while True:
                try:
                    records.append(pickle.load(f))
                except EOFError:
                    break
    return records


def _obs_key(obs):
    """Stable key for matching the SAME logical obs across two different
    code paths (a live agent() call vs. the same state reconstructed from
    env.steps afterward). Two things confirmed empirically NOT safe here:
    (1) pickle.dumps() comparison -- two dicts can be `==` equal with
    different internal key insertion order, which pickle serializes
    byte-differently; (2) including the full obs -- `remainingOverageTime`
    (a live decrementing clock) and `step` were found to differ/be
    inconsistently populated (None in one reconstruction path, a real int in
    the other) between the two representations for at least one seat
    direction, even for the logically same decision. Only `current` (full
    game state: turn, players, board) + `select` (the options being chosen
    from) are used -- together they uniquely identify a decision without
    the volatile metadata fields."""
    return json.dumps({"current": obs.get("current"), "select": obs.get("select")},
                       sort_keys=True, default=str)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--games", type=int, default=150)
    ap.add_argument("--ckpt", required=True, help="checkpoint for BOTH sides (true self-play)")
    ap.add_argument("--sims", type=int, default=40, help="reduced sims/decision (vs eval-time's 100)")
    ap.add_argument("--temp", type=float, default=1.0, help="opponent-side temperature (selfplay_agent.py)")
    ap.add_argument("--workers", type=int, default=None, help="None = auto (cpu_count-1)")
    ap.add_argument("--out", default=os.path.join(REPO_ROOT, "training", "mcts_p2_r2.pkl.gz"))
    args = ap.parse_args()

    os.environ["NET_CKPT"] = os.path.abspath(args.ckpt)
    os.environ["MCTS_SIMS"] = str(args.sims)
    os.environ["NET_TEMP"] = str(args.temp)
    # Phase 2 checkpoints are train_sp.py-lineage (separate value_head, logits
    # are policy preferences) — mcts.py's MCTSSearcher net-leaf-eval must read
    # the value_head, not the DMC "max policy logit as Q" convention its
    # net_value_source default preserves for the older Phase 0 probe. See
    # docs/report-log.md 2026-07-07 "mcts.py _net_leaf_value convention
    # mismatch" — the missing override here is exactly the bug that corrupted
    # the round-2 (mcts_p2_r3.pkl.gz) training corpus's bootstrap values.
    os.environ["MCTS_NET_VALUE_SOURCE"] = "head"
    collect_log_base = os.path.join(REPO_ROOT, "training", "mcts_collect")
    os.environ["MCTS_COLLECT_LOG"] = collect_log_base
    for stale in glob.glob(f"{collect_log_base}.*"):
        os.remove(stale)

    bootstrap_model = load_model(os.path.abspath(args.ckpt))

    samples = []
    counters = {"games": 0, "wins": 0, "relabel_errors": 0, "decisions_seen": 0, "decisions_matched": 0}
    per_opp = {}  # label -> {"games":n, "wins":n}
    game_id_counter = 0

    for net_seat, n_seat in ((0, args.games - args.games // 2), (1, args.games // 2)):
        if n_seat == 0:
            continue
        # opponent_pool.allocate splits this seat's games across the real-
        # meta-weighted archetype pool -- one run_matches call per archetype
        # (not per-game) since run_matches binds one fixed opponent agent
        # path for its whole batch; deck overrides (for generic_pilot.py
        # archetypes) still vary via the `decks` param.
        for label, opp_path, opp_deck, n_this_opp in opponent_pool.allocate(n_seat, seed=net_seat):
            paths = (OUR_AGENT, opp_path) if net_seat == 0 else (opp_path, OUR_AGENT)
            deck_override = (None, opp_deck) if net_seat == 0 else (opp_deck, None)
            decks = [deck_override] * n_this_opp
            game_ids = list(range(game_id_counter, game_id_counter + n_this_opp))
            game_id_counter += n_this_opp
            extra_envs = [{"MCTS_GAME_ID": str(gid)} for gid in game_ids]

            print(f"[mcts_collect] net_seat={net_seat} opponent={label} n={n_this_opp}",
                  file=sys.stderr)
            results = run_matches(paths[0], paths[1], n_this_opp, workers=args.workers,
                                   keep_steps=True, progress=True, extra_envs=extra_envs,
                                   decks=decks)

            all_records = _read_all_collect_logs(collect_log_base)
            records_by_game = {}
            for rec in all_records:
                records_by_game.setdefault(rec.get("game_id"), []).append(rec)

            opp_stats = per_opp.setdefault(label, {"games": 0, "wins": 0})
            for r in results:
                if "error" in r or "steps" not in r:
                    continue
                gid = (r.get("extra_env") or {}).get("MCTS_GAME_ID")
                outcome = r["rewards"][net_seat]
                counters["games"] += 1
                opp_stats["games"] += 1
                if outcome == 1:
                    counters["wins"] += 1
                    opp_stats["wins"] += 1
                game_records_raw = records_by_game.get(gid, [])
                record_by_obs = {_obs_key(rec["obs"]): rec for rec in game_records_raw}

                all_decisions = extract_decisions(r["steps"], seat=net_seat)
                counters["decisions_seen"] += len(all_decisions)
                game_decisions, game_records = [], []
                for d in all_decisions:
                    rec = record_by_obs.get(_obs_key(d["obs"]))
                    if rec is not None:
                        game_decisions.append(d)
                        game_records.append(rec)
                counters["decisions_matched"] += len(game_decisions)
                if not game_decisions:
                    counters["relabel_errors"] += 1
                    print(f"[mcts_collect] game {counters['games']} (id={gid}, "
                          f"opponent={label}): 0/{len(all_decisions)} decisions matched "
                          "a search record -- skipping", file=sys.stderr)
                    continue
                mcts_root_values = [rec["root_value"] for rec in game_records]
                for d, rec in zip(game_decisions, game_records):
                    d["policy_target"] = rec["policy_target"]
                compute_value_targets(bootstrap_model, game_decisions, outcome,
                                       mcts_root_values=mcts_root_values)
                samples.extend(game_decisions)
            # clear this archetype's collect logs before the next run_matches
            # call reuses the same worker pool -- otherwise the next
            # archetype's _read_all_collect_logs would also pick up stale
            # records from this one (game_ids never repeat, so matching would
            # still be correct, but the file would grow unboundedly).
            for stale in glob.glob(f"{collect_log_base}.*"):
                os.remove(stale)

    opener = gzip.open if args.out.endswith(".gz") else open
    with opener(args.out, "wb") as f:
        pickle.dump(samples, f, protocol=pickle.HIGHEST_PROTOCOL)

    match_rate = counters["decisions_matched"] / max(counters["decisions_seen"], 1)
    print(f"games={counters['games']} wins={counters['wins']} "
          f"winrate={counters['wins']/max(counters['games'],1):.3f} "
          f"samples={len(samples)} relabel_errors={counters['relabel_errors']} "
          f"decisions_seen={counters['decisions_seen']} "
          f"decisions_matched={counters['decisions_matched']} match_rate={match_rate:.3f}")
    for label, s in sorted(per_opp.items(), key=lambda kv: -kv[1]["games"]):
        wr = s["wins"] / max(s["games"], 1)
        print(f"  opponent={label:16s} games={s['games']:4d} winrate={wr:.3f}", file=sys.stderr)
    print(f"wrote {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile training/nn/net_common.py
"""Shared model-loading + encoding glue for net_agent.py / selfplay_agent.py."""
import os
import sys

_HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, _HERE)
_REPO_ROOT = os.path.dirname(os.path.dirname(_HERE))
sys.path.insert(0, _REPO_ROOT)

import torch

from encode import encode_sample, MAX_ACTIONS
from model import PTCGNet

DEVICE = torch.device("cpu")
_models = {}


def load_model(ckpt_path):
    """Cache by path so a process that loads two different checkpoints (e.g.
    self-play mirror vs a frozen teacher) doesn't reload on every call."""
    m = _models.get(ckpt_path)
    if m is None:
        m = PTCGNet()
        state = torch.load(ckpt_path, map_location=DEVICE)
        # older checkpoints' value_head was narrower (no oracle features) —
        # drop only the mismatched keys rather than error; a fresh value head
        # still gives usable (if uncalibrated) values, and the policy path
        # (unaffected by the oracle widening) loads and behaves unchanged.
        own = m.state_dict()
        state = {k: v for k, v in state.items() if k in own and v.shape == own[k].shape}
        m.load_state_dict(state, strict=False)
        m.eval()
        _models[ckpt_path] = m
    return m


def encode_batch(obs, sel):
    enc = encode_sample(obs, sel)
    n = enc["n_actions"]
    board_ids = torch.tensor([enc["board_ids"]], dtype=torch.long)
    h = enc["hand_ids"][:20]
    hand_ids = torch.zeros(1, 20, dtype=torch.long)
    hand_ids[0, :len(h)] = torch.tensor(h, dtype=torch.long)
    d = enc["discard_ids"][:20]
    discard_ids = torch.zeros(1, 20, dtype=torch.long)
    discard_ids[0, :len(d)] = torch.tensor(d, dtype=torch.long)
    numeric = torch.tensor([enc["numeric"]], dtype=torch.float)

    action_type = torch.zeros(1, MAX_ACTIONS, dtype=torch.long)
    action_card = torch.zeros(1, MAX_ACTIONS, dtype=torch.long)
    action_attack = torch.zeros(1, MAX_ACTIONS, dtype=torch.long)
    action_numeric = torch.zeros(1, MAX_ACTIONS, 4, dtype=torch.float)
    action_mask = torch.zeros(1, MAX_ACTIONS, dtype=torch.float)
    for j, a in enumerate(enc["actions"][:MAX_ACTIONS]):
        action_type[0, j] = a["type"]
        action_card[0, j] = a["card_id"]
        action_attack[0, j] = a["attack_id"]
        action_numeric[0, j] = torch.tensor(a["numeric"], dtype=torch.float)
        action_mask[0, j] = 1.0
    return (board_ids, hand_ids, discard_ids, numeric, action_type, action_card,
            action_attack, action_numeric, action_mask), n


def clamp(indices, sel):
    mn = sel.get("minCount", 0) or 0
    mx = sel.get("maxCount", 1) or 1
    n = len(sel.get("option", []))
    out = []
    for i in indices:
        if 0 <= i < n and i not in out:
            out.append(i)
        if len(out) >= mx:
            break
    i = 0
    while len(out) < mn and i < n:
        if i not in out:
            out.append(i)
        i += 1
    return out if out else ([0] if n > 0 else [])


def value_estimate(model, obs, sel):
    """Run just the value head for one decision point (used for n-step
    bootstrapping during self-play data collection)."""
    batch, n_actions = encode_batch(obs, sel)
    with torch.no_grad():
        _, value = model(*batch)
    return value.item()


In [ ]:
%%writefile training/nn/encode.py
"""Observation -> tensor encoding for the BC/self-play net.

v1 rebuilt design (all prior encoding code was lost with the reset — see
docs/nn-training.md). Deliberately simpler than the original 22000-vocab
transformer-decoder sketch: a 13-slot board sequence + hand/discard bag
embeddings + a small numeric feature vector, and a per-candidate-action
feature vector for the policy head. No cg-lib dependency for the base
features — everything there is read directly off the raw obs_dict,
identical to what `main.py` consumes.

CARD_VOCAB / ATTACK_VOCAB are hardcoded upper bounds (real max observed card
ID is 1267 as of 2026-07-01; enums may grow during the competition per the
official docs, hence the safety margin) rather than calling all_card_data() —
this keeps the base features usable with or without cg-lib attached.

2026-07-05 (user-directed): added situational-belief features to
numeric_feats -- the net previously saw only 13 raw counters (HP ratios,
hand/deck/prize counts, one hardcoded Mist flag, turn) with no archetype
belief, no opponent-threat estimate, and no evolution-line progress, despite
all three already existing elsewhere in the project (main.py's own belief
model, this session's threat.py). This IS a new dependency on `main` (repo
root) and `threat` (needs cg.api's local shim, training-side only, never
used by the ladder-submitted main.py itself) -- acceptable here since
encode.py is training-pipeline-only, unlike main.py's dependency-free
constraint.
"""
import math
import os
import sys

_HERE = os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.dirname(os.path.dirname(_HERE))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

import main as _heuristic  # noqa: E402
from threat import net_threat_diff as _net_threat_diff  # noqa: E402

CARD_VOCAB = 2000
ATTACK_VOCAB = 2000
OPTION_TYPE_VOCAB = 17  # OptionType enum, 0..16
N_BOARD_SLOTS = 13      # my_active, my_bench x5, opp_active, opp_bench x5, stadium
MAX_ACTIONS = 64

# 2026-07-05 ablation (docs/report-log.md "AlphaZero-style push, Phase 1"):
# the combined 25-feature encoding scored WORSE than the plain 13-feature
# baseline on the real-replay gate, despite each addition being individually
# well-motivated. ENCODE_FEATURE_SET isolates which addition (if any) is
# responsible, matching the isolated-component-before-trusting-the-
# combination discipline already used for Φ. "full" (default) preserves the
# already-tested combined behavior unchanged.
_FEATURE_SET = os.environ.get("ENCODE_FEATURE_SET", "full")
_FEATURE_SET_SIZES = {
    "base": 13,
    "base+threat": 14,
    "base+census": 16,   # +line_progress, +has_alakazam, +hand_advantage
    "base+belief": 21,   # +5 posterior probs, +wall_revealed, +crustle_seen, +unavailable flag
    "full": 25,
}
NUM_FEATS = _FEATURE_SET_SIZES[_FEATURE_SET]


def _pk_id(pk):
    return (pk or {}).get("id", 0) or 0


def _active(p):
    a = (p or {}).get("active")
    return a[0] if a and len(a) > 0 and a[0] else None


def board_slot_ids(obs):
    """13 card-id tokens: [my_active, my_bench(5), opp_active, opp_bench(5), stadium]."""
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    opp = pl[1 - me_idx] if len(pl) == 2 else {}
    my_bench = (me.get("bench") or [])[:5]
    opp_bench = (opp.get("bench") or [])[:5]
    ids = [_pk_id(_active(me))]
    ids += [_pk_id(b) for b in my_bench] + [0] * (5 - len(my_bench))
    ids.append(_pk_id(_active(opp)))
    ids += [_pk_id(b) for b in opp_bench] + [0] * (5 - len(opp_bench))
    stadium = (cur.get("stadium") or [None])
    ids.append(_pk_id(stadium[0]) if stadium else 0)
    return [min(i, CARD_VOCAB - 1) for i in ids]


def hand_ids(obs, cap=20):
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    hand = me.get("hand") or []
    ids = [min(_pk_id(c), CARD_VOCAB - 1) for c in hand if _pk_id(c)]
    return ids[:cap] or [0]


def discard_ids(obs, cap=20):
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    disc = me.get("discard") or []
    ids = [min(_pk_id(c), CARD_VOCAB - 1) for c in disc if _pk_id(c)]
    return ids[:cap] or [0]


_BELIEF_CLASS_ORDER = _heuristic._BELIEF_CLASSES  # fixed 5-class order, reused for a stable feature layout


def _belief_feats(opp, turn):
    """5 archetype posterior probabilities + wall_revealed + crustle_seen,
    via main.py's own embedded belief model (the same function main.py
    itself calls at inference) -- zero-filled + a bit set on failure so a
    belief-model exception never breaks the whole feature vector."""
    try:
        post, wall_revealed, crustle_seen = _heuristic._belief_posterior(opp, turn)
    except Exception:
        post, wall_revealed, crustle_seen = None, False, False
    if post is None:
        return [0.0] * 5 + [0.0, 0.0, 1.0]  # last 1.0 flags "belief unavailable"
    probs = [post.get(c, 0.0) for c in _BELIEF_CLASS_ORDER]
    return probs + [1.0 if wall_revealed else 0.0, 1.0 if crustle_seen else 0.0, 0.0]


def numeric_feats(obs):
    """Fixed-size float feature vector, roughly matching main.py's _census inputs,
    plus situational-belief features added 2026-07-05 (archetype posterior,
    zero-sum threat estimate, evolution-line progress, hand-vs-KO-threshold
    advantage) -- see module docstring."""
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    opp = pl[1 - me_idx] if len(pl) == 2 else {}
    my_active = _active(me)
    opp_active = _active(opp)
    my_hp = (my_active or {}).get("hp", 0) or 0
    my_maxhp = (my_active or {}).get("maxHp", 1) or 1
    opp_hp = (opp_active or {}).get("hp", 0) or 0
    opp_maxhp = (opp_active or {}).get("maxHp", 1) or 1
    my_hand_n = me.get("handCount") or len(me.get("hand") or [])
    opp_hand_n = opp.get("handCount", 0) or 0
    my_deck = me.get("deckCount", 0) or 0
    opp_deck = opp.get("deckCount", 0) or 0
    my_prizes = len(me.get("prize") or [])
    opp_prizes = len(opp.get("prize") or [])
    opp_energies = set()
    for ec in (opp_active or {}).get("energyCards") or []:
        opp_energies.add(ec.get("id"))
    opp_mist = 1.0 if (11 in opp_energies or 20 in opp_energies) else 0.0
    turn = cur.get("turn", 0) or 0

    try:
        cen = _heuristic._census(my_active, me.get("bench") or [])
        line_progress = cen["line_count"] / 2.0
        has_alakazam = 1.0 if cen["has_alakazam"] else 0.0
    except Exception:
        line_progress, has_alakazam = 0.0, 0.0

    if opp_hp:
        cards_needed = math.ceil(opp_hp / _heuristic.PH_DMG_PER_CARD)
        hand_advantage = max(-1.0, min(1.0, (my_hand_n - cards_needed) / 10.0))
    else:
        hand_advantage = min(my_hand_n / 10.0, 1.0)

    try:
        threat_diff = _net_threat_diff(cur, me_idx)
    except Exception:
        threat_diff = 0.0

    base = [
        my_hp / max(my_maxhp, 1),
        opp_hp / max(opp_maxhp, 1),
        min(my_hand_n, 30) / 30.0,
        min(opp_hand_n, 30) / 30.0,
        min(my_deck, 60) / 60.0,
        min(opp_deck, 60) / 60.0,
        min(my_prizes, 6) / 6.0,
        min(opp_prizes, 6) / 6.0,
        opp_mist,
        1.0 if cur.get("supporterPlayed") else 0.0,
        1.0 if cur.get("energyAttached") else 0.0,
        1.0 if cur.get("retreated") else 0.0,
        min(turn, 60) / 60.0,
    ]
    census_group = [line_progress, has_alakazam, hand_advantage]
    threat_group = [threat_diff]
    belief_group = _belief_feats(opp, turn)

    if _FEATURE_SET == "base":
        return base
    if _FEATURE_SET == "base+threat":
        return base + threat_group
    if _FEATURE_SET == "base+census":
        return base + census_group
    if _FEATURE_SET == "base+belief":
        return base + belief_group
    return base + census_group + threat_group + belief_group  # "full"


def _opt_card_id(o, hand, bench):
    """Mirrors main.py._opt_card_id — resolve an option's associated card id."""
    ot = o.get("type")
    idx = o.get("index")
    if ot in (3, 4, 5, 7, 8, 9):  # CARD/TOOL_CARD/ENERGY_CARD/PLAY/ATTACH/EVOLVE
        if idx is not None and 0 <= idx < len(hand):
            return _pk_id(hand[idx])
        return 0
    if ot == 10:  # ABILITY
        area = o.get("area")
        if area == 4:
            return 0  # active resolved separately by caller if needed
        if area == 5 and 0 <= idx < len(bench):
            return _pk_id(bench[idx])
    return 0


def encode_action(obs, o):
    """Per-candidate-option feature dict: type id, resolved card id, attack id, numerics."""
    cur = obs.get("current") or {}
    me_idx = cur.get("yourIndex", 0)
    pl = cur.get("players") or []
    me = pl[me_idx] if len(pl) > me_idx else {}
    hand = me.get("hand") or []
    bench = me.get("bench") or []
    ot = o.get("type") or 0
    cid = _opt_card_id(o, hand, bench)
    if ot == 10 and o.get("area") == 4:  # ABILITY on active
        cid = _pk_id(_active(me))
    attack_id = o.get("attackId") or 0
    area = o.get("area") or 0
    in_play_area = o.get("inPlayArea") or 0
    index = o.get("index") or 0
    in_play_index = o.get("inPlayIndex") or 0
    return {
        "type": min(ot, OPTION_TYPE_VOCAB - 1),
        "card_id": min(cid, CARD_VOCAB - 1),
        "attack_id": min(attack_id, ATTACK_VOCAB - 1),
        "numeric": [area / 12.0, in_play_area / 12.0, index / 20.0, in_play_index / 6.0],
    }


def encode_sample(obs, sel):
    """Full encoding for one decision point. Returns a dict of plain python
    lists/ints (torch-free) so this module works without a torch import —
    the Dataset class converts to tensors at collate time."""
    opts = (sel.get("option") or [])[:MAX_ACTIONS]
    return {
        "board_ids": board_slot_ids(obs),
        "hand_ids": hand_ids(obs),
        "discard_ids": discard_ids(obs),
        "numeric": numeric_feats(obs),
        "actions": [encode_action(obs, o) for o in opts],
        "n_actions": len(opts),
    }


In [ ]:
%%writefile training/nn/model.py
"""BC/self-play actor-critic net. See encode.py for the feature design and
docs/nn-training.md for the architecture rationale. Small on purpose: this is
a warm-start policy, not the final word — self-play (Phase 1+) can grow it.
"""
import torch
import torch.nn as nn

from encode import (
    CARD_VOCAB, ATTACK_VOCAB, OPTION_TYPE_VOCAB, N_BOARD_SLOTS, NUM_FEATS,
)

D_CARD = 128
D_ATTACK = 64
D_TYPE = 32
D_MODEL = 128


class PTCGNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.card_embed = nn.Embedding(CARD_VOCAB, D_CARD, padding_idx=0)
        self.attack_embed = nn.Embedding(ATTACK_VOCAB, D_ATTACK, padding_idx=0)
        self.type_embed = nn.Embedding(OPTION_TYPE_VOCAB, D_TYPE)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=D_CARD, nhead=2, dim_feedforward=256, batch_first=True)
        self.board_transformer = nn.TransformerEncoder(enc_layer, num_layers=1)

        self.hand_bag = nn.EmbeddingBag(CARD_VOCAB, D_CARD, mode="sum", padding_idx=0)
        self.discard_bag = nn.EmbeddingBag(CARD_VOCAB, D_CARD, mode="sum", padding_idx=0)

        self.numeric_proj = nn.Sequential(nn.Linear(NUM_FEATS, D_CARD), nn.ReLU())
        self.trunk = nn.Sequential(
            nn.Linear(D_CARD * 4, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )
        # oracle: privileged opponent-hand feature for the value head only
        # (PerfectDou-style critic — see docs/report-log.md 2026-07-04 lit review).
        # Not available at inference (main.py never sees the opponent's hand), so
        # it's zeroed + flagged off whenever the caller doesn't supply it.
        self.oracle_embed = nn.EmbeddingBag(CARD_VOCAB, D_CARD, mode="sum", padding_idx=0)
        self.value_head = nn.Linear(256 + D_CARD + 1, 1)

        act_in_dim = D_TYPE + D_CARD + D_ATTACK + 4
        self.action_mlp = nn.Sequential(nn.Linear(act_in_dim, D_MODEL), nn.ReLU())
        self.logit_mlp = nn.Sequential(
            nn.Linear(D_MODEL + 256, 128), nn.ReLU(), nn.Linear(128, 1))

    def forward(self, board_ids, hand_ids, discard_ids, numeric,
                action_type, action_card, action_attack, action_numeric, action_mask,
                oracle_ids=None, oracle_offsets=None, oracle_flag=None):
        board_emb = self.card_embed(board_ids)                  # (B,13,128)
        board_ctx = self.board_transformer(board_emb)            # (B,13,128)
        board_vec = board_ctx.mean(dim=1)                        # (B,128)
        hand_vec = self.hand_bag(hand_ids)                       # (B,128)
        discard_vec = self.discard_bag(discard_ids)              # (B,128)
        feat_vec = self.numeric_proj(numeric)                    # (B,128)

        trunk_in = torch.cat([board_vec, hand_vec, discard_vec, feat_vec], dim=-1)
        trunk = self.trunk(trunk_in)                             # (B,256)
        B = trunk.shape[0]
        if oracle_ids is not None:
            oracle_vec = self.oracle_embed(oracle_ids)            # (B,128)
        else:
            oracle_vec = torch.zeros(B, D_CARD, device=trunk.device, dtype=trunk.dtype)
        if oracle_flag is None:
            oracle_flag = torch.zeros(B, 1, device=trunk.device, dtype=trunk.dtype)
        value = torch.tanh(self.value_head(
            torch.cat([trunk, oracle_vec, oracle_flag], dim=-1)).squeeze(-1))  # (B,)

        B, A = action_type.shape
        type_emb = self.type_embed(action_type)                  # (B,A,32)
        card_emb = self.card_embed(action_card)                  # (B,A,128)
        attack_emb = self.attack_embed(action_attack)             # (B,A,64)
        act_in = torch.cat([type_emb, card_emb, attack_emb, action_numeric], dim=-1)
        act_vec = self.action_mlp(act_in)                         # (B,A,128)

        trunk_exp = trunk.unsqueeze(1).expand(-1, A, -1)          # (B,A,256)
        logits = self.logit_mlp(torch.cat([act_vec, trunk_exp], dim=-1)).squeeze(-1)  # (B,A)
        logits = logits.masked_fill(action_mask == 0, -1e9)
        return logits, value


In [ ]:
%%writefile training/nn/dataset.py
"""Turns training/bc_data*.pkl(.gz) shards into batched tensors for PTCGNet.

BC value target = game outcome (+1/-1/0), the simplest valid target for the
imitation warmup. n-step bootstrapped value targets (docs/nn-training.md
§Value Targets) apply to the self-play phase, not this warmup.
"""
import glob
import gzip
import pickle
import random

import torch
from torch.utils.data import Dataset

from encode import encode_sample, MAX_ACTIONS, CARD_VOCAB, NUM_FEATS

# probability of zeroing an available oracle in collate() (regularizes the
# value head so it doesn't collapse onto always expecting the oracle feature
# at inference, where main.py never supplies one). Eval scripts that want the
# pure oracle/no-oracle path unconditionally can override: dataset.ORACLE_DROPOUT = 0
ORACLE_DROPOUT = 0.25


def _opener(path):
    return gzip.open if path.endswith(".gz") else open


def load_shards(pattern, limit=None):
    """pattern e.g. '/kaggle/input/**/bc_data*.pkl' (recursive — Kaggle's exact
    mount subdirectory name can differ from the dataset slug). Comma-separate
    multiple patterns to combine sources, e.g. BC + DAgger-round shards.

    `limit`, if given, stops reading further shards once enough samples are
    loaded — capping AFTER a full glob-matched load (e.g. slicing the return
    value) still momentarily materializes every shard at once, which is what
    actually exhausted RAM (~37GB transient peak) even with a small final
    sample count; this avoids ever reading past the limit."""
    paths = []
    for part in pattern.split(","):
        paths.extend(glob.glob(part.strip(), recursive=True))
    paths = sorted(set(paths))
    if not paths:
        raise FileNotFoundError(f"no shards matched {pattern}")
    if limit:
        # shuffle which shards get read (not the samples within — that would
        # need everything in memory first, the exact problem `limit` avoids)
        random.Random(0).shuffle(paths)
    samples = []
    for p in paths:
        with _opener(p)(p, "rb") as f:
            samples.extend(pickle.load(f))
        if limit and len(samples) >= limit:
            break
    if limit:
        samples = samples[:limit]
    return samples


class BCDataset(Dataset):
    """Wraps raw {obs, action, outcome} samples; encodes lazily in __getitem__
    (cheap — pure python dict indexing, no torch ops until collate)."""

    def __init__(self, raw_samples):
        self.raw = raw_samples

    def __len__(self):
        return len(self.raw)

    def __getitem__(self, i):
        d = self.raw[i]
        obs, action = d["obs"], d["action"]
        # SP samples carry an n-step bootstrapped value_target (see
        # selfplay_collect.py); plain BC samples fall back to terminal outcome.
        value = d.get("value_target", d.get("outcome", 0))
        sel = obs.get("select")
        enc = encode_sample(obs, sel)
        n_actions = enc["n_actions"]
        label = action[0] if action else 0
        label = min(label, n_actions - 1) if n_actions else 0
        # MCTS samples (future: mcts_collect.py) carry a soft policy_target —
        # normalized root visit counts, aligned to sel['option']. BC/direct-SP
        # samples have none; collate() falls back to a one-hot of `label`.
        policy_target = d.get("policy_target")
        if policy_target is not None:
            policy_target = list(policy_target[:n_actions]) if n_actions else []
        # advantage = value_target - v_pred (Stage 2 AWR, docs/nn-training.md
        # §3): only self-play samples carry v_pred (the collecting net's own
        # value-head estimate at s_t); BC samples have none, so advantage is
        # None there and train_sp.py falls back to a flat policy-loss weight.
        advantage = (value - d["v_pred"]) if "v_pred" in d else None
        # oracle: privileged opponent-hand card ids (oracle-critic collections
        # only; absent on plain BC/DAgger/SP samples → no-oracle training).
        opp_hand = d.get("opp_hand") or []
        opp_hand = [min(c, CARD_VOCAB - 1) for c in opp_hand if c]
        return enc, label, float(value or 0), policy_target, advantage, opp_hand


def collate(batch):
    B = len(batch)
    board_ids = torch.zeros(B, 13, dtype=torch.long)
    hand_ids = torch.zeros(B, 20, dtype=torch.long)
    discard_ids = torch.zeros(B, 20, dtype=torch.long)
    numeric = torch.zeros(B, NUM_FEATS, dtype=torch.float)
    action_type = torch.zeros(B, MAX_ACTIONS, dtype=torch.long)
    action_card = torch.zeros(B, MAX_ACTIONS, dtype=torch.long)
    action_attack = torch.zeros(B, MAX_ACTIONS, dtype=torch.long)
    action_numeric = torch.zeros(B, MAX_ACTIONS, 4, dtype=torch.float)
    action_mask = torch.zeros(B, MAX_ACTIONS, dtype=torch.float)
    labels = torch.zeros(B, dtype=torch.long)
    values = torch.zeros(B, dtype=torch.float)
    policy_targets = torch.zeros(B, MAX_ACTIONS, dtype=torch.float)
    advantages = torch.zeros(B, dtype=torch.float)
    has_advantage = torch.zeros(B, dtype=torch.float)
    oracle_ids = torch.zeros(B, 20, dtype=torch.long)
    oracle_flag = torch.zeros(B, 1, dtype=torch.float)

    for i, (enc, label, outcome, policy_target, advantage, opp_hand) in enumerate(batch):
        board_ids[i] = torch.tensor(enc["board_ids"], dtype=torch.long)
        h = enc["hand_ids"][:20]
        hand_ids[i, :len(h)] = torch.tensor(h, dtype=torch.long)
        dcd = enc["discard_ids"][:20]
        discard_ids[i, :len(dcd)] = torch.tensor(dcd, dtype=torch.long)
        numeric[i] = torch.tensor(enc["numeric"], dtype=torch.float)
        n = enc["n_actions"]
        for j, a in enumerate(enc["actions"][:MAX_ACTIONS]):
            action_type[i, j] = a["type"]
            action_card[i, j] = a["card_id"]
            action_attack[i, j] = a["attack_id"]
            action_numeric[i, j] = torch.tensor(a["numeric"], dtype=torch.float)
            action_mask[i, j] = 1.0
        labels[i] = label
        values[i] = outcome
        if policy_target and len(policy_target) == n:
            policy_targets[i, :n] = torch.tensor(policy_target, dtype=torch.float)
        elif n > 0:
            policy_targets[i, label] = 1.0  # one-hot fallback (BC / direct-SP samples)
        if advantage is not None:
            advantages[i] = advantage
            has_advantage[i] = 1.0
        oh = opp_hand[:20]
        if oh and random.random() >= ORACLE_DROPOUT:
            oracle_ids[i, :len(oh)] = torch.tensor(oh, dtype=torch.long)
            oracle_flag[i, 0] = 1.0

    return {
        "board_ids": board_ids, "hand_ids": hand_ids, "discard_ids": discard_ids,
        "numeric": numeric, "action_type": action_type, "action_card": action_card,
        "action_attack": action_attack, "action_numeric": action_numeric,
        "action_mask": action_mask, "labels": labels, "values": values,
        "policy_targets": policy_targets,
        "advantages": advantages, "has_advantage": has_advantage,
        "oracle_ids": oracle_ids, "oracle_flag": oracle_flag,
    }


In [ ]:
%%writefile training/nn/threat.py
"""Zero-sum-consistent threat estimation, per a user design session
2026-07-05: Φ (docs/nn-training.md, phi_baseline.py) is NOT actually
zero-sum -- prize_diff is antisymmetric (flips sign from the opponent's
seat), but hand_advantage/wall_penalty/line_progress are one-sided "my own
progress" measures with no opposing term. This module builds a genuinely
antisymmetric term: `my_threat_against(opp) - opp_threat_against(me)`, where
threat_against is a WELL-DEFINED one-sided function of state (independent of
whose "turn" it is), so the difference is antisymmetric by construction --
evaluated from the other seat, the two terms swap and the sign flips
automatically, matching the real 2-player zero-sum game.

Threat estimate: for a side's Active Pokemon, look up its REAL attacks
(cardId -> attackIds -> (damage, energy cost) via the local cg.api shim,
same real card database used by main.py's own deck), and score each attack
by how much of the opponent's current Active HP it could deal, discounted by
how many more turns of energy attachment (assumed 1/turn, ignoring color
requirements) would be needed to afford it:

    attack_threat = min(1, damage / defender_hp) / (1 + turns_to_afford)
    threat(attacker, defender) = max over attacker's known attacks

KNOWN LIMITATION, found and accepted rather than hidden during a real check
against `all_attack()`: static `Attack.damage` is 0 for our own Powerful
Hand (its real damage is computed dynamically from hand size via the skill
text, not a static field) -- this generalizes to ANY attack with
conditional/scaling damage, not just ours. This will systematically
undercount such attacks. Energy cost also ignores per-type color
requirements (treats `len(attack.energies)` as a fungible count) -- both are
deliberate simplifications, not oversights; a color-aware bag-matching
implementation would be the natural next refinement if this proves useful.

Requires `cg.api` on sys.path (training/setup_local_search.py's local shim,
or the real Kaggle environment) -- NOT available to main.py's own
dependency-free ladder submission; this is a training/eval-side module only.
"""
import os
import sys

_HERE = os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.dirname(os.path.dirname(_HERE))
_LOCAL_CG = os.path.join(_REPO_ROOT, "training", "local_cg")
if _LOCAL_CG not in sys.path:
    sys.path.insert(0, _LOCAL_CG)

from cg.api import all_card_data, all_attack  # noqa: E402

_CARD_ATTACKS = None
_ATTACK_INFO = None


def _load_tables():
    global _CARD_ATTACKS, _ATTACK_INFO
    if _CARD_ATTACKS is None:
        _CARD_ATTACKS = {c.cardId: c.attacks for c in all_card_data()}
        _ATTACK_INFO = {a.attackId: (a.damage, len(a.energies)) for a in all_attack()}


def _pk_id(pk):
    return (pk or {}).get("id", -1)


def _active(p):
    a = p.get("active")
    return a[0] if a and len(a) > 0 and a[0] else None


def threat_against(attacker_pk, defender_pk):
    """One-sided: how threatening is attacker_pk to defender_pk right now,
    in [0, 1] (0 if either side is missing/unknown or attacker has no
    attacks on record). Well-defined regardless of whose "turn" it is --
    same inputs always give the same output."""
    _load_tables()
    if not attacker_pk or not defender_pk:
        return 0.0
    defender_hp = defender_pk.get("hp") or 0
    if defender_hp <= 0:
        return 0.0
    attack_ids = _CARD_ATTACKS.get(_pk_id(attacker_pk)) or []
    if not attack_ids:
        return 0.0
    current_energy = len(attacker_pk.get("energies") or [])
    best = 0.0
    for aid in attack_ids:
        info = _ATTACK_INFO.get(aid)
        if not info:
            continue
        damage, needed = info
        if damage <= 0:
            continue  # 0-damage attacks (utility moves, or our own dynamic
            # Powerful Hand) carry no static threat signal here -- see
            # module docstring's known-limitation note
        turns_to_afford = max(0, needed - current_energy)
        score = min(1.0, damage / defender_hp) / (1 + turns_to_afford)
        best = max(best, score)
    return best


def net_threat_diff(cur, me_idx):
    """Antisymmetric by construction: evaluated with me_idx swapped to the
    opponent's seat, this returns the negation of this call's result, since
    it is literally my_threat - opp_threat computed from the SAME
    perspective-independent threat_against() calls either way."""
    players = cur.get("players") or []
    if len(players) != 2:
        return 0.0
    my_p, opp_p = players[me_idx], players[1 - me_idx]
    my_active, opp_active = _active(my_p), _active(opp_p)
    my_threat = threat_against(my_active, opp_active)
    opp_threat = threat_against(opp_active, my_active)
    return my_threat - opp_threat


In [ ]:
%%writefile training/nn/dmc_nstep.py
"""Phase 0 items 1+2 (docs/nn-training.md Phase 0): n-step bootstrapped value
targets, and an orthogonal potential-based Φ-shaping arm, for DMC-style
(s, taken_action, outcome) samples -- alternatives/additions to
train_dmc.py's current full-episode Monte Carlo target.

DMC treats the model's per-action logit as Q(s,a) (see train_dmc.py), so the
bootstrap value at a future state is max_a Q(s,a) at that state -- NOT the
separate value_head output (untrained/uncalibrated for DMC checkpoints,
since train_dmc.py never regresses it).

n-step alone (use_phi_shaping=False): pure n-step TD bootstrapping, no
intermediate reward --

    target_t = Q_max(s_{t+n})   if t+n is still within the game
             = outcome           if t+n reaches or passes the terminal decision

n_step=None (or >= the game's decision count) reduces exactly to the
existing full-Monte-Carlo target (every decision gets the flat game
outcome) -- this is the n="full" arm of the sweep and the current
train_dmc.py behavior, so this module is a strict superset, not a
replacement.

Φ-shaping (use_phi_shaping=True, orthogonal to n_step -- can combine with
any n_step value, including None/full-MC): adds the potential-based shaping
reward F_k = phi_gamma*Φ(s_{k+1}) - Φ(s_k) at every step along the
accumulation window (Ng/Harada/Russell 1999 -- provably policy-invariant,
i.e. doesn't change which action is optimal, only rescales/recenters the
absolute Q values). Φ is `phi_baseline.phi()`, the SAME fixed, never-fit
potential function already gated against 1356 real replays (ALL sign_acc
0.563, LATE 0.606 -- docs/report-log.md 2026-07-05 "Φ-only real-replay
baseline" entry) -- reused rather than reimplemented so the same validated
function is what both diagnoses the honest gate baseline AND drives
training. Φ(terminal) is defined as 0 by convention. Combined with n_step,
this is exactly n-step TD with potential-based shaping:

    G_t^(n) = sum_{k=t}^{end-1} phi_gamma^(k-t) * F_k
              + phi_gamma^(end-t) * [Q_max(s_end) if end<m else outcome]

which gives every decision in a game its OWN target shaped by how much
closer to winning that specific decision's follow-up state got, instead of
every decision in the game sharing one identical sparse ±1 label -- the
credit-assignment fix the Phase 0 diagnosis is about. Clipped to [-1, 1]
since combined with shaping the raw sum can exceed the Q-head's tanh-free
but Huber-loss-fit range.

CONFIRMED NEGATIVE RESULT (2026-07-05, see docs/report-log.md "Phase 0
ablation grid" + its root-cause follow-up entry): use_phi_shaping=True
trains the net toward target_t ~= outcome - Phi(s_t) (the full-MC telescoped
form), which is NOT comparable in sign to the true outcome across different
states -- Ng/Harada/Russell's policy-invariance guarantee is about
preserving the optimal ACTION RANKING WITHIN one state (Phi(s_t) is a
constant term across actions from that state), not about the shaped value's
sign meaning "win" vs "loss" relative to an external label. Measured: 11.5%
of training labels flip sign vs the true game outcome; that rate nearly
DOUBLES (21.1%) on real ladder replay states vs the self-play training
distribution, so a net that fits this target well in-distribution is
mechanically MORE wrong on real-replay sign-accuracy specifically, not just
noisier. Do not evaluate a Phi-shaped checkpoint with a cross-state sign-
accuracy gate; do not re-enable use_phi_shaping for training without first
redesigning how Phi is consumed (e.g. as a fixed leaf-eval/warm-start prior
kept OUTSIDE the regression target, not folded into it).

Must be called with each game's decisions in their real per-game, per-seat
order (exactly what bc_collect.extract_decisions returns) -- BEFORE they get
concatenated into a flat cross-game samples list, since game boundaries
can't be recovered after that point.
"""
import os
import sys

_HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, _HERE)

import torch  # noqa: E402

from net_common import load_model, encode_batch  # noqa: E402
from phi_baseline import phi as phi_state  # noqa: E402

PHI_GAMMA = 0.997  # matches selfplay_collect.py's GAMMA for consistency


def q_max_value(model, obs, sel):
    """max_a Q(s,a) at one decision point, using the DMC convention (action
    logit AS Q), not the model's separate value_head."""
    batch, n_actions = encode_batch(obs, sel)
    with torch.no_grad():
        logits, _ = model(*batch)
    return float(logits[0, :n_actions].max().item())


def _phi_at(d):
    """Reads the acting seat straight from the sample's own obs
    ('current'.'yourIndex') rather than assuming 0 -- extract_decisions()
    does NOT re-perspective samples to a fixed seat, so a learner playing as
    seat 1 in a given game has yourIndex=1 in every one of its samples."""
    obs = d.get("obs")
    cur = (obs or {}).get("current") or {}
    me_idx = cur.get("yourIndex")
    if me_idx is None:
        return 0.0
    try:
        v = phi_state(cur, me_idx)
    except Exception:
        v = None
    return v if v is not None else 0.0


def compute_nstep_targets(bootstrap_ckpt, decisions, outcome, n_step=None,
                           use_phi_shaping=False, phi_gamma=PHI_GAMMA):
    """decisions: ordered list of {'obs':..., 'action':...} for ONE game, ONE
    seat, in real play order (each obs's own 'current'.'yourIndex' identifies
    which seat -- extract_decisions does NOT renormalize this to 0/1). Adds
    'outcome' (the training target, in place -- keeps train_dmc.py's
    existing field name so it needs no changes) and 'mc_outcome' (the true
    full-episode label, always preserved for gating/diagnostics regardless
    of n_step/shaping)."""
    m = len(decisions)
    if m == 0:
        return
    use_bootstrap = n_step is not None and n_step < m
    values = None
    if use_bootstrap:
        model = load_model(bootstrap_ckpt)
        values = []
        for d in decisions:
            sel = d["obs"].get("select") if d.get("obs") else None
            if not sel or not sel.get("option"):
                values.append(0.0)
                continue
            try:
                values.append(q_max_value(model, d["obs"], sel))
            except Exception:
                values.append(0.0)

    phis = None
    if use_phi_shaping:
        phis = [_phi_at(d) for d in decisions]  # phis[m] (one past last) treated as 0.0 (terminal)

    for t, d in enumerate(decisions):
        d["mc_outcome"] = float(outcome)
        end = (t + n_step) if (n_step is not None) else m
        end = min(end, m)

        if not use_phi_shaping:
            if not use_bootstrap:
                d["outcome"] = float(outcome)
            else:
                d["outcome"] = values[end] if end < m else float(outcome)
            continue

        G, disc = 0.0, 1.0
        for k in range(t, end):
            phi_next = phis[k + 1] if k + 1 < m else 0.0
            F_k = phi_gamma * phi_next - phis[k]
            G += disc * F_k
            disc *= phi_gamma
        G += disc * (values[end] if (use_bootstrap and end < m) else float(outcome))
        d["outcome"] = max(-1.0, min(1.0, G))


In [ ]:
%%writefile training/nn/phi_baseline.py
"""Phase 0 gate baseline (docs/nn-training.md Phase 0 amendment (a)):
sign-accuracy of the hand-crafted potential function Phi(s) ALONE against
real ladder replays, before any learned n-step value head is trained. Phi is
outcome-correlated by construction (it's built from prize differential and
hand size, i.e. the actual win condition), so this is the real bar the
learned component must clear -- not a flat borrowed number from a different
experiment (the oracle-critic's 62.5%, measured on a different, self-play-
derived holdout, not this replay set).

Phi(s) = 2.0*prize_diff + 1.0*hand_advantage - 1.5*wall_penalty + 0.5*line_progress
  prize_diff      = (opp_prizes_left - my_prizes_left) / 6        in [-1, 1]
  hand_advantage  = clamp((hand_size - cards_needed_for_KO) / 10, -1, 1)
                    in [-1, 1] -- two real bugs found and fixed here in turn,
                    both via isolated-component sign-accuracy checks before
                    trusting the combined formula (a 1361-replay diagnostic
                    run each time):
                      (1) an uncapped raw hand_size/10 swamped prize_diff
                          late-game (hand size grows for BOTH players every
                          turn regardless of who's winning -- an absolute
                          quantity, not a relative one), driving LATE
                          sign-acc to 45.5%, BELOW chance and the opposite of
                          the expected late-game-should-be-most-predictable
                          pattern;
                      (2) capping the raw value at 1.0 fixed the swamping but
                          not the sign: hand_size is only meaningful relative
                          to what's needed for a KO (exactly how main.py's
                          own `at_threshold`/`cards_needed` features use it,
                          `ceil(opp_hp/PH_DMG_PER_CARD)`), not on its own --
                          an isolated-component check on 400 late-game
                          replays showed prize_diff ALONE at 65.8% sign-acc,
                          dropping to 52.2% once the raw hand term was added,
                          confirming the raw term was net-harmful, not just
                          weak. Subtracting cards_needed before normalizing
                          fixes this: now genuinely measures "how close to
                          actually being able to KO," not just "hand is
                          growing." Falls back to hand_size/10 (no
                          subtraction) only when the opponent's Active is
                          empty/HP unknown (cards_needed undefined).
  wall_penalty    = 1.0 if the opponent's Active currently blocks Powerful
                    Hand (Mist/Rock Energy observed), else 0.0
  line_progress   = census()['line_count'] / 2.0                   in [0, 1]
                    (also isolated-checked: 62.6% alone on the same 400-game
                    late slice, a real but weaker signal than prize_diff --
                    kept at low weight per its "setup progress," lowest-
                    priority role.)
Weights are hand-set once from the heuristic's own priority ordering (prize
lead > hand advantage > wall > setup progress), never fit to the outcome
labels -- required for potential-based shaping (Ng/Harada/Russell 1999) to
stay policy-invariant, and it's also what keeps this a fair, ungamed floor
for the learned value head to beat. Every component was sanity-checked in
isolation against real replay outcomes before being trusted in the combined
formula -- see docs/report-log.md 2026-07-05 "Phi-only baseline" entries for
the full diagnostic trail.

Usage:
  python training/nn/phi_baseline.py [--replays-dir replays/bulk] [--max-games N]
"""
import argparse
import glob
import json
import math
import os
import random
import sys

_HERE = os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.dirname(os.path.dirname(_HERE))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

import main as heuristic  # noqa: E402

OUR_TEAM = "Jason Anderson"


def our_seat(info):
    team_names = info.get("TeamNames", [])
    return team_names.index(OUR_TEAM) if OUR_TEAM in team_names else None


def phi(cur, me_idx):
    players = cur.get("players") or []
    if len(players) != 2:
        return None
    my_p, opp_p = players[me_idx], players[1 - me_idx]
    my_active = heuristic._active(my_p)
    bench = my_p.get("bench") or []
    hand_n = heuristic._hand_size(cur, me_idx)
    my_prizes = len(my_p.get("prize") or [])
    opp_prizes = len(opp_p.get("prize") or [])
    opp_active = heuristic._active(opp_p)
    wall = heuristic._opp_has_blocking_energy(opp_active)
    cen = heuristic._census(my_active, bench)

    prize_diff = (opp_prizes - my_prizes) / 6.0
    opp_hp = (opp_active or {}).get("hp") if opp_active else None
    if opp_hp:
        cards_needed = math.ceil(opp_hp / heuristic.PH_DMG_PER_CARD)
        hand_advantage = max(-1.0, min(1.0, (hand_n - cards_needed) / 10.0))
    else:
        hand_advantage = min(hand_n / 10.0, 1.0)
    wall_penalty = 1.0 if wall else 0.0
    line_progress = cen["line_count"] / 2.0
    return 2.0 * prize_diff + 1.0 * hand_advantage - 1.5 * wall_penalty + 0.5 * line_progress


def phi_v2(cur, me_idx):
    """Variant (2026-07-05 user design session): swaps the one-sided
    hand_advantage term for `threat.net_threat_diff` -- a genuinely
    antisymmetric (zero-sum: evaluated from the opponent's seat, negates
    exactly) estimate of "my attack readiness against them minus their
    attack readiness against me," built from the real card/attack database
    (damage, energy cost) via the local cg.api shim, rather than a deck-
    specific hand-size heuristic. See threat.py's docstring for the
    known limitations (0-damage/conditional-damage attacks undercounted,
    energy color requirements ignored). Everything else identical to phi().
    """
    players = cur.get("players") or []
    if len(players) != 2:
        return None
    my_p, opp_p = players[me_idx], players[1 - me_idx]
    my_active = heuristic._active(my_p)
    bench = my_p.get("bench") or []
    my_prizes = len(my_p.get("prize") or [])
    opp_prizes = len(opp_p.get("prize") or [])
    opp_active = heuristic._active(opp_p)
    wall = heuristic._opp_has_blocking_energy(opp_active)
    cen = heuristic._census(my_active, bench)

    from threat import net_threat_diff  # local import: needs cg.api on sys.path

    prize_diff = (opp_prizes - my_prizes) / 6.0
    ntd = net_threat_diff(cur, me_idx)
    wall_penalty = 1.0 if wall else 0.0
    line_progress = cen["line_count"] / 2.0
    return 2.0 * prize_diff + 1.0 * ntd - 1.5 * wall_penalty + 0.5 * line_progress


def extract_rows(path, phi_fn=phi):
    """Returns list of (phi_value, outcome, turn) for our-seat decision points
    in one game, or [] if the game is unusable (no our-team seat, no clean
    terminal reward, etc)."""
    try:
        d = json.load(open(path, encoding="utf-8"))
    except Exception:
        return []
    info = d.get("info", {})
    you = our_seat(info)
    if you is None:
        return []
    rewards = d.get("rewards")
    if not rewards or len(rewards) != 2 or rewards[you] not in (1, -1):
        return []
    outcome = rewards[you]

    rows = []
    for step in d.get("steps", []):
        if len(step) <= you:
            continue
        rec = step[you]
        obs = rec.get("observation") if rec else None
        if not obs:
            continue
        cur = obs.get("current") or {}
        if cur.get("yourIndex") != you:
            continue
        sel = obs.get("select")
        if not sel or not sel.get("option"):
            continue
        turn = cur.get("turn") or 0
        try:
            v = phi_fn(cur, you)
        except Exception:
            continue
        if v is None:
            continue
        rows.append((v, outcome, turn))
    return rows


def bootstrap_ci(game_rows, n_resamples=2000, seed=13):
    """Game-level bootstrap: resample GAMES (not individual decisions) with
    replacement, since decisions within one game are highly correlated --
    per-state CIs would overstate power on a replay corpus with only a few
    hundred distinct games (docs/nn-training.md Phase 0 amendment (e))."""
    rng = random.Random(seed)
    n_games = len(game_rows)
    accs = []
    for _ in range(n_resamples):
        sample_games = [game_rows[rng.randrange(n_games)] for _ in range(n_games)]
        flat = [r for g in sample_games for r in g]
        if not flat:
            continue
        acc = sum(1 for v, o, _ in flat if (v >= 0) == (o >= 0)) / len(flat)
        accs.append(acc)
    accs.sort()
    lo = accs[int(0.025 * len(accs))]
    hi = accs[int(0.975 * len(accs)) - 1]
    return lo, hi


def report(label, game_rows):
    flat = [r for g in game_rows for r in g]
    if not flat:
        print(f"{label}: no samples")
        return
    n = len(flat)
    acc = sum(1 for v, o, _ in flat if (v >= 0) == (o >= 0)) / n
    lo, hi = bootstrap_ci(game_rows) if game_rows else (float("nan"), float("nan"))
    print(f"{label}: n_decisions={n} n_games={len(game_rows)} sign_acc={acc:.3f} "
          f"game_level_95%CI=[{lo:.3f}, {hi:.3f}]")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--replays-dir", default=os.path.join(_REPO_ROOT, "replays", "bulk"))
    ap.add_argument("--max-games", type=int, default=None)
    ap.add_argument("--version", type=int, default=1, choices=[1, 2],
                     help="1 = original Φ (hand_advantage), 2 = threat.net_threat_diff variant")
    args = ap.parse_args()
    phi_fn = phi if args.version == 1 else phi_v2

    paths = sorted(glob.glob(os.path.join(args.replays_dir, "*.json")))
    if args.max_games:
        paths = paths[: args.max_games]

    all_games, early_games, mid_games, late_games = [], [], [], []
    skipped = 0
    for p in paths:
        rows = extract_rows(p, phi_fn)
        if not rows:
            skipped += 1
            continue
        all_games.append(rows)
        early = [r for r in rows if r[2] <= 4]
        mid = [r for r in rows if 5 <= r[2] <= 10]
        late = [r for r in rows if r[2] >= 11]
        if early:
            early_games.append(early)
        if mid:
            mid_games.append(mid)
        if late:
            late_games.append(late)

    print(f"replay files scanned={len(paths)} usable_games={len(all_games)} skipped={skipped}")
    report("ALL", all_games)
    report("EARLY (turn<=4)", early_games)
    report("MID (5<=turn<=10)", mid_games)
    report("LATE (turn>=11)", late_games)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile training/nn/train_sp.py
"""Self-play training step: warm-starts from a BC (or prior SP) checkpoint,
trains on a 40% BC / 60% SP mix (non-negotiable per docs/nn-training.md —
SP-only collapsed the prior project's attempt: 46%->20% vs teacher in 3 iters).

Usage:
  python train_sp.py --bc-data "../bc_data*.pkl.gz" --sp-data "../sp_data.pkl.gz" \
      --init ../ptcg_bc_v1.pth --out ../ptcg_sp_iter1.pth --epochs 3
"""
import argparse
import math
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, ConcatDataset

from dataset import BCDataset, collate, load_shards
from model import PTCGNet


def awr_normalizer(sp_raw, beta):
    """Mean of exp(advantage/beta) over SP samples that carry a v_pred
    (docs/nn-training.md Stage 2) — used to rescale weights to mean ~1.0
    within the SP portion, so AWR doesn't silently shift the "non-negotiable"
    40% BC / 60% SP batch composition away from its target ratio."""
    ws = [math.exp(max(-10.0, min(10.0, (s["value_target"] - s["v_pred"]) / beta)))
          for s in sp_raw if "v_pred" in s]
    return (sum(ws) / len(ws)) if ws else 1.0


def build_mixed_loader(bc_raw, sp_raw, batch_size, bc_frac=0.4):
    bc_ds = BCDataset(bc_raw)
    sp_ds = BCDataset(sp_raw)
    combined = ConcatDataset([bc_ds, sp_ds])
    n_bc, n_sp = len(bc_ds), len(sp_ds)
    # per-sample weight so the EXPECTED batch composition is bc_frac/  (1-bc_frac),
    # regardless of how imbalanced the two pools are in raw sample count.
    w_bc = bc_frac / max(n_bc, 1)
    w_sp = (1 - bc_frac) / max(n_sp, 1)
    weights = [w_bc] * n_bc + [w_sp] * n_sp
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    return DataLoader(combined, batch_size=batch_size, sampler=sampler, collate_fn=collate)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bc-data", default="../bc_data*.pkl.gz")
    ap.add_argument("--sp-data", default="../sp_data.pkl.gz")
    ap.add_argument("--init", default="../ptcg_bc_v1.pth")
    ap.add_argument("--out", default="../ptcg_sp_iter1.pth")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch-size", type=int, default=128)
    ap.add_argument("--lr", type=float, default=5e-5)  # lower than BC warmup — fine-tuning
    ap.add_argument("--bc-frac", type=float, default=0.4)
    ap.add_argument("--steps-per-epoch", type=int, default=2000)
    ap.add_argument("--bc-limit", type=int, default=None,
                     help="cap BC raw samples loaded into RAM (each source is "
                          "resampled with replacement anyway, so a large corpus "
                          "gains nothing but memory pressure over a capped one)")
    ap.add_argument("--sp-limit", type=int, default=None)
    ap.add_argument("--awr-beta", type=float, default=1.0,
                     help="temperature for the AWR policy-loss weight "
                          "exp(advantage/beta) (Stage 2, docs/nn-training.md); "
                          "advantages are ~[-2,2] (both terms tanh-clipped), "
                          "beta=1.0 is the calibrated default")
    ap.add_argument("--awr-clip", type=float, default=20.0,
                     help="clip normalized AWR weights to [1/awr-clip, awr-clip] "
                          "so a few high-advantage samples can't dominate the gradient")
    ap.add_argument("--winner-only", action="store_true",
                     help="dumb-baseline ablation (docs/nn-training.md Stage 2): "
                          "drop AWR weighting, just filter SP samples to winning "
                          "games (outcome > 0)")
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device}")

    bc_raw = load_shards(args.bc_data, limit=args.bc_limit)
    sp_raw = load_shards(args.sp_data, limit=args.sp_limit)
    if args.winner_only:
        sp_raw = [s for s in sp_raw if s.get("outcome", 0) > 0]
        print(f"--winner-only: filtered sp_samples to {len(sp_raw)} (outcome>0)")
    print(f"bc_samples={len(bc_raw)} sp_samples={len(sp_raw)}")

    awr_norm = 1.0 if args.winner_only else awr_normalizer(sp_raw, args.awr_beta)
    print(f"awr_norm={awr_norm:.4f} beta={args.awr_beta} winner_only={args.winner_only}")

    loader = build_mixed_loader(bc_raw, sp_raw, args.batch_size, args.bc_frac)

    model = PTCGNet().to(device)
    state = torch.load(args.init, map_location=device)
    own = model.state_dict()
    state = {k: v for k, v in state.items() if k in own and v.shape == own[k].shape}
    model.load_state_dict(state, strict=False)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)
    value_loss_fn = nn.HuberLoss(delta=0.2)

    for epoch in range(args.epochs):
        total_loss = 0.0
        n_batches = 0
        for batch in loader:
            if n_batches >= args.steps_per_epoch:
                break
            batch = {k: v.to(device) for k, v in batch.items()}
            logits, value = model(
                batch["board_ids"], batch["hand_ids"], batch["discard_ids"],
                batch["numeric"], batch["action_type"], batch["action_card"],
                batch["action_attack"], batch["action_numeric"], batch["action_mask"])
            # Soft-target cross-entropy against policy_targets: a strict
            # generalization of CrossEntropyLoss(logits, labels) — dataset.py's
            # collate() fills policy_targets with a one-hot(label) fallback for
            # plain BC/direct-SP samples (no MCTS visit counts yet), so this is
            # mathematically identical to hard CE until mcts_collect.py starts
            # producing real soft targets.
            log_probs = torch.log_softmax(logits, dim=-1)
            per_sample_p_loss = -(batch["policy_targets"] * log_probs).sum(dim=-1)
            if args.winner_only:
                weight = torch.ones_like(per_sample_p_loss)
            else:
                # AWR (Stage 2): BC samples (has_advantage==0) get weight 1.0;
                # SP samples get exp(advantage/beta), rescaled by the corpus-
                # level normalizer so the SP portion's mean weight stays ~1.0
                # (protects the non-negotiable 40/60 BC/SP mix) and clipped so
                # a few high-advantage samples can't dominate the gradient.
                raw_w = torch.exp(torch.clamp(batch["advantages"] / args.awr_beta, -10.0, 10.0))
                sp_w = torch.clamp(raw_w / awr_norm, 1.0 / args.awr_clip, args.awr_clip)
                weight = torch.where(batch["has_advantage"] > 0, sp_w, torch.ones_like(sp_w))
            p_loss = (weight * per_sample_p_loss).mean()
            v_loss = value_loss_fn(value, batch["values"])
            loss = p_loss + 0.5 * v_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_batches += 1
        print(f"epoch {epoch}: avg_loss={total_loss/max(n_batches,1):.4f} ({n_batches} steps)")

    torch.save(model.state_dict(), args.out)
    print(f"saved {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile training/nn/dmc_replay_gate.py
"""Phase 0 gate (docs/nn-training.md Phase 0 amendment (a)/(e)): sign-accuracy
of a trained DMC checkpoint's Q-head (max_a Q(s,a), the DMC convention -- see
dmc_nstep.py) against the same 1356-game real ladder replay corpus
phi_baseline.py was gated on, using the same game-level bootstrapped CI
methodology so the two numbers are directly comparable. A candidate passes
Phase 0's gate only if this beats phi_baseline's Φ-only numbers (ALL 0.563
[0.543, 0.583], LATE 0.606 [0.576, 0.635]) by a statistically meaningful
margin -- not the flat, differently-measured 62.5% oracle-critic figure.

Usage:
  python training/nn/dmc_replay_gate.py --ckpt training/ptcg_dmc_r2.pth [--replays-dir replays/bulk] [--max-games N]
"""
import argparse
import glob
import json
import os
import sys

_HERE = os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.dirname(os.path.dirname(_HERE))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)
sys.path.insert(0, _HERE)

from net_common import load_model, value_estimate  # noqa: E402
from dmc_nstep import q_max_value  # noqa: E402
from phi_baseline import our_seat, bootstrap_ci  # noqa: E402


def extract_rows(path, model, value_source="qmax"):
    try:
        d = json.load(open(path, encoding="utf-8"))
    except Exception:
        return []
    info = d.get("info", {})
    you = our_seat(info)
    if you is None:
        return []
    rewards = d.get("rewards")
    if not rewards or len(rewards) != 2 or rewards[you] not in (1, -1):
        return []
    outcome = rewards[you]

    rows = []
    for step in d.get("steps", []):
        if len(step) <= you:
            continue
        rec = step[you]
        obs = rec.get("observation") if rec else None
        if not obs:
            continue
        cur = obs.get("current") or {}
        if cur.get("yourIndex") != you:
            continue
        sel = obs.get("select")
        if not sel or not sel.get("option"):
            continue
        turn = cur.get("turn") or 0
        try:
            v = value_estimate(model, obs, sel) if value_source == "head" else q_max_value(model, obs, sel)
        except Exception:
            continue
        rows.append((v, outcome, turn))
    return rows


def report(label, game_rows):
    flat = [r for g in game_rows for r in g]
    if not flat:
        print(f"{label}: no samples")
        return
    n = len(flat)
    acc = sum(1 for v, o, _ in flat if (v >= 0) == (o >= 0)) / n
    lo, hi = bootstrap_ci(game_rows) if game_rows else (float("nan"), float("nan"))
    print(f"{label}: n_decisions={n} n_games={len(game_rows)} sign_acc={acc:.3f} "
          f"game_level_95%CI=[{lo:.3f}, {hi:.3f}]")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--replays-dir", default=os.path.join(_REPO_ROOT, "replays", "bulk"))
    ap.add_argument("--max-games", type=int, default=None)
    ap.add_argument("--value-source", choices=["qmax", "head"], default="qmax",
                     help="qmax = DMC convention (max_a logits, default); "
                          "head = the model's own value_head output "
                          "(use for train_sp.py-trained checkpoints, whose "
                          "logits are policy preferences, not Q-values)")
    args = ap.parse_args()

    model = load_model(args.ckpt)
    paths = sorted(glob.glob(os.path.join(args.replays_dir, "*.json")))
    if args.max_games:
        paths = paths[: args.max_games]

    all_games, early_games, mid_games, late_games = [], [], [], []
    skipped = 0
    for p in paths:
        rows = extract_rows(p, model, value_source=args.value_source)
        if not rows:
            skipped += 1
            continue
        all_games.append(rows)
        early = [r for r in rows if r[2] <= 4]
        mid = [r for r in rows if 5 <= r[2] <= 10]
        late = [r for r in rows if r[2] >= 11]
        if early:
            early_games.append(early)
        if mid:
            mid_games.append(mid)
        if late:
            late_games.append(late)

    print(f"ckpt={args.ckpt} value_source={args.value_source} "
          f"replay files scanned={len(paths)} usable_games={len(all_games)} skipped={skipped}")
    report("ALL", all_games)
    report("EARLY (turn<=4)", early_games)
    report("MID (5<=turn<=10)", mid_games)
    report("LATE (turn>=11)", late_games)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile opponents/lucario_agent.py
"""
Mega Lucario ex training opponent — official Kaggle sample agent.
Source: kiyotah/a-sample-rule-based-agent-mega-lucario-ex-deck

Archetype: Fighting aggro, multi-attacker (Mega Lucario ex / Hariyama / Solrock).
Mega Lucario ex: 340 HP, Stage 1 megaEx, 3 prizes.
  Aura Jab {F} 130 + attach 3 Fighting from discard to bench.
  Mega Brave {F}{F} 270 (can't reuse next turn).
"""
import sys, glob
from collections import defaultdict

for _p in ['/kaggle/input/**/cg-lib', '/kaggle/input/cg-lib']:
    _m = glob.glob(_p, recursive=True)
    if _m: sys.path.insert(0, _m[0]); break

try:
    from cg.api import AreaType, CardType, EnergyType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class
    all_card = all_card_data()
    card_table = {c.cardId: c for c in all_card}
except Exception:
    card_table = {}
    # Stub classes so the module imports cleanly without cg-lib
    class _Stub:
        def __getattr__(self, n): return None
    AreaType = SelectContext = OptionType = EnergyType = CardType = _Stub()
    to_observation_class = lambda x: x

# ── Deck (60 cards, embedded) ─────────────────────────────────────────────────
Makuhita           = 673   # ×2
Hariyama           = 674   # ×2
Lunatone           = 675   # ×2
Solrock            = 676   # ×3
Riolu              = 677   # ×3
Mega_Lucario_ex    = 678   # ×4
Dusk_Ball          = 1102  # ×4
Switch             = 1123  # ×2
Premium_Power_Pro  = 1141  # ×4
Fighting_Gong      = 1142  # ×4
Poke_Pad           = 1152  # ×4
Hero_Cape          = 1159  # ×1
Boss_Orders        = 1182  # ×2
Carmine            = 1192  # ×4
Lillie_Determination = 1227  # ×4
Gravity_Mountain   = 1252  # ×2
Basic_Fighting_Energy = 6  # ×13

DECK = (
    [Makuhita] * 2 + [Hariyama] * 2 + [Lunatone] * 2 + [Solrock] * 3 +
    [Riolu] * 3 + [Mega_Lucario_ex] * 4 +
    [Dusk_Ball] * 4 + [Switch] * 2 + [Premium_Power_Pro] * 4 +
    [Fighting_Gong] * 4 + [Poke_Pad] * 4 + [Hero_Cape] +
    [Boss_Orders] * 2 + [Carmine] * 4 + [Lillie_Determination] * 4 +
    [Gravity_Mountain] * 2 + [Basic_Fighting_Energy] * 13
)
assert len(DECK) == 60, f"Lucario deck has {len(DECK)} cards, expected 60"


class AttackPlan:
    attacker = -1
    target = -1
    attack_index = -1
    remain_hp = -1
    energy = False


plan = AttackPlan()
pre_turn = 0
ability_used = False


def get_card(obs, area, index, player_index):
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:    return obs.select.deck[index]
        case AreaType.HAND:    return ps.hand[index]
        case AreaType.DISCARD: return ps.discard[index]
        case AreaType.ACTIVE:  return ps.active[index]
        case AreaType.BENCH:   return ps.bench[index]
        case AreaType.PRIZE:   return ps.prize[index]
        case AreaType.STADIUM: return obs.current.stadium[index]
        case AreaType.LOOKING: return obs.current.looking[index]
        case _:                return None


def prize_count(pokemon):
    data = card_table.get(pokemon.id)
    if not data: return 1
    count = 3 if data.megaEx else 2 if data.ex else 1
    for card in pokemon.energyCards:
        if card.id == 12:  # Legacy Energy
            count -= 1
    for card in pokemon.tools:
        if card.id == 1172 and "Lillie" in data.name:
            count -= 1
    return max(0, count)


def pokemon_score(pokemon):
    data = card_table.get(pokemon.id)
    if not data: return 0
    score = prize_count(pokemon) * 1000
    score += len(pokemon.energies) * 150
    score += len(pokemon.tools) * 100
    if data.stage2:  score += 250
    elif data.stage1: score += 130
    pid = pokemon.id
    if pid in (173, 174, 190, 1071):  # Noctowl, Fan Rotom, Archaludon ex, Meowth ex
        score -= 200
    if pid == 112 and len(pokemon.energies) >= 1:  # Munkidori
        score += 300
    score += pokemon.hp
    return score


def agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        return DECK

    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]
    my_prize = len(my_state.prize)

    global plan, pre_turn, ability_used
    if pre_turn != state.turn:
        pre_turn = state.turn
        plan = AttackPlan()
        ability_used = False

    field_counts = defaultdict(int)
    hand_counts = defaultdict(int)
    discard_counts = defaultdict(int)

    attacker1 = False
    attacker2 = False
    for card in my_state.active + my_state.bench:
        if card is None: continue
        field_counts[card.id] += 1
        if card.id in (Makuhita, Hariyama):
            if len(card.energies) >= 3: attacker2 = True
        elif card.id in (Riolu, Mega_Lucario_ex):
            if len(card.energies) >= 2: attacker1 = True

    for card in my_state.hand:
        hand_counts[card.id] += 1

    for card in my_state.discard:
        discard_counts[card.id] += 1

    stadium_id = 0
    for card in state.stadium:
        stadium_id = card.id

    can_attack = False
    if context == SelectContext.MAIN:
        can_switch = False
        can_op_switch = False
        can_use_mega_brave = False
        for o in select.option:
            if o.type == OptionType.PLAY:
                card = get_card(obs, AreaType.HAND, o.index, my_index)
                if card.id == Switch:         can_switch = True
                elif card.id == Boss_Orders:  can_op_switch = True
            elif o.type == OptionType.EVOLVE:
                card = get_card(obs, AreaType.HAND, o.index, my_index)
                if card.id == Hariyama: can_op_switch = True
            elif o.type == OptionType.RETREAT:
                can_switch = True
            elif o.type == OptionType.ATTACK:
                can_attack = True
                if o.attackId == 983:  # Mega Brave
                    can_use_mega_brave = True

        my_cards = [my_state.active[0]] + list(my_state.bench)
        op_cards = [op_state.active[0]] + list(op_state.bench)

        if state.turn >= 2:
            best_score = -1
            for i, my_pokemon in enumerate(my_cards):
                if i != 0 and not can_switch: break
                for a in range(2):
                    energy_required = 0
                    base_damage = 0
                    base_score = 0
                    if my_pokemon.id == Mega_Lucario_ex:
                        if a == 0:
                            energy_required = 1
                            base_damage = 130
                            base_score += 60 * min(3, discard_counts[Basic_Fighting_Energy])
                        else:
                            energy_required = 2
                            base_damage = 270
                        if my_prize in (2, 3): base_score -= 500
                    elif a == 1:
                        break
                    elif my_pokemon.id == Hariyama:
                        energy_required = 3
                        base_damage = 210
                    elif my_pokemon.id == Makuhita:
                        for o in select.option:
                            if o.type == OptionType.EVOLVE:
                                index = o.inPlayIndex
                                if o.inPlayArea == AreaType.BENCH: index += 1
                                if index == i: break
                        else:
                            break
                        base_score -= 100
                        energy_required = 3
                        base_damage = 210
                    elif my_pokemon.id == Solrock:
                        if field_counts[Lunatone] >= 1:
                            energy_required = 1
                            base_damage = 70
                    if base_damage <= 0: continue

                    more_energy = False
                    energy_count = len(my_pokemon.energies)
                    if a == 1 and i == 0 and energy_count >= 2 and not can_use_mega_brave:
                        break
                    if energy_count < energy_required:
                        if hand_counts[Basic_Fighting_Energy] >= 1 and not state.energyAttached:
                            energy_count += 1
                            if energy_count < energy_required: continue
                            else: more_energy = True
                        else:
                            continue

                    for j, op_pokemon in enumerate(op_cards):
                        if j != 0 and not can_op_switch: break
                        damage = base_damage
                        data = card_table.get(op_pokemon.id)
                        if data:
                            if data.weakness == EnergyType.FIGHTING:   damage *= 2
                            elif data.resistance == EnergyType.FIGHTING: damage -= 30
                        prize = 0
                        score = pokemon_score(op_pokemon)
                        if op_pokemon.hp <= damage:
                            prize = prize_count(op_pokemon)
                        else:
                            score *= damage / op_pokemon.hp
                        score += base_score
                        if len(op_state.prize) <= prize: score = 50000
                        if i == 0: score += 220
                        if j == 0: score += 300
                        score += energy_count
                        if best_score < score:
                            best_score = score
                            plan.attacker = i
                            plan.target = j
                            plan.attack_index = a
                            plan.remain_hp = op_pokemon.hp - damage
                            plan.energy = more_energy

    def energy_score(pokemon, active):
        energy_count = len(pokemon.energies)
        score = 8000
        if active: score += 10
        if pokemon.id in (Makuhita, Hariyama):
            if pokemon.id == Hariyama: score += 1
            if energy_count < 3:  score += 100
            if attacker2:         score -= 50
        elif pokemon.id == Lunatone:
            score -= 100
        elif pokemon.id == Solrock:
            if energy_count < 1: score += 20
            else:                score -= 100
        elif pokemon.id in (Riolu, Mega_Lucario_ex):
            if pokemon.id == Mega_Lucario_ex: score += 1
            if energy_count < 2: score += 100
            if attacker1:        score -= 50
        return score

    scores = []
    for o in select.option:
        score = 0
        if o.type == OptionType.NUMBER:
            score = o.number
        elif o.type == OptionType.YES:
            score = 1
        elif o.type == OptionType.CARD:
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card is not None:
                energy_count = 0
                if isinstance(card, Pokemon): energy_count = len(card.energies)
                if context in (SelectContext.SWITCH, SelectContext.TO_ACTIVE):
                    if o.playerIndex == my_index:
                        score += energy_count * 2
                        if o.index == plan.attacker - 1: score += 100
                        if card.id == Mega_Lucario_ex:
                            score += 8 if my_prize in (2, 3) else 20
                        elif card.id == Hariyama and energy_count >= 2: score += 15
                        elif card.id == Makuhita and energy_count >= 2: score += 10
                        elif card.id == Solrock: score += 5
                        elif card.id == Riolu:   score += 4
                    else:
                        if o.index == plan.target - 1: score += 100
                elif context == SelectContext.SETUP_ACTIVE_POKEMON:
                    if card.id == Solrock:
                        score = 2 if state.firstPlayer == my_index else 4
                    elif card.id == Riolu:   score = 3
                    elif card.id == Makuhita: score = 1
                elif context == SelectContext.TO_HAND:
                    score = 200 - hand_counts[card.id] * 100
                    if card.id == Makuhita:
                        score += 10 if field_counts[card.id] < 1 else -10
                    elif card.id == Hariyama:
                        score += 20 if field_counts[Makuhita] >= 1 else -20
                    elif card.id == Lunatone:
                        score += -250 if field_counts[card.id] >= 1 else 60
                    elif card.id == Solrock:
                        score += -250 if field_counts[card.id] >= 1 else 50
                    elif card.id == Riolu:
                        n_line = field_counts[Riolu] + field_counts[Mega_Lucario_ex]
                        if n_line >= 2:   score -= 150
                        elif n_line >= 1: score -= 3
                        else:             score += 40
                    elif card.id == Mega_Lucario_ex:
                        score += 40 if field_counts[Riolu] >= 1 else -15
                    elif card.id == Basic_Fighting_Energy:
                        if not ability_used or not state.energyAttached: score += 30
                        else:                                              score -= 1
                elif context == SelectContext.ATTACH_FROM:
                    score = energy_score(card, o.area == AreaType.ACTIVE)
        elif o.type == OptionType.PLAY:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            data = card_table.get(card.id)
            if data and data.cardType == CardType.POKEMON:
                score = 20000
                if card.id in (Lunatone, Solrock):
                    if field_counts[card.id] >= 1: score = -1
                elif card.id == Riolu:
                    if field_counts[Riolu] + field_counts[Mega_Lucario_ex] >= 2: score = -1
            else:
                score = 10000
                if card.id == Switch:
                    score = 6000 if plan.attacker > 0 else -1
                elif card.id == Premium_Power_Pro:
                    if state.supporterPlayed and plan.remain_hp <= 0: score = -1
                    elif not can_attack:
                        if not state.supporterPlayed and hand_counts[Carmine] > 0 and hand_counts[Lillie_Determination] == 0:
                            score = 3050
                        else: score = -1
                    else: score = 5000
                elif card.id == Boss_Orders:
                    score = 3200 if plan.target >= 1 else -1
                elif card.id == Carmine:           score = 3000
                elif card.id == Lillie_Determination: score = 3100
                elif card.id == Gravity_Mountain:
                    if stadium_id == 0: score = -1
        elif o.type == OptionType.ATTACH:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            if card.id == Hero_Cape:
                score = 7000
                if pokemon.id == Riolu:         score += 100
                elif pokemon.id == Mega_Lucario_ex: score += 200
            else:
                score = energy_score(pokemon, o.inPlayArea == AreaType.ACTIVE)
                if o.inPlayArea == AreaType.ACTIVE:
                    if plan.attacker == 0 and plan.energy: score += 200
                else:
                    if plan.attacker == 1 + o.inPlayIndex and plan.energy: score += 200
        elif o.type == OptionType.EVOLVE:
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = 9000 + len(pokemon.energies)
            if pokemon.id == Makuhita and plan.target == 0: score = -1
        elif o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card.id == 1267:  # Lumiose City
                score = 1
            else:
                score = 30000
        elif o.type == OptionType.RETREAT:
            score = 2000 if plan.attacker >= 1 else -1
        elif o.type == OptionType.ATTACK:
            score = 1000
            if plan.attack_index == 1:
                if o.attackId == 983: score += 100  # Mega Brave
            else:
                if o.attackId != 983: score += 100

        scores.append(score)

    desc_indices = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]
    if context == SelectContext.MAIN:
        o = select.option[desc_indices[0]]
        if o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card.id == Lunatone: ability_used = True
    return desc_indices[:select.maxCount]


In [ ]:
%%writefile opponents/dragapult_agent.py
"""
Dragapult ex training opponent — official Kaggle sample agent.
Source: kiyotah/a-sample-rule-based-agent-dragapult-ex-deck

Archetype: Stage 2 spread. Phantom Dive: 200 + 6 damage counters on opponent bench.
Dragapult ex: 320 HP, Stage 2 ex, Tera Dragon (bench protection while benched).
"""
import sys, glob
from collections import defaultdict

for _p in ['/kaggle/input/**/cg-lib', '/kaggle/input/cg-lib']:
    _m = glob.glob(_p, recursive=True)
    if _m: sys.path.insert(0, _m[0]); break

try:
    from cg.api import AreaType, CardType, Log, LogType, Observation, SelectContext, OptionType, Card, Pokemon, State, all_card_data, to_observation_class
    all_card = all_card_data()
    card_table = {c.cardId: c for c in all_card}
except Exception:
    card_table = {}
    class _Stub:
        def __getattr__(self, n): return None
    AreaType = SelectContext = OptionType = CardType = LogType = _Stub()
    to_observation_class = lambda x: x
    # real (empty) classes, not _Stub instances: these names are used in
    # isinstance() checks at runtime (e.g. add_card_count), which need types —
    # their absence crashed 100% of local games as NameError: 'Pokemon'
    class Pokemon: pass
    class Card: pass
    class State: pass
    class Log: pass
    class Observation: pass

# ── Deck (60 cards, embedded) ─────────────────────────────────────────────────
Dreepy               = 119   # ×4
Drakloak             = 120   # ×4
Dragapult_ex         = 121   # ×3
Fezandipiti_ex       = 140   # ×1
Latias_ex            = 184   # ×1
Budew                = 235   # ×2
Meowth_ex            = 1071  # ×1
Rare_Candy           = 1079  # ×2
Unfair_Stamp         = 1080  # ×1
Buddy_Buddy_Poffin   = 1086  # ×4
Night_Stretcher      = 1097  # ×2
Crushing_Hammer      = 1120  # ×4
Ultra_Ball           = 1121  # ×4
Poke_Pad             = 1152  # ×3
Lucky_Helmet         = 1156  # ×1
Boss_Orders          = 1182  # ×3
Crispin              = 1198  # ×4
Brock_Scouting       = 1210  # ×2
Lillie_Determination = 1227  # ×4
Team_Rocket_Watchtower = 1256  # ×2
Basic_Fire_Energy    = 2     # ×4
Basic_Psychic_Energy = 5     # ×4

DECK = (
    [Dreepy] * 4 + [Drakloak] * 4 + [Dragapult_ex] * 3 +
    [Fezandipiti_ex] + [Latias_ex] + [Budew] * 2 + [Meowth_ex] +
    [Rare_Candy] * 2 + [Unfair_Stamp] + [Buddy_Buddy_Poffin] * 4 +
    [Night_Stretcher] * 2 + [Crushing_Hammer] * 4 + [Ultra_Ball] * 4 +
    [Poke_Pad] * 3 + [Lucky_Helmet] + [Boss_Orders] * 3 +
    [Crispin] * 4 + [Brock_Scouting] * 2 + [Lillie_Determination] * 4 +
    [Team_Rocket_Watchtower] * 2 + [Basic_Fire_Energy] * 4 + [Basic_Psychic_Energy] * 4
)
assert len(DECK) == 60, f"Dragapult deck has {len(DECK)} cards, expected 60"

UNNECESSARY = -10000000

class AttackPlan:
    attack: int = 0
    counter: list = []

can_switch = False
can_attack = False
can_main_attack = False
can_energy_attach = False
use_support = 0
bench_attacker = False
pre_turn_log = []
current_turn_log = []

prize = []
card_counts = defaultdict(int)
serial_set = set()
plan_a = AttackPlan()
plan_b = AttackPlan()


def no_damage_dex(pid):
    # Drednaw, Milotic ex, Sylveon, Crustle
    return pid in (158, 207, 330, 345)


def no_damage_counter(pokemon):
    if pokemon.id in (28, 199, 203, 207, 362, 1136):
        return True
    for card in pokemon.energyCards:
        if card.id in (11, 20):  # Mist Energy, Rock Fighting Energy
            return True
    return False


def prize_count(pokemon, is_attack_damage):
    data = card_table.get(pokemon.id)
    if not data: return 1
    count = 3 if data.megaEx else 2 if data.ex else 1
    if is_attack_damage:
        for card in pokemon.energyCards:
            if card.id == 12: count -= 1
        for card in pokemon.tools:
            if card.id == 1172 and "Lillie" in data.name: count -= 1
    return max(0, count)


def pokemon_score(pokemon, is_attack_damage):
    data = card_table.get(pokemon.id)
    if not data: return 0
    score = prize_count(pokemon, is_attack_damage) * 1000
    score += len(pokemon.energies) * 150
    score += len(pokemon.tools) * 100
    if data.stage2:   score += 250
    elif data.stage1: score += 130
    pid = pokemon.id
    if pid in (173, 174, 190, 1071): score -= 200
    if pid == 112 and len(pokemon.energies) >= 1: score += 300
    score += pokemon.hp
    return score


def add_card_count(card, my_index):
    if card is None: return
    if isinstance(card, Pokemon) or card.playerIndex == my_index:
        if card.serial not in serial_set:
            card_counts[card.id] -= 1
            serial_set.add(card.serial)
    if isinstance(card, Pokemon):
        for c in card.energyCards: add_card_count(c, my_index)
        for c in card.tools:       add_card_count(c, my_index)
        for c in card.preEvolution: add_card_count(c, my_index)


def set_card_counts(obs, my_index):
    card_counts.clear()
    serial_set.clear()
    for pid in DECK:
        card_counts[pid] += 1
    state = obs.current
    my_state = state.players[my_index]
    for card in my_state.hand:    add_card_count(card, my_index)
    for card in my_state.discard: add_card_count(card, my_index)
    for card in my_state.bench:   add_card_count(card, my_index)
    for card in my_state.active:  add_card_count(card, my_index)
    for card in state.stadium:    add_card_count(card, my_index)
    if state.looking is not None:
        for card in state.looking: add_card_count(card, my_index)
    add_card_count(obs.select.effect, my_index)


def get_card(obs, area, index, player_index):
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:    return obs.select.deck[index]
        case AreaType.HAND:    return ps.hand[index]
        case AreaType.DISCARD: return ps.discard[index]
        case AreaType.ACTIVE:  return ps.active[index]
        case AreaType.BENCH:   return ps.bench[index]
        case AreaType.PRIZE:   return ps.prize[index]
        case AreaType.STADIUM: return obs.current.stadium[index]
        case AreaType.LOOKING: return obs.current.looking[index]
        case _:                return None


def main_option_proc(obs, damage):
    state = obs.current
    select = obs.select
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]

    global can_switch, can_attack, can_main_attack, can_energy_attach
    can_switch = False
    can_attack = False
    can_main_attack = False
    can_energy_attach = False
    for o in select.option:
        if o.type == OptionType.RETREAT:     can_switch = True
        elif o.type == OptionType.ATTACK:
            can_attack = True
            if o.attackId == 154: can_main_attack = True  # Phantom Dive

    plan_a.attack = -1
    plan_b.attack = -1
    if not can_main_attack and not (bench_attacker and can_switch):
        return

    cards = [op_state.active[0]] + list(op_state.bench)
    counter_indices = []
    ci = [0]
    remain_damage = 60
    while ci:
        index = ci[-1]
        hp = cards[index].hp
        if remain_damage >= hp:
            counter_indices.append(ci.copy())
            if index < len(cards) - 1:
                remain_damage -= hp
                ci.append(index + 1)
                continue
        if index == len(cards) - 1:
            ci.pop()
            if ci: remain_damage += cards[ci[-1]].hp
        if ci: ci[-1] += 1
    counter_indices.append([])

    remain_prize = len(my_state.prize)
    plan_score = 0
    for i, pokemon in enumerate(cards):
        base_prize_count = 0
        base_score = pokemon_score(pokemon, True)
        active_damage = 0 if no_damage_dex(pokemon.id) else damage
        if pokemon.hp <= active_damage:
            base_prize_count += prize_count(pokemon, True)
        else:
            base_score *= active_damage / pokemon.hp
        ci = []
        max_score = base_score
        if remain_prize <= base_prize_count:
            max_score = 50000
        else:
            for indices in counter_indices:
                if i in indices: continue
                p = base_prize_count
                s = base_score
                for idx in indices:
                    p += prize_count(cards[idx], False)
                    s += pokemon_score(cards[idx], False)
                if remain_prize <= p:
                    s = 50000
                else:
                    if p >= 2:
                        if remain_prize <= 4: s -= 1200
                    elif p == 1:
                        s -= 300
                    else:
                        s += 1200
                if max_score < s:
                    max_score = s
                    ci = indices
        if plan_score < max_score:
            plan_score = max_score
            plan_a.attack = i
            plan_a.counter = ci
        if i == 0:
            plan_b.attack = plan_a.attack
            plan_b.counter = plan_a.counter


def agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        return DECK

    global pre_turn_log, current_turn_log

    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]

    if state.turn == 0:
        prize.clear()
        pre_turn_log.clear()
        current_turn_log.clear()
    else:
        for log in obs.logs:
            current_turn_log.append(log)
            if log.type == LogType.TURN_END:
                pre_turn_log[:] = current_turn_log
                current_turn_log.clear()

    pre_ko = False
    no_item = False
    for log in pre_turn_log:
        if log.type == LogType.ATTACK:
            if log.attackId == 323: no_item = True  # Itchy Pollen
        elif log.type == LogType.MOVE_CARD:
            if (log.playerIndex == my_index
                    and log.fromArea in (AreaType.BENCH, AreaType.ACTIVE)
                    and log.toArea == AreaType.DISCARD):
                pre_ko = True

    if select.deck is not None:
        set_card_counts(obs, my_index)
        for card in select.deck:
            card_counts[card.id] -= 1
        prize.clear()
        for pid in card_counts:
            for _ in range(card_counts[pid]):
                prize.append(pid)

    set_card_counts(obs, my_index)
    for pid in prize:
        card_counts[pid] -= 1
    deck_counts = card_counts

    prize_diff = len(my_state.prize) - len(op_state.prize)

    global bench_attacker
    field_counts = defaultdict(int)
    hand_counts = defaultdict(int)
    discard_counts = defaultdict(int)

    active_id = 0
    bench_attacker = False
    can_evolve_dreepy = False
    evolve_dreepy_count = 0
    can_evolve_drakloak = False
    damage = 200
    for card in my_state.active:
        if card is None: continue
        active_id = card.id
        field_counts[card.id] += 1
        if not card.appearThisTurn:
            if card.id == Dreepy:
                can_evolve_dreepy = True
                evolve_dreepy_count += 1
            elif card.id == Drakloak:
                can_evolve_drakloak = True
    for card in my_state.bench:
        field_counts[card.id] += 1
        if not card.appearThisTurn:
            if card.id == Dreepy:
                can_evolve_dreepy = True
                evolve_dreepy_count += 1
            elif card.id == Drakloak:
                can_evolve_drakloak = True
        if card.id == Dragapult_ex and len(card.energies) >= 2:
            bench_attacker = True
    main_pokemon_count = field_counts[Dreepy] + field_counts[Drakloak] + field_counts[Dragapult_ex]
    no_more_dex = (field_counts[Dragapult_ex] * 2 >= len(op_state.prize))

    stadium_id = 0
    for card in state.stadium:
        stadium_id = card.id

    support_count = 0
    for card in my_state.discard:
        discard_counts[card.id] += 1

    def attach_score(attach_id, pokemon, active):
        energy_count = len(pokemon.energies)
        data = card_table.get(attach_id)
        if data and data.cardType == CardType.TOOL:
            score = 60000
            if active: score += 1000
            return score
        if pokemon.id == Budew: return -1
        elif pokemon.id in (Meowth_ex, Fezandipiti_ex, Latias_ex):
            if active and not can_switch and not my_state.asleep and not my_state.paralyzed:
                return 22000 if bench_attacker or field_counts[Budew] >= 1 else 18000
            else: return -1
        if active and can_main_attack: return -1
        score = 20000
        if energy_count >= 2:
            if active and not can_switch and not my_state.asleep and not my_state.paralyzed:
                score += 200
            else: return -1
        elif energy_count == 1:
            if attach_id == pokemon.energyCards[0].id: return -1
            if pokemon.id == Dragapult_ex:   score += 250
            elif pokemon.id == Dreepy:        score -= 150
            else:                             score -= 200
            if active: score += 200
        else:
            if active:
                if bench_attacker: score += 400
            else:
                if pokemon.id == Dragapult_ex:   score += 150
                elif pokemon.id == Dreepy:        score += 100
                else:                             score += 50
                if bench_attacker: score -= 200
        if no_more_dex and pokemon.id in (Dreepy, Drakloak):
            score -= 500
        return score

    def hand_score(pid, ignore_count):
        score = 0
        if pid == Dreepy:
            score = 1000 if main_pokemon_count >= 3 else 18000
        elif pid == Drakloak:
            score = 20000 if can_evolve_dreepy else 3000
        elif pid == Dragapult_ex:
            if no_more_dex: return UNNECESSARY
            elif can_evolve_dreepy and hand_counts[Rare_Candy] >= 1 and not no_item:
                score = 40000
            elif can_evolve_drakloak:
                if field_counts[pid] == 0:   score = 30000
                elif field_counts[pid] == 1: score = 10000
                else:                        score = 50
            else:
                score = 50 if field_counts[pid] >= 2 else 2000
        elif pid == Fezandipiti_ex:
            if pre_ko:             score = 50000
            elif prize_diff <= -2: score = 5
            elif len(op_state.prize) == 1: return UNNECESSARY
        elif pid == Latias_ex:
            if active_id in (Fezandipiti_ex, Meowth_ex, Dreepy):
                score = 28000 if field_counts[Drakloak] + field_counts[Dragapult_ex] == 0 else 15000
            else:
                score = 10
        elif pid == Budew:
            if field_counts[pid] + field_counts[Drakloak] + field_counts[Dragapult_ex] >= 1:
                return UNNECESSARY
            score = 30000 if state.turn >= 2 else 0
        elif pid == Meowth_ex:
            if support_count > hand_counts[Boss_Orders] or stadium_id == Team_Rocket_Watchtower:
                score = 5
            elif state.supporterPlayed: score = 40
            else:                       score = 35000
        elif pid == Rare_Candy:
            if no_more_dex: return UNNECESSARY
            elif can_evolve_dreepy and hand_counts[Dragapult_ex] >= 1: score = 40000
        elif pid == Unfair_Stamp:
            if pre_ko:                      score = 80000
            elif len(op_state.prize) == 1:  return UNNECESSARY
            else:                           score = 80
        elif pid == Buddy_Buddy_Poffin:
            count = deck_counts[Dreepy]
            if count == 0: return UNNECESSARY
            if state.turn <= 2 and field_counts[Budew] == 0 and deck_counts[Budew] >= 1:
                count += 1
            score = 35000 if count >= 2 else 0
        elif pid == Night_Stretcher:
            for i in discard_counts:
                if discard_counts[i] >= 1:
                    ct = card_table.get(i)
                    if ct and ct.cardType in (CardType.POKEMON, CardType.BASIC_ENERGY):
                        score = max(score, hand_score(i, ignore_count))
        elif pid == Crushing_Hammer:  score = 20
        elif pid == Ultra_Ball:
            score = 70 if main_pokemon_count <= 2 or field_counts[Dreepy] >= 1 else 5
        elif pid == Poke_Pad:
            score = max(hand_score(Dreepy, ignore_count), hand_score(Drakloak, ignore_count))
        elif pid == Lucky_Helmet: score = 15
        elif pid == Boss_Orders:
            if plan_a.attack > 0: score = 60000
        elif pid == Crispin:
            if not ignore_count or support_count == 0:
                if deck_counts[Basic_Fire_Energy] == 0 or deck_counts[Basic_Psychic_Energy] == 0:
                    score = 10
                if not can_main_attack and not bench_attacker and field_counts[Dragapult_ex] >= 1:
                    score = 55000
                else: score = 25000
        elif pid == Brock_Scouting:
            if not ignore_count or support_count == 0:
                if state.turn == 2 and field_counts[Budew] + field_counts[Latias_ex] == 0:
                    score = 50000
                else: score = 30000
        elif pid == Lillie_Determination:
            if not ignore_count or support_count == 0: score = 45000
        elif pid == Team_Rocket_Watchtower:
            if stadium_id != 0 and stadium_id != Team_Rocket_Watchtower: score = 4000
        elif pid in (Basic_Fire_Energy, Basic_Psychic_Energy):
            if can_main_attack and (len(op_state.prize) <= 2
                    or (bench_attacker and len(op_state.prize) <= 4)):
                return UNNECESSARY
            else:
                max_sc = -10000
                for pokemon in my_state.active:
                    if pokemon is None: continue
                    max_sc = max(max_sc, attach_score(pid, pokemon, True))
                for pokemon in my_state.bench:
                    max_sc = max(max_sc, attach_score(pid, pokemon, False))
                score = max_sc - 5000
                if can_main_attack or bench_attacker: score //= 10

        if not ignore_count and hand_counts[pid] > 0:
            if pid == Drakloak and hand_counts[pid] < evolve_dreepy_count: score -= 10
            elif pid == Dreepy: score -= 100
            else:               score -= 100000
        return score

    global use_support
    if context == SelectContext.MAIN:
        main_option_proc(obs, damage)
        use_support = 0
        if not state.supporterPlayed:
            support_score = 0
            for o in select.option:
                if o.type == OptionType.PLAY:
                    card = get_card(obs, AreaType.HAND, o.index, state.yourIndex)
                    ct = card_table.get(card.id)
                    if ct and ct.cardType == CardType.SUPPORTER:
                        s = hand_score(card.id, True)
                        if support_score < s:
                            support_score = s
                            use_support = card.id

    hand_scores = []
    negative_hand_count = 0
    for card in my_state.hand:
        s = hand_score(card.id, False)
        hand_scores.append(s)
        if s < 0: negative_hand_count += 1
        hand_counts[card.id] += 1
        ct = card_table.get(card.id)
        if ct and ct.cardType == CardType.SUPPORTER and card.id != Boss_Orders:
            support_count += 1

    no_draw = (my_state.deckCount <= 8)
    do_switch = (not can_main_attack and (bench_attacker
        or (active_id != Budew and field_counts[Budew] >= 1 and state.turn >= 2)))
    effect_card_id = 0 if select.effect is None else select.effect.id
    context_card_id = 0 if select.contextCard is None else select.contextCard.id

    scores = []
    for o in select.option:
        score = 0
        if o.type == OptionType.NUMBER:
            score = o.number
        elif o.type == OptionType.YES:
            score = -1 if context == SelectContext.IS_FIRST else 1
        elif o.type == OptionType.CARD:
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card is not None:
                energy_count = 0
                hp = 0
                if isinstance(card, Pokemon):
                    energy_count = len(card.energies)
                    hp = card.hp
                if context in (SelectContext.SWITCH, SelectContext.TO_ACTIVE, SelectContext.SETUP_ACTIVE_POKEMON):
                    if o.playerIndex == my_index:
                        if card.id == Dreepy:          score += 10000
                        elif card.id == Drakloak:
                            score += 20000 if energy_count >= 1 else -10000
                        elif card.id == Dragapult_ex:  score += 50000
                        elif card.id == Budew:
                            score += 100000 if context != SelectContext.SWITCH else (30000 if not bench_attacker else 0)
                        elif card.id == Fezandipiti_ex: score -= 1000
                        elif card.id == Meowth_ex:      score -= 2000
                    else:
                        if plan_a.attack == o.index + 1: score += 100000
                    score += energy_count * 1000 + hp
                elif context == SelectContext.SETUP_BENCH_POKEMON:
                    score = -1 if my_index == state.firstPlayer or card.id != Dreepy else 0
                elif context in (SelectContext.TO_BENCH, SelectContext.TO_HAND):
                    score = hand_score(card.id, False)
                    hand_counts[card.id] += 1
                    if effect_card_id == Crispin:
                        score = 100000 - hand_score(card.id, True)
                elif context == SelectContext.DISCARD:
                    hand_counts[card.id] -= 1
                    ct = card_table.get(card.id)
                    if ct and ct.cardType == CardType.SUPPORTER: support_count -= 1
                    score = -hand_score(card.id, False)
                elif context in (SelectContext.DAMAGE_COUNTER, SelectContext.DAMAGE_COUNTER_ANY):
                    if hp > 0:
                        score = 100000 - 10 * hp + pokemon_score(card, False)
                        if context == SelectContext.DAMAGE_COUNTER:
                            if 210 <= hp <= 230:
                                score += 20000 + hp * 20
                                if o.area == AreaType.ACTIVE: score += 10000
                            elif 40 <= hp <= 90:  score += 10000 + hp * 20
                            elif hp <= 30:        score += -10000 + hp * 20
                            if card.id in (133, 351): score += 30000
                        else:
                            index = o.index + 1
                            if index in plan_b.counter: score += 100000
                            else:
                                remain_damage = select.remainDamageCounter * 10
                                if 210 <= hp <= 200 + remain_damage: score += 30000
                                elif 20 <= hp <= 60 + remain_damage: score += 10000
                                elif hp == 10:                       score -= 100000
                            if no_damage_counter(card): score = -1
                elif context == SelectContext.ATTACH_FROM:
                    score = attach_score(context_card_id, card, o.area == AreaType.ACTIVE)
                    if card.id == Dragapult_ex: score += 200
        elif o.type in (OptionType.ENERGY_CARD, OptionType.ENERGY):
            if o.playerIndex != state.yourIndex:
                score = 20 if o.area == AreaType.BENCH else 10
                card = get_card(obs, o.area, o.index, o.playerIndex)
                ct = card_table.get(card.id)
                if ct and ct.cardType == CardType.SPECIAL_ENERGY: score += 1
        elif o.type == OptionType.PLAY:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            card_score = hand_scores[o.index]
            if card.id == Dreepy:           score = 51000
            elif card.id == Fezandipiti_ex: score = 53000 if card_score > 0 else -1
            elif card.id == Latias_ex:
                score = 51000 if active_id not in (Drakloak, Dragapult_ex) else -1
            elif card.id == Budew:
                score = 52000 if field_counts[Budew] == 0 and field_counts[Dragapult_ex] == 0 else -1
            elif card.id == Meowth_ex:
                if state.supporterPlayed or stadium_id == Team_Rocket_Watchtower: score = -1
                elif support_count == 0:                                            score = 50000
                elif support_count == hand_counts[Boss_Orders] and plan_a.attack > 0: score = 50000
                else:                                                                   score = -1
            elif card.id == Rare_Candy:
                score = -1 if no_more_dex else 75000
            elif card.id == Unfair_Stamp:   score = 15000
            elif card.id == Night_Stretcher:
                score = 42000 if card_score >= 18000 else -1
            elif card.id == Crushing_Hammer: score = 40000
            elif card.id == Boss_Orders:
                score = 35000 if card.id == use_support else -1
            elif card.id == Lillie_Determination:
                score = 14000 if card.id == use_support else -1
            elif card.id == Team_Rocket_Watchtower:
                score = 80000 if stadium_id > 0 or state.turn == 1 else -1
            elif no_draw:
                score = -1
            elif card.id == Buddy_Buddy_Poffin:
                score = 46000 if deck_counts[Dreepy] > 0 else -1
            elif card.id == Ultra_Ball:
                score = 44000 if negative_hand_count >= 2 else -1
            elif card.id == Poke_Pad:
                score = 45000 if deck_counts[Dreepy] + deck_counts[Drakloak] > 0 else -1
            elif card.id in (Crispin, Brock_Scouting):
                score = 35000 if card.id == use_support else -1
        elif o.type == OptionType.ATTACH:
            card = get_card(obs, o.area, o.index, my_index)
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = attach_score(card.id, pokemon, o.inPlayArea == AreaType.ACTIVE)
        elif o.type == OptionType.EVOLVE:
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score += len(pokemon.energies)
            if pokemon.id == Dreepy:
                score += 30000
            elif field_counts[Dragapult_ex] >= 2 or (field_counts[Dragapult_ex] == 1 and len(op_state.prize) <= 2):
                score = -1
            else:
                score += 70000
        elif o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if no_draw:       score = -1
            elif card.id == 1267: score = 1  # Lumiose City
            else:             score = 40000
        elif o.type == OptionType.RETREAT:
            score = 10000 if do_switch else -1
        elif o.type == OptionType.ATTACK:
            score = o.attackId

        scores.append(score)

    output = []
    if scores:
        sorted_scores = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        for i in range(select.maxCount):
            if (sorted_scores[i][1] >= 0
                    or select.minCount > i
                    or context not in (SelectContext.TO_BENCH, SelectContext.SETUP_BENCH_POKEMON)):
                output.append(sorted_scores[i][0])
    return output


In [ ]:
%%writefile opponents/abomasnow_agent.py
"""
Mega Abomasnow ex training opponent — official Kaggle sample agent.
Source: kiyotah/a-sample-rule-based-agent-mega-abomasnow-ex-deck

Archetype: Energy-heavy discard mill. Hammer-lanche: discard top 6 cards,
100 per Basic Water Energy discarded. Backup attacker: Kyogre Riptide (20x Water in discard).
Mega Abomasnow ex: Stage 1 megaEx (confirm HP from all_card_data).
"""
import sys, glob
from collections import defaultdict

for _p in ['/kaggle/input/**/cg-lib', '/kaggle/input/cg-lib']:
    _m = glob.glob(_p, recursive=True)
    if _m: sys.path.insert(0, _m[0]); break

try:
    from cg.api import AreaType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class
    all_card = all_card_data()
    card_table = {c.cardId: c for c in all_card}
except Exception:
    card_table = {}
    class _Stub:
        def __getattr__(self, n): return None
    AreaType = SelectContext = OptionType = _Stub()
    to_observation_class = lambda x: x

# ── Deck (60 cards, embedded) ─────────────────────────────────────────────────
Kyogre               = 721   # ×2
Snover               = 722   # ×4
Mega_Abomasnow_ex    = 723   # ×4
Ultra_Ball           = 1121  # ×4
Precious_Trolley     = 1126  # ×1
Carmine              = 1192  # ×4
Lillie_Determination = 1227  # ×4
Surfing_Beach        = 1262  # ×3
Basic_Water_Energy   = 3     # ×34

DECK = (
    [Kyogre] * 2 + [Snover] * 4 + [Mega_Abomasnow_ex] * 4 +
    [Ultra_Ball] * 4 + [Precious_Trolley] +
    [Carmine] * 4 + [Lillie_Determination] * 4 +
    [Surfing_Beach] * 3 + [Basic_Water_Energy] * 34
)
assert len(DECK) == 60, f"Abomasnow deck has {len(DECK)} cards, expected 60"


def get_card(obs, area, index, player_index):
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:    return obs.select.deck[index]
        case AreaType.HAND:    return ps.hand[index]
        case AreaType.DISCARD: return ps.discard[index]
        case AreaType.ACTIVE:  return ps.active[index]
        case AreaType.BENCH:   return ps.bench[index]
        case AreaType.PRIZE:   return ps.prize[index]
        case AreaType.STADIUM: return obs.current.stadium[index]
        case AreaType.LOOKING: return obs.current.looking[index]
        case _:                return None


def agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        return DECK

    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]

    field_counts = defaultdict(int)
    hand_counts = defaultdict(int)
    discard_counts = defaultdict(int)

    bench_attacker_index0 = -1  # Mega Abomasnow ex
    bench_attacker_index1 = -1  # Kyogre
    for i, card in enumerate(my_state.bench):
        field_counts[card.id] += 1
        if card.id == Mega_Abomasnow_ex and len(card.energies) >= 2:
            bench_attacker_index0 = i
        elif card.id == Kyogre and len(card.energies) >= 1:
            bench_attacker_index1 = i

    for card in my_state.hand:
        hand_counts[card.id] += 1

    for card in my_state.discard:
        discard_counts[card.id] += 1

    op_active_hp = 0
    for card in state.players[1 - my_index].active:
        if card is None: continue
        op_active_hp = card.hp

    prefer_ky = op_active_hp <= 20 * discard_counts[Basic_Water_Energy]
    switch_index = -1
    for card in my_state.active:
        if card is None: continue
        field_counts[card.id] += 1
        if card.id == Mega_Abomasnow_ex and len(card.energies) >= 2:
            if prefer_ky and bench_attacker_index1 >= 0:
                switch_index = bench_attacker_index1
        elif card.id == Kyogre and len(card.energies) >= 1:
            if not prefer_ky and bench_attacker_index0 >= 0:
                switch_index = bench_attacker_index0
        elif bench_attacker_index0 >= 0:
            switch_index = bench_attacker_index0

    scores = []
    for o in select.option:
        score = 0
        if o.type == OptionType.NUMBER:
            score = o.number
        elif o.type == OptionType.YES:
            score = 1
        elif o.type == OptionType.CARD:
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card is not None:
                energy_count = 0
                if isinstance(card, Pokemon): energy_count = len(card.energies)
                if context in (SelectContext.SWITCH, SelectContext.TO_ACTIVE, SelectContext.SETUP_ACTIVE_POKEMON):
                    score += energy_count * 2
                    if o.index == switch_index: score += 100
                    if card.id == Mega_Abomasnow_ex: score += 20
                    elif card.id == Kyogre:           score += 10
                elif context in (SelectContext.TO_BENCH, SelectContext.TO_HAND):
                    if card.id == Snover:
                        if field_counts[card.id] >= 1:               score += 5
                        elif field_counts[Mega_Abomasnow_ex] >= 1:   score += 15
                        else:                                          score += 30
                    elif card.id == Mega_Abomasnow_ex:
                        if field_counts[Snover] >= 1 and field_counts[card.id] + hand_counts[card.id] == 0:
                            score += 100
                        else: score += 10
                    elif card.id == Kyogre:
                        score += 1 if field_counts[card.id] >= 1 else 20
                elif context == SelectContext.DISCARD:
                    if card.id == Basic_Water_Energy:   score += 100
                    elif card.id == Mega_Abomasnow_ex:  score += 10
                    elif card.id == Carmine:
                        if hand_counts[Lillie_Determination] >= 1: score += 30
                    elif card.id == Lillie_Determination: score -= 20
                    if hand_counts[card.id] >= 2: score += 500
                    hand_counts[card.id] -= 1
        elif o.type == OptionType.PLAY:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            score = 10000
            if card.id == Ultra_Ball:
                if (hand_counts[Basic_Water_Energy] >= 3
                        or (my_state.handCount >= 4
                            and (field_counts[Mega_Abomasnow_ex] + hand_counts[Mega_Abomasnow_ex] == 0
                                 or field_counts[Mega_Abomasnow_ex] + field_counts[Snover] == 0
                                 or field_counts[Kyogre] == 0))):
                    score = 4000
                else:
                    score = -1
            elif card.id == Carmine:
                if field_counts[Snover] >= 1 and hand_counts[Mega_Abomasnow_ex] >= 1: score = -1
                else: score = 3000
            elif card.id == Lillie_Determination:
                if field_counts[Snover] >= 1 and field_counts[Mega_Abomasnow_ex] == 0 and hand_counts[Mega_Abomasnow_ex] >= 1:
                    score = -1
                else: score = 3100
        elif o.type == OptionType.ATTACH:
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = 5000
            energy_count = len(pokemon.energies)
            if energy_count == 0 and o.inPlayArea == AreaType.BENCH:
                score += 1
            if pokemon.id == Snover:
                score += 1
                if energy_count == 1:    score -= 100
                elif energy_count >= 2:  score -= 400
                if bench_attacker_index0 >= 0: score -= 300
            elif pokemon.id == Mega_Abomasnow_ex:
                score += 10
                if energy_count == 1:   score += 30
                elif energy_count >= 2: score -= 300
                if bench_attacker_index0 >= 0: score -= 200
            elif pokemon.id == Kyogre:
                score += 5
                if len(pokemon.energies) >= 1: score -= 200
                if bench_attacker_index1 >= 0: score -= 200
            if o.inPlayArea == AreaType.ACTIVE:
                if bench_attacker_index0 >= 0 and bench_attacker_index1 >= 0 and energy_count <= 2:
                    score += 200
        elif o.type == OptionType.EVOLVE:
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = 10000 + len(pokemon.energies)
        elif o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card.id == Surfing_Beach and switch_index >= 0: score = 2000
            else:                                               score = -1
        elif o.type == OptionType.RETREAT:
            score = 1500 if switch_index >= 0 else -1
        elif o.type == OptionType.ATTACK:
            score = 1000
            if o.attackId == 1042:  # Riptide
                score += discard_counts[Basic_Water_Energy] * 20 - 90
            elif o.attackId == 1046:  # Hammer-lanche
                score += -100 if op_active_hp <= 200 else 100

        scores.append(score)

    desc_indices = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]
    return desc_indices[:select.maxCount]


In [ ]:
%%writefile opponents/starmie_agent.py
"""
Mega Starmie ex training opponent for self-play pool.
Archetype: Water spread + bench snipe. Stage 1 megaEx (330 HP, 3 prizes).
  Jetting Blow {W}: 120 to Active + 50 to one Benched Pokemon.
  Nebula Beam {C}{C}{C}: 210, ignores all effects on opponent's Active (bypasses Mist/Rocky Energy).

Card IDs confirmed from EN_Card_Data.csv (docs/):
  STARYU    = 1030  (Basic Water, 70 HP)
  STARMIE_EX = 1031 (Stage 1 megaEx, 330 HP, 3 prizes)

Strategy encoded here:
  - Bench Staryu ASAP, evolve to Mega Starmie ex
  - Always attack when possible (Nebula Beam vs protected targets is automatic
    via stype=6 highest-damage selection)
  - For Jetting Blow bench-snipe follow-up: pick lowest-HP opponent bench
  - Prioritize energy on Mega Starmie ex (megaEx target)
  - Never retreat — spread is cumulative, staying up compounds damage
"""
import sys, glob

for _p in ['/kaggle/input/**/cg-lib', '/kaggle/input/cg-lib']:
    _m = glob.glob(_p, recursive=True)
    if _m: sys.path.insert(0, _m[0]); break

try:
    from cg.api import all_attack
    _ATK = {a.attackId: getattr(a, 'damage', 0) or 0 for a in all_attack()}
except Exception:
    _ATK = {}

NUMBER, YES, NO, CARD, TOOL_CARD, ENERGY_CARD, ENERGY, PLAY, ATTACH, EVOLVE, \
    ABILITY, DISCARD, RETREAT, ATTACK, END, SKILL, SPECIAL_CONDITION = range(17)
CTX_SETUP_ACTIVE = 1
CTX_SETUP_BENCH  = 2

# ── Card IDs ─────────────────────────────────────────────────────────────────
STARYU     = 1030  # Basic Water, 70 HP
STARMIE_EX = 1031  # Stage 1 megaEx, 330 HP, 3 prizes

# Trainer IDs (confirmed from EN_Card_Data.csv)
_ULTRA_BALL       = 1121  # ×4  — search any Pokemon
_BUDDY_POFFIN     = 1086  # ×4  — bench 2 Basics ≤70 HP (hits Staryu)
_BOSS_ORDERS      = 1182  # ×2  — gust opponent bench
_CARMINE          = 1192  # ×4  — draw supporter
_LILLIE_DET       = 1227  # ×4  — draw supporter
_SURFING_BEACH    = 1262  # ×3  — Water stadium: free switch once/turn
_BROCK_SCOUTING   = 1210  # ×2  — search 2 Basics or 1 Evolution
_NIGHT_STRETCHER  = 1097  # ×2  — recover Pokemon or Basic Energy from discard
_BASIC_WATER      = 3     # ×27 — Water Energy

DECK = (
    [STARYU] * 4 + [STARMIE_EX] * 4 +
    [_ULTRA_BALL] * 4 + [_BUDDY_POFFIN] * 4 + [_BOSS_ORDERS] * 2 +
    [_CARMINE] * 4 + [_LILLIE_DET] * 4 +
    [_SURFING_BEACH] * 3 + [_BROCK_SCOUTING] * 2 + [_NIGHT_STRETCHER] * 2 +
    [_BASIC_WATER] * 27
)
assert len(DECK) == 60, f"Starmie deck has {len(DECK)} cards, expected 60"

# ── Helpers ──────────────────────────────────────────────────────────────────
def _pk_id(pk):  return (pk or {}).get('id', -1)
def _active(p):  a = p.get('active'); return a[0] if a and a[0] else None
def _bench(p):   return p.get('bench') or []
def _hand(p):    return p.get('hand') or []

def _prize_val(pk):
    if not pk: return 0
    if pk.get('megaEx'): return 3
    if pk.get('ex'):     return 2
    return 1

def _clamp(idxs, sel):
    mn = sel.get('minCount', 0) or 0
    mx = sel.get('maxCount', 1) or 1
    n  = len(sel.get('option', []))
    out = []
    for i in idxs:
        if 0 <= i < n and i not in out: out.append(i)
        if len(out) >= mx: break
    j = 0
    while len(out) < mn and j < n:
        if j not in out: out.append(j)
        j += 1
    return out

def _players(obs):
    cur = obs.get('current') or {}
    me  = cur.get('yourIndex', 0)
    pl  = cur.get('players', [])
    my  = pl[me]   if len(pl) > me  else {}
    opp = pl[1-me] if len(pl) == 2  else {}
    return cur, me, my, opp

# ── Bench snipe target: pick lowest remaining HP on opponent bench ────────────
def _pick_snipe_target(obs, sel):
    _, me, _, opp = _players(obs)
    opp_bench = _bench(opp)
    opts = sel.get('option', [])
    best_i, best_hp = 0, 99999
    for i, o in enumerate(opts):
        bi  = o.get('index', 0)
        pk  = opp_bench[bi] if 0 <= bi < len(opp_bench) else None
        hp  = (pk or {}).get('hp', 99999) or 99999
        pv  = _prize_val(pk)
        # prefer KO-able targets; tiebreak on lowest HP
        val = (hp, -pv)
        if hp < best_hp or (hp == best_hp and pv > _prize_val(opp_bench[opts[best_i].get('index', 0)] if opts else None)):
            best_hp, best_i = hp, i
    return [best_i]

# ── Main phase ────────────────────────────────────────────────────────────────
def _main_phase(obs, sel):
    opts = sel['option']
    n    = len(opts)
    _, me, my, opp = _players(obs)
    my_active  = _active(my)
    my_bench   = _bench(my)
    hand       = _hand(my)

    def score(o):
        ot = o.get('type')
        if ot == ATTACK:   return 100.0
        if ot == EVOLVE:   return 20.0
        if ot == ATTACH:
            ta  = o.get('inPlayArea', -1)
            ti  = o.get('inPlayIndex', 0)
            tgt = my_active if ta == 4 else (my_bench[ti] if ta == 5 and 0 <= ti < len(my_bench) else None)
            return 10.0 + _prize_val(tgt) * 5.0   # route to Starmie ex first
        if ot == PLAY:     return 5.0
        if ot == RETREAT:  return -10.0   # spread is cumulative — stay up
        if ot == END:      return 0.5
        return 1.0

    return [max(range(n), key=lambda i: score(opts[i]))]

# ── Agent entry point ─────────────────────────────────────────────────────────
def agent(obs_dict: dict) -> list:
    try:
        sel  = obs_dict.get('select')
        if sel is None: return DECK
        opts = sel.get('option', [])
        n    = len(opts)
        if n == 0: return []
        stype = sel.get('type')
        ctx   = sel.get('context', 0)
        mn    = sel.get('minCount', 0) or 0
        mx    = sel.get('maxCount', 1) or 1

        if stype == 0: return _main_phase(obs_dict, sel)
        if stype == 1:
            if ctx == CTX_SETUP_ACTIVE:
                # Prefer Staryu as starting active
                for i, o in enumerate(opts):
                    if o.get('cardId', o.get('id', -1)) == STARYU: return [i]
                return [0]
            if ctx == CTX_SETUP_BENCH: return list(range(n))
            # Check for bench-snipe target selection (opponent bench picks)
            cur = obs_dict.get('current') or {}
            me  = cur.get('yourIndex', 0)
            if all(o.get('playerIndex') == (1-me) and o.get('area') == 5 for o in opts):
                return _pick_snipe_target(obs_dict, sel)
            yes_i = [i for i, o in enumerate(opts) if o.get('type') == YES]
            return yes_i if yes_i else [0]
        if stype in (2, 3, 4, 7): return _clamp(list(range(n)), sel)
        if stype == 5:             return [0]
        if stype == 6:
            return [max(range(n), key=lambda i: _ATK.get(opts[i].get('attackId'), 0))]
        if stype == 9:
            for i, o in enumerate(opts):
                if o.get('type') == YES: return [i]
            return [0]
        k = mn if mn > 0 else (1 if mx >= 1 else 0)
        return _clamp(list(range(n))[:k] if k else [], sel)
    except Exception:
        sel = obs_dict.get('select')
        if not sel: return DECK
        n = len(sel.get('option', [])); mn = sel.get('minCount', 0) or 0
        return list(range(min(max(mn, 1), n))) if n else []


In [ ]:
with open('training/archetype_decks.json', 'w', encoding='utf-8') as f:
    f.write('{\n  "alakazam": [\n    {\n      "cardId": 743,\n      "name": "Alakazam",\n      "copies": 4,\n      "games_seen": 71,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 741,\n      "name": "Abra",\n      "copies": 4,\n      "games_seen": 70,\n      "game_frac": 0.986\n    },\n    {\n      "cardId": 1152,\n      "name": "Pok\\u00e9 Pad",\n      "copies": 4,\n      "games_seen": 68,\n      "game_frac": 0.958\n    },\n    {\n      "cardId": 742,\n      "name": "Kadabra",\n      "copies": 4,\n      "games_seen": 68,\n      "game_frac": 0.958\n    },\n    {\n      "cardId": 1086,\n      "name": "Buddy-Buddy Poffin",\n      "copies": 4,\n      "games_seen": 67,\n      "game_frac": 0.944\n    },\n    {\n      "cardId": 66,\n      "name": "Dudunsparce",\n      "copies": 4,\n      "games_seen": 65,\n      "game_frac": 0.915\n    },\n    {\n      "cardId": 1231,\n      "name": "Dawn",\n      "copies": 4,\n      "games_seen": 65,\n      "game_frac": 0.915\n    },\n    {\n      "cardId": 1225,\n      "name": "Hilda",\n      "copies": 4,\n      "games_seen": 60,\n      "game_frac": 0.845\n    },\n    {\n      "cardId": 19,\n      "name": "Telepath Psychic Energy",\n      "copies": 4,\n      "games_seen": 60,\n      "game_frac": 0.845\n    },\n    {\n      "cardId": 305,\n      "name": "Dunsparce",\n      "copies": 4,\n      "games_seen": 59,\n      "game_frac": 0.831\n    },\n    {\n      "cardId": 1079,\n      "name": "Rare Candy",\n      "copies": 4,\n      "games_seen": 52,\n      "game_frac": 0.732\n    },\n    {\n      "cardId": 5,\n      "name": "Basic {P} Energy",\n      "copies": 4,\n      "games_seen": 47,\n      "game_frac": 0.662\n    },\n    {\n      "cardId": 1129,\n      "name": "Sacred Ash",\n      "copies": 4,\n      "games_seen": 43,\n      "game_frac": 0.606\n    },\n    {\n      "cardId": 1182,\n      "name": "Boss\\u2019s Orders",\n      "copies": 3,\n      "games_seen": 38,\n      "game_frac": 0.535\n    },\n    {\n      "cardId": 1097,\n      "name": "Night Stretcher",\n      "copies": 3,\n      "games_seen": 34,\n      "game_frac": 0.479\n    },\n    {\n      "cardId": 1081,\n      "name": "Enhanced Hammer",\n      "copies": 2,\n      "games_seen": 29,\n      "game_frac": 0.408\n    }\n  ],\n  "archaludon": [\n    {\n      "cardId": 169,\n      "name": "Duraludon",\n      "copies": 4,\n      "games_seen": 27,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 190,\n      "name": "Archaludon ex",\n      "copies": 4,\n      "games_seen": 27,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 8,\n      "name": "Basic {M} Energy",\n      "copies": 4,\n      "games_seen": 26,\n      "game_frac": 0.963\n    },\n    {\n      "cardId": 1122,\n      "name": "Pok\\u00e9gear 3.0",\n      "copies": 4,\n      "games_seen": 25,\n      "game_frac": 0.926\n    },\n    {\n      "cardId": 1121,\n      "name": "Ultra Ball",\n      "copies": 4,\n      "games_seen": 24,\n      "game_frac": 0.889\n    },\n    {\n      "cardId": 57,\n      "name": "Relicanth",\n      "copies": 4,\n      "games_seen": 24,\n      "game_frac": 0.889\n    },\n    {\n      "cardId": 1097,\n      "name": "Night Stretcher",\n      "copies": 4,\n      "games_seen": 23,\n      "game_frac": 0.852\n    },\n    {\n      "cardId": 1227,\n      "name": "Lillie\'s Determination",\n      "copies": 4,\n      "games_seen": 23,\n      "game_frac": 0.852\n    },\n    {\n      "cardId": 1152,\n      "name": "Pok\\u00e9 Pad",\n      "copies": 4,\n      "games_seen": 23,\n      "game_frac": 0.852\n    },\n    {\n      "cardId": 1244,\n      "name": "Full Metal Lab",\n      "copies": 4,\n      "games_seen": 21,\n      "game_frac": 0.778\n    },\n    {\n      "cardId": 1182,\n      "name": "Boss\\u2019s Orders",\n      "copies": 4,\n      "games_seen": 19,\n      "game_frac": 0.704\n    },\n    {\n      "cardId": 1185,\n      "name": "Explorer\\u2019s Guidance",\n      "copies": 3,\n      "games_seen": 14,\n      "game_frac": 0.519\n    },\n    {\n      "cardId": 666,\n      "name": "Cinderace",\n      "copies": 3,\n      "games_seen": 13,\n      "game_frac": 0.481\n    },\n    {\n      "cardId": 1213,\n      "name": "Judge",\n      "copies": 3,\n      "games_seen": 10,\n      "game_frac": 0.37\n    },\n    {\n      "cardId": 1192,\n      "name": "Carmine",\n      "copies": 2,\n      "games_seen": 8,\n      "game_frac": 0.296\n    },\n    {\n      "cardId": 1147,\n      "name": "Jumbo Ice Cream",\n      "copies": 2,\n      "games_seen": 8,\n      "game_frac": 0.296\n    },\n    {\n      "cardId": 1123,\n      "name": "Switch",\n      "copies": 2,\n      "games_seen": 7,\n      "game_frac": 0.259\n    },\n    {\n      "cardId": 1159,\n      "name": "Hero\\u2019s Cape",\n      "copies": 1,\n      "games_seen": 4,\n      "game_frac": 0.148\n    }\n  ],\n  "bellibolt": [\n    {\n      "cardId": 4,\n      "name": "Basic {L} Energy",\n      "copies": 4,\n      "games_seen": 11,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 268,\n      "name": "Iono\\u2019s Tadbulb",\n      "copies": 4,\n      "games_seen": 11,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 269,\n      "name": "Iono\\u2019s Bellibolt ex",\n      "copies": 4,\n      "games_seen": 11,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 270,\n      "name": "Iono\\u2019s Wattrel",\n      "copies": 4,\n      "games_seen": 9,\n      "game_frac": 0.818\n    },\n    {\n      "cardId": 265,\n      "name": "Iono\\u2019s Voltorb",\n      "copies": 4,\n      "games_seen": 9,\n      "game_frac": 0.818\n    },\n    {\n      "cardId": 1233,\n      "name": "Canari",\n      "copies": 4,\n      "games_seen": 8,\n      "game_frac": 0.727\n    },\n    {\n      "cardId": 1121,\n      "name": "Ultra Ball",\n      "copies": 4,\n      "games_seen": 8,\n      "game_frac": 0.727\n    },\n    {\n      "cardId": 1097,\n      "name": "Night Stretcher",\n      "copies": 4,\n      "games_seen": 8,\n      "game_frac": 0.727\n    },\n    {\n      "cardId": 271,\n      "name": "Iono\\u2019s Kilowattrel",\n      "copies": 4,\n      "games_seen": 8,\n      "game_frac": 0.727\n    },\n    {\n      "cardId": 1152,\n      "name": "Pok\\u00e9 Pad",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 0.636\n    },\n    {\n      "cardId": 1254,\n      "name": "Levincia",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 0.636\n    },\n    {\n      "cardId": 1227,\n      "name": "Lillie\'s Determination",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 0.636\n    },\n    {\n      "cardId": 1086,\n      "name": "Buddy-Buddy Poffin",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 0.636\n    },\n    {\n      "cardId": 1110,\n      "name": "Max Rod",\n      "copies": 3,\n      "games_seen": 4,\n      "game_frac": 0.364\n    },\n    {\n      "cardId": 1118,\n      "name": "Energy Retrieval",\n      "copies": 2,\n      "games_seen": 3,\n      "game_frac": 0.273\n    },\n    {\n      "cardId": 1182,\n      "name": "Boss\\u2019s Orders",\n      "copies": 2,\n      "games_seen": 2,\n      "game_frac": 0.182\n    },\n    {\n      "cardId": 1102,\n      "name": "Dusk Ball",\n      "copies": 1,\n      "games_seen": 1,\n      "game_frac": 0.091\n    }\n  ],\n  "crustle": [\n    {\n      "cardId": 345,\n      "name": "Crustle",\n      "copies": 4,\n      "games_seen": 53,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 344,\n      "name": "Dwebble",\n      "copies": 4,\n      "games_seen": 52,\n      "game_frac": 0.981\n    },\n    {\n      "cardId": 1086,\n      "name": "Buddy-Buddy Poffin",\n      "copies": 4,\n      "games_seen": 38,\n      "game_frac": 0.717\n    },\n    {\n      "cardId": 1,\n      "name": "Basic {G} Energy",\n      "copies": 4,\n      "games_seen": 33,\n      "game_frac": 0.623\n    },\n    {\n      "cardId": 1227,\n      "name": "Lillie\'s Determination",\n      "copies": 3,\n      "games_seen": 21,\n      "game_frac": 0.396\n    },\n    {\n      "cardId": 1182,\n      "name": "Boss\\u2019s Orders",\n      "copies": 3,\n      "games_seen": 20,\n      "game_frac": 0.377\n    },\n    {\n      "cardId": 18,\n      "name": "Grow Grass Energy",\n      "copies": 3,\n      "games_seen": 20,\n      "game_frac": 0.377\n    },\n    {\n      "cardId": 1121,\n      "name": "Ultra Ball",\n      "copies": 2,\n      "games_seen": 17,\n      "game_frac": 0.321\n    },\n    {\n      "cardId": 11,\n      "name": "Mist Energy",\n      "copies": 2,\n      "games_seen": 16,\n      "game_frac": 0.302\n    },\n    {\n      "cardId": 1152,\n      "name": "Pok\\u00e9 Pad",\n      "copies": 2,\n      "games_seen": 14,\n      "game_frac": 0.264\n    },\n    {\n      "cardId": 1122,\n      "name": "Pok\\u00e9gear 3.0",\n      "copies": 2,\n      "games_seen": 13,\n      "game_frac": 0.245\n    },\n    {\n      "cardId": 1123,\n      "name": "Switch",\n      "copies": 2,\n      "games_seen": 12,\n      "game_frac": 0.226\n    },\n    {\n      "cardId": 14,\n      "name": "Spiky Energy",\n      "copies": 2,\n      "games_seen": 10,\n      "game_frac": 0.189\n    },\n    {\n      "cardId": 1224,\n      "name": "Cheren",\n      "copies": 2,\n      "games_seen": 9,\n      "game_frac": 0.17\n    },\n    {\n      "cardId": 1235,\n      "name": "Waitress",\n      "copies": 2,\n      "games_seen": 8,\n      "game_frac": 0.151\n    },\n    {\n      "cardId": 1159,\n      "name": "Hero\\u2019s Cape",\n      "copies": 2,\n      "games_seen": 8,\n      "game_frac": 0.151\n    },\n    {\n      "cardId": 1197,\n      "name": "Xerosic\\u2019s Machinations",\n      "copies": 1,\n      "games_seen": 7,\n      "game_frac": 0.132\n    },\n    {\n      "cardId": 1225,\n      "name": "Hilda",\n      "copies": 1,\n      "games_seen": 6,\n      "game_frac": 0.113\n    },\n    {\n      "cardId": 756,\n      "name": "Mega Kangaskhan ex",\n      "copies": 1,\n      "games_seen": 6,\n      "game_frac": 0.113\n    },\n    {\n      "cardId": 352,\n      "name": "Ethan\'s Cyndaquil",\n      "copies": 1,\n      "games_seen": 5,\n      "game_frac": 0.094\n    },\n    {\n      "cardId": 2,\n      "name": "Basic {R} Energy",\n      "copies": 1,\n      "games_seen": 5,\n      "game_frac": 0.094\n    },\n    {\n      "cardId": 1215,\n      "name": "Ethan\'s Adventure",\n      "copies": 1,\n      "games_seen": 5,\n      "game_frac": 0.094\n    },\n    {\n      "cardId": 1147,\n      "name": "Jumbo Ice Cream",\n      "copies": 1,\n      "games_seen": 5,\n      "game_frac": 0.094\n    },\n    {\n      "cardId": 1120,\n      "name": "Crushing Hammer",\n      "copies": 1,\n      "games_seen": 5,\n      "game_frac": 0.094\n    },\n    {\n      "cardId": 1186,\n      "name": "Eri",\n      "copies": 1,\n      "games_seen": 5,\n      "game_frac": 0.094\n    },\n    {\n      "cardId": 1204,\n      "name": "Lisia\\u2019s Appeal",\n      "copies": 1,\n      "games_seen": 5,\n      "game_frac": 0.094\n    },\n    {\n      "cardId": 1097,\n      "name": "Night Stretcher",\n      "copies": 1,\n      "games_seen": 4,\n      "game_frac": 0.075\n    },\n    {\n      "cardId": 1198,\n      "name": "Crispin",\n      "copies": 1,\n      "games_seen": 4,\n      "game_frac": 0.075\n    },\n    {\n      "cardId": 354,\n      "name": "Ethan\'s Typhlosion",\n      "copies": 1,\n      "games_seen": 4,\n      "game_frac": 0.075\n    },\n    {\n      "cardId": 1194,\n      "name": "Colress\\u2019s Tenacity",\n      "copies": 1,\n      "games_seen": 4,\n      "game_frac": 0.075\n    },\n    {\n      "cardId": 1210,\n      "name": "Brock\\u2019s Scouting",\n      "copies": 1,\n      "games_seen": 4,\n      "game_frac": 0.075\n    },\n    {\n      "cardId": 1079,\n      "name": "Rare Candy",\n      "copies": 1,\n      "games_seen": 3,\n      "game_frac": 0.057\n    },\n    {\n      "cardId": 20,\n      "name": "Rock Fighting Energy",\n      "copies": 1,\n      "games_seen": 3,\n      "game_frac": 0.057\n    }\n  ],\n  "gardevoir": [\n    {\n      "cardId": 745,\n      "name": "Ralts",\n      "copies": 4,\n      "games_seen": 4,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 747,\n      "name": "Mega Gardevoir ex",\n      "copies": 4,\n      "games_seen": 4,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 272,\n      "name": "Lillie\\u2019s Clefairy ex",\n      "copies": 4,\n      "games_seen": 4,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1227,\n      "name": "Lillie\'s Determination",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 0.75\n    },\n    {\n      "cardId": 5,\n      "name": "Basic {P} Energy",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 0.75\n    },\n    {\n      "cardId": 746,\n      "name": "Kirlia",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 0.75\n    },\n    {\n      "cardId": 1079,\n      "name": "Rare Candy",\n      "copies": 3,\n      "games_seen": 2,\n      "game_frac": 0.5\n    },\n    {\n      "cardId": 184,\n      "name": "Latias ex",\n      "copies": 3,\n      "games_seen": 2,\n      "game_frac": 0.5\n    },\n    {\n      "cardId": 1145,\n      "name": "Mega Signal",\n      "copies": 3,\n      "games_seen": 2,\n      "game_frac": 0.5\n    },\n    {\n      "cardId": 1121,\n      "name": "Ultra Ball",\n      "copies": 3,\n      "games_seen": 2,\n      "game_frac": 0.5\n    },\n    {\n      "cardId": 1225,\n      "name": "Hilda",\n      "copies": 3,\n      "games_seen": 2,\n      "game_frac": 0.5\n    },\n    {\n      "cardId": 1152,\n      "name": "Pok\\u00e9 Pad",\n      "copies": 3,\n      "games_seen": 2,\n      "game_frac": 0.5\n    },\n    {\n      "cardId": 1194,\n      "name": "Colress\\u2019s Tenacity",\n      "copies": 3,\n      "games_seen": 2,\n      "game_frac": 0.5\n    },\n    {\n      "cardId": 19,\n      "name": "Telepath Psychic Energy",\n      "copies": 3,\n      "games_seen": 2,\n      "game_frac": 0.5\n    },\n    {\n      "cardId": 1235,\n      "name": "Waitress",\n      "copies": 2,\n      "games_seen": 1,\n      "game_frac": 0.25\n    },\n    {\n      "cardId": 1205,\n      "name": "Cyrano",\n      "copies": 2,\n      "games_seen": 1,\n      "game_frac": 0.25\n    },\n    {\n      "cardId": 1231,\n      "name": "Dawn",\n      "copies": 2,\n      "games_seen": 1,\n      "game_frac": 0.25\n    },\n    {\n      "cardId": 1072,\n      "name": "Snorlax",\n      "copies": 2,\n      "games_seen": 1,\n      "game_frac": 0.25\n    },\n    {\n      "cardId": 765,\n      "name": "Meloetta",\n      "copies": 2,\n      "games_seen": 1,\n      "game_frac": 0.25\n    },\n    {\n      "cardId": 1097,\n      "name": "Night Stretcher",\n      "copies": 2,\n      "games_seen": 1,\n      "game_frac": 0.25\n    }\n  ],\n  "grimmsnarl": [\n    {\n      "cardId": 1152,\n      "name": "Pok\\u00e9 Pad",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 646,\n      "name": "Marnie\'s Impidimp",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 7,\n      "name": "Basic {D} Energy",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 647,\n      "name": "Marnie\'s Morgrem",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 648,\n      "name": "Marnie\'s Grimmsnarl ex",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1182,\n      "name": "Boss\\u2019s Orders",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1086,\n      "name": "Buddy-Buddy Poffin",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1097,\n      "name": "Night Stretcher",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1227,\n      "name": "Lillie\'s Determination",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1259,\n      "name": "Spikemuth Gym",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 112,\n      "name": "Munkidori",\n      "copies": 4,\n      "games_seen": 3,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1079,\n      "name": "Rare Candy",\n      "copies": 4,\n      "games_seen": 2,\n      "game_frac": 0.667\n    },\n    {\n      "cardId": 1219,\n      "name": "Team Rocket\'s Petrel",\n      "copies": 4,\n      "games_seen": 2,\n      "game_frac": 0.667\n    },\n    {\n      "cardId": 860,\n      "name": "Snorunt",\n      "copies": 4,\n      "games_seen": 2,\n      "game_frac": 0.667\n    },\n    {\n      "cardId": 104,\n      "name": "Froslass",\n      "copies": 4,\n      "games_seen": 2,\n      "game_frac": 0.667\n    }\n  ],\n  "kyogre": [\n    {\n      "cardId": 721,\n      "name": "Kyogre",\n      "copies": 4,\n      "games_seen": 11,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 3,\n      "name": "Basic {W} Energy",\n      "copies": 4,\n      "games_seen": 11,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1145,\n      "name": "Mega Signal",\n      "copies": 3,\n      "games_seen": 5,\n      "game_frac": 0.455\n    },\n    {\n      "cardId": 1121,\n      "name": "Ultra Ball",\n      "copies": 2,\n      "games_seen": 3,\n      "game_frac": 0.273\n    },\n    {\n      "cardId": 1219,\n      "name": "Team Rocket\'s Petrel",\n      "copies": 2,\n      "games_seen": 3,\n      "game_frac": 0.273\n    },\n    {\n      "cardId": 1227,\n      "name": "Lillie\'s Determination",\n      "copies": 2,\n      "games_seen": 3,\n      "game_frac": 0.273\n    },\n    {\n      "cardId": 1235,\n      "name": "Waitress",\n      "copies": 2,\n      "games_seen": 3,\n      "game_frac": 0.273\n    },\n    {\n      "cardId": 1205,\n      "name": "Cyrano",\n      "copies": 2,\n      "games_seen": 2,\n      "game_frac": 0.182\n    },\n    {\n      "cardId": 1262,\n      "name": "Surfing Beach",\n      "copies": 2,\n      "games_seen": 2,\n      "game_frac": 0.182\n    },\n    {\n      "cardId": 724,\n      "name": "Clauncher",\n      "copies": 2,\n      "games_seen": 2,\n      "game_frac": 0.182\n    },\n    {\n      "cardId": 725,\n      "name": "Clawitzer",\n      "copies": 2,\n      "games_seen": 2,\n      "game_frac": 0.182\n    },\n    {\n      "cardId": 1182,\n      "name": "Boss\\u2019s Orders",\n      "copies": 1,\n      "games_seen": 1,\n      "game_frac": 0.091\n    },\n    {\n      "cardId": 1092,\n      "name": "Secret Box",\n      "copies": 1,\n      "games_seen": 1,\n      "game_frac": 0.091\n    },\n    {\n      "cardId": 1163,\n      "name": "Powerglass",\n      "copies": 1,\n      "games_seen": 1,\n      "game_frac": 0.091\n    }\n  ],\n  "raging-bolt": [\n    {\n      "cardId": 1,\n      "name": "Basic {G} Energy",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 4,\n      "name": "Basic {L} Energy",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 6,\n      "name": "Basic {F} Energy",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 63,\n      "name": "Raging Bolt ex",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 96,\n      "name": "Teal Mask Ogerpon ex",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1198,\n      "name": "Crispin",\n      "copies": 4,\n      "games_seen": 6,\n      "game_frac": 0.857\n    },\n    {\n      "cardId": 1227,\n      "name": "Lillie\'s Determination",\n      "copies": 4,\n      "games_seen": 6,\n      "game_frac": 0.857\n    },\n    {\n      "cardId": 1121,\n      "name": "Ultra Ball",\n      "copies": 4,\n      "games_seen": 6,\n      "game_frac": 0.857\n    },\n    {\n      "cardId": 1122,\n      "name": "Pok\\u00e9gear 3.0",\n      "copies": 4,\n      "games_seen": 6,\n      "game_frac": 0.857\n    },\n    {\n      "cardId": 1080,\n      "name": "Unfair Stamp",\n      "copies": 4,\n      "games_seen": 5,\n      "game_frac": 0.714\n    },\n    {\n      "cardId": 1094,\n      "name": "Bug Catching Set",\n      "copies": 4,\n      "games_seen": 5,\n      "game_frac": 0.714\n    },\n    {\n      "cardId": 1118,\n      "name": "Energy Retrieval",\n      "copies": 4,\n      "games_seen": 5,\n      "game_frac": 0.714\n    },\n    {\n      "cardId": 1124,\n      "name": "Pok\\u00e9mon Catcher",\n      "copies": 4,\n      "games_seen": 5,\n      "game_frac": 0.714\n    },\n    {\n      "cardId": 1127,\n      "name": "Tera Orb",\n      "copies": 4,\n      "games_seen": 5,\n      "game_frac": 0.714\n    },\n    {\n      "cardId": 1182,\n      "name": "Boss\\u2019s Orders",\n      "copies": 3,\n      "games_seen": 4,\n      "game_frac": 0.571\n    },\n    {\n      "cardId": 184,\n      "name": "Latias ex",\n      "copies": 1,\n      "games_seen": 3,\n      "game_frac": 0.429\n    }\n  ],\n  "rockets-mewtwo": [\n    {\n      "cardId": 1218,\n      "name": "Team Rocket\'s Giovanni",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1220,\n      "name": "Team Rocket\'s Proton",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 431,\n      "name": "Team Rocket\'s Mewtwo ex",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 400,\n      "name": "Team Rocket\'s Tarountula",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 401,\n      "name": "Team Rocket\'s Spidops",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 434,\n      "name": "Team Rocket\'s Mimikyu",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 414,\n      "name": "Team Rocket\'s Articuno",\n      "copies": 4,\n      "games_seen": 7,\n      "game_frac": 1.0\n    },\n    {\n      "cardId": 1216,\n      "name": "Team Rocket\'s Ariana",\n      "copies": 4,\n      "games_seen": 6,\n      "game_frac": 0.857\n    },\n    {\n      "cardId": 1257,\n      "name": "Team Rocket\'s Factory",\n      "copies": 4,\n      "games_seen": 6,\n      "game_frac": 0.857\n    },\n    {\n      "cardId": 1134,\n      "name": "Team Rocket\'s Transceiver",\n      "copies": 4,\n      "games_seen": 6,\n      "game_frac": 0.857\n    },\n    {\n      "cardId": 1,\n      "name": "Basic {G} Energy",\n      "copies": 4,\n      "games_seen": 6,\n      "game_frac": 0.857\n    },\n    {\n      "cardId": 1094,\n      "name": "Bug Catching Set",\n      "copies": 4,\n      "games_seen": 5,\n      "game_frac": 0.714\n    },\n    {\n      "cardId": 1121,\n      "name": "Ultra Ball",\n      "copies": 3,\n      "games_seen": 4,\n      "game_frac": 0.571\n    },\n    {\n      "cardId": 15,\n      "name": "Team Rocket\'s Energy",\n      "copies": 3,\n      "games_seen": 4,\n      "game_frac": 0.571\n    },\n    {\n      "cardId": 1097,\n      "name": "Night Stretcher",\n      "copies": 3,\n      "games_seen": 3,\n      "game_frac": 0.429\n    },\n    {\n      "cardId": 1227,\n      "name": "Lillie\'s Determination",\n      "copies": 3,\n      "games_seen": 3,\n      "game_frac": 0.429\n    }\n  ]\n}')
print('wrote training/archetype_decks.json')


## Sanity import check

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo")
sys.path.insert(0, "/kaggle/working/repo/training")
sys.path.insert(0, "/kaggle/working/repo/training/nn")
sys.path.insert(0, "/kaggle/working/repo/training/local_cg")

import harness, opponent_pool, mcts, mcts_collect
print("modules imported OK")
for label, path, deck, n in opponent_pool.allocate(300):
    print(f"  {label:16s} n={n:3d} {'own_deck' if deck is None else 'archetype_deck'}")


## Locate the init checkpoint and run collection (300 games, redesigned pool)

In [ ]:
import glob

init_candidates = glob.glob("/kaggle/input/**/ptcg_dmc_p0_v2_n1_richenc_v2.pth", recursive=True)
assert init_candidates, "ptcg_dmc_p0_v2_n1_richenc_v2.pth not found under /kaggle/input"
init_ckpt = init_candidates[0]
print("init_ckpt:", init_ckpt)

get_ipython().system(
    f'python training/nn/mcts_collect.py --games 300 --sims 40 '
    f'--ckpt "{init_ckpt}" --out /kaggle/working/mcts_p3_r1.pkl.gz'
)


## Retrain on the new corpus (same settings as round 2, pointed at the new corpus)

In [ ]:
get_ipython().system(
    f'python training/nn/train_sp.py --bc-data /kaggle/working/mcts_p3_r1.pkl.gz --bc-limit 10 --bc-frac 0 '
    f'--sp-data /kaggle/working/mcts_p3_r1.pkl.gz --epochs 8 --steps-per-epoch 165 '
    f'--init "{init_ckpt}" --out /kaggle/working/ptcg_sp_p3_r1.pth'
)


## Gate: real-replay value-head sign accuracy (`--value-source head`)

In [ ]:
import glob, os

replay_candidates = glob.glob("/kaggle/input/**/episode-*-replay.json", recursive=True)
assert replay_candidates, "no replay JSONs found under /kaggle/input"
replays_dir = os.path.dirname(replay_candidates[0])
print("replays_dir:", replays_dir, "n_replays:", len(glob.glob(os.path.join(replays_dir, "*.json"))))

print("=== pre-training baseline (init checkpoint) ===")
get_ipython().system(f'python training/nn/dmc_replay_gate.py --ckpt "{init_ckpt}" --replays-dir "{replays_dir}" --value-source head')

print("=== post-training (round 3, redesigned opponent pool) ===")
get_ipython().system(f'python training/nn/dmc_replay_gate.py --ckpt /kaggle/working/ptcg_sp_p3_r1.pth --replays-dir "{replays_dir}" --value-source head')


## Next steps

- Download `/kaggle/working/mcts_p3_r1.pkl.gz` (new corpus) and
  `/kaggle/working/ptcg_sp_p3_r1.pth` (new checkpoint).
- Compare the gate output above against: pre-training baseline (0.584 on
  this replay set), round 2's collapsed result (0.446), Φ v2 (0.604/0.696),
  best DMC checkpoint (0.609/0.700).
- If this clears the pre-training baseline, it's the best real-replay value
  signal of the project so far — log it and proceed per
  `docs/nn-training.md` "Resume Here". If not, log the honest negative —
  same discipline as round 2.